# Eesti triibuseelikute mustrite analüüs
Kood koosenb järgnevatest sammudest:
1. Peamine kanga-analüüs, mis leiab iga Eesti kanga kohta tema top5 parimat vastet (analüüsib värvi, tekstuuri, mustrit)
2. Analüüsib, kuidas värvi, tekstuuri ja mustri kaalude osakaal mõjutab seda, milliseid vasteid saadakse.
3. Kuvab visuaalselt kaalude mõju, et saaks silmaga hinnata sarnasusi ja leida parim kaalude komponent.
4. Visualiseerib peamise kanga-analüüsi tabeli üldinfo.
5. Võrdleb Eesti-Eesti ja Inglise-Inglise kangaid ning leiab, millistel andmestikel on kõige enam sarnasusi.
6. Loob Eesti kangastest andmebaasi.
7. Kasutab loodud andmebaasi, et leida kasutaja valitud konkreetsele ühele kangale 10 parimat vastet ning visualiseerib vasted.
8. Selgitab, mida erinevad analüüsiosas kangapildiga täpsemalt teevad ja kuidas masin kangast "näeb"

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


Mounted at /content/drive


## Etapp 1. Peamine kanga-sarnasuse analüüs

### Eesmärk

Selle etapi eesmärk on leida iga Eesti kanga jaoks visuaalselt kõige sarnasemad Inglismaa kangad.

Tulemuseks on **TOP 5 vastet iga Eesti kanga kohta**, mida kasutatakse järgmistes analüüsi etappides.

### Sisendandmed

- **Andmetüüp:** pildifailid (`.jpg`, `.jpeg`, `.png`, `.bmp`)
- **Andmete asukoht:** Google Drive / Colab
- **Kaustad:**
  - `Eesti_10`
  - `Eesti_20`
  - `Inglismaa_20`
  - `Inglismaa9`
- **Maht:** ligikaudu 700 Eesti kangapilti ning eraldi Inglismaa kangaste võrdluskogum
- **Ühik:** üks pilt tähistab ühte kangaproovi

### Analüüsi loogika

Sarnasust ei hinnata ainult ühe tunnuse põhjal. Iga kangapildi kohta arvutatakse kolm erinevat tunnuste rühma:

1. **CNN-tunnused**
   - kasutatakse eeltreenitud ResNet50 mudelit;
   - kirjeldavad pildi üldist visuaalset struktuuri, tekstuuri ja mustrit.

2. **Värvipalett**
   - arvutatakse LAB-värviruumis;
   - kirjeldab pildi domineerivaid värve ja nende osakaale.

3. **Triibumustri proportsioonid**
   - tuvastatakse heledusprofiili põhjal;
   - kirjeldavad triipude suhtelisi laiusi ja mustri rütmi.

### Kasutatavad kaalud

Sarnasuse koondskoor arvutatakse kolme komponendi kaalutud summana:

- **CNN:** `0.55`
- **Värv:** `0.25`
- **Triibumustri proportsioon:** `0.20`

Need kaalud määravad, milline tunnus mõjutab lõplikku sarnasusskoori kõige rohkem.

### Peamised tegevused

Selles etapis tehakse järgmised tegevused:

1. kontrollitakse, mitu pilti on igas sisendkaustas;
2. laaditakse eeltreenitud ResNet50 mudel;
3. arvutatakse Inglismaa kangaste tunnused;
4. salvestatakse Inglismaa kangaste tunnused indeksina;
5. arvutatakse iga Eesti kanga tunnused;
6. võrreldakse iga Eesti kangast kõigi Inglismaa kangastega;
7. valitakse iga Eesti kanga jaoks viis kõige sarnasemat Inglismaa kangast;
8. salvestatakse tulemused CSV-failina.

### Väljundid

Selle etapi tulemusena tekib kaks peamist väljundfaili:

- `england_index_v1.joblib`  
  Inglismaa kangaste tunnuste andmebaas. Seda kasutatakse järgmistes sammudes, et vältida samade tunnuste korduvat arvutamist.

- `results_similarity_main.csv`  
  Tabel, kus iga Eesti kanga kohta on viis parimat Inglismaa vastet.

CSV-fail sisaldab muu hulgas järgmisi veerge:

- Eesti kanga faili nimi;
- Eesti kanga suuruskategooria;
- vaste järjekoht;
- Inglismaa kanga faili nimi;
- Inglismaa kanga suuruskategooria;
- koondsarnasus;
- CNN-sarnasus;
- värvisarnasus;
- triibumustri proportsiooni sarnasus;
- kasutatud kaalud.

### Ajahinnang

See on üks töövoo ajamahukamaid samme.

Esmakordsel käivitamisel võtab rohkem aega Inglismaa kangaste indeksi loomine. Kui indeks on juba olemas, saab seda järgmistel kordadel uuesti kasutada ning analüüs on kiirem.

Ajakulu sõltub peamiselt:

- piltide arvust;
- piltide suurusest;
- sellest, kas kasutatakse GPU-d või CPU-d;
- sellest, kas Inglismaa kangaste indeks on juba varem loodud.

### Nähtavad vaheväljundid

Koodi käivitamisel kuvatakse notebookis mitu kontrollväljundit:

- piltide arv igas kaustas;
- Eesti ja Inglismaa kangaste koguarv;
- kasutatav arvutusseade;
- info selle kohta, kas Inglismaa indeks luuakse või laaditakse olemasolevast failist;
- töödeldud Eesti kangaste arv;
- TOP 1 vastete keskmine sarnasusskoor.

Need väljundid aitavad kontrollida, kas sisendandmed on õigesti loetud ja kas analüüs lõppes edukalt.

### Tõlgendamise märkus

Tulemused sõltuvad valitud kaaludest.

Kui **CNN-komponendi kaal** on suurem, muutub olulisemaks kanga üldine visuaalne mulje.  
Kui **värvikomponendi kaal** on suurem, eelistatakse sarnaste toonidega kangaid.  
Kui **triibumustri komponendi kaal** on suurem, eelistatakse sarnase rütmi ja triibulaiustega mustreid.

Seetõttu analüüsitakse järgmistes etappides eraldi, kuidas kaalude muutmine mõjutab saadud vasteid.

In [ ]:
"""
STEP 1: MAIN FABRIC ANALYSIS
==============================
Analyzes ALL Estonian fabrics (~700) against English fabrics.
Creates index and CSV with TOP 5 matches per fabric.

OUTPUT:
- /content/drive/MyDrive/Masinõpe_tulemused/england_index_v1.joblib
- /content/drive/MyDrive/Masinõpe_tulemused/results_similarity_main.csv

RUN ONCE, then use Step 2 and Step 3 scripts
"""

import os
import sys
import torch
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import cv2
import joblib
from typing import Optional, Tuple, List

# ============================================================
# SETTINGS
# ============================================================
BASE_PATH = "/content/drive/MyDrive/Kangad"
RESULTS_DIR = "/content/drive/MyDrive/Masinõpe_tulemused"

# Weights for similarity calculation
W_CNN = 0.55
W_COLOR = 0.25
W_RATIO = 0.20

print("=" * 70)
print("STEP 1: MAIN FABRIC SIMILARITY ANALYSIS")
print("=" * 70)
print(f"Weights: CNN={W_CNN}, COLOR={W_COLOR}, RATIO={W_RATIO}")
print(f"Results will be saved to: {RESULTS_DIR}")


# ============================================================
# HELPER FUNCTIONS
# ============================================================
def count_images(path, label):
    try:
        files = [f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        print(f"{label:30s}: {len(files):4d} images")
        return files
    except Exception as e:
        print(f"{label:30s}: ERROR - {e}")
        return []


def get_transform(size_category):
    if size_category == '10cm':
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.CenterCrop(224),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])


def extract_cnn_features(img_path, model, device, size_category='20cm'):
    try:
        img = Image.open(img_path).convert('RGB')
        transform = get_transform(size_category)
        img_t = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            features = model(img_t).squeeze().cpu().numpy()
        return features
    except Exception as e:
        print(f"Error (CNN): {img_path} - {e}")
        return None


# ============================================================
# LAB COLOR PALETTE
# ============================================================
def extract_lab_palette(img: Image.Image, n_colors: int = 6) -> Tuple[np.ndarray, np.ndarray]:
    arr = np.array(img)
    if arr.ndim != 3 or arr.shape[0] < 10 or arr.shape[1] < 10:
        raise ValueError("Image too small")

    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB).astype(np.float32)
    pixels = lab.reshape(-1, 3)

    if len(pixels) > 50000:
        idx = np.random.choice(len(pixels), 50000, replace=False)
        pixels = pixels[idx]

    K = min(n_colors, max(3, len(pixels) // 5000 + 3))
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1.0)
    flags = cv2.KMEANS_PP_CENTERS

    _, labels, centers = cv2.kmeans(pixels, K, None, criteria, 3, flags)
    labels = labels.flatten()
    centers = centers.astype(np.float32)

    counts = np.bincount(labels, minlength=K).astype(np.float32)
    weights = counts / counts.sum()

    order = np.argsort(weights)[::-1]
    return centers[order], weights[order]


def _greedy_min_cost_matching(cost: np.ndarray) -> List[Tuple[int, int]]:
    cost = cost.copy()
    pairs = []
    used_r, used_c = set(), set()
    while len(used_r) < cost.shape[0] and len(used_c) < cost.shape[1]:
        best = None
        best_val = None
        for i in range(cost.shape[0]):
            if i in used_r:
                continue
            for j in range(cost.shape[1]):
                if j in used_c:
                    continue
                v = cost[i, j]
                if best_val is None or v < best_val:
                    best_val = v
                    best = (i, j)
        if best is None:
            break
        i, j = best
        used_r.add(i)
        used_c.add(j)
        pairs.append((i, j))
    return pairs


def lab_palette_similarity(centers1: np.ndarray, w1: np.ndarray,
                          centers2: np.ndarray, w2: np.ndarray) -> float:
    if centers1 is None or centers2 is None:
        return 0.0

    cost = np.linalg.norm(centers1[:, None, :] - centers2[None, :, :], axis=2)

    try:
        from scipy.optimize import linear_sum_assignment
        r, c = linear_sum_assignment(cost)
        pairs = list(zip(r.tolist(), c.tolist()))
    except:
        pairs = _greedy_min_cost_matching(cost)

    d_sum, w_sum = 0.0, 0.0
    for i, j in pairs:
        wij = float(min(w1[i], w2[j]))
        d_sum += wij * float(cost[i, j])
        w_sum += wij

    if w_sum <= 1e-9:
        return 0.0

    d = d_sum / w_sum
    scale = 60.0
    sim = float(np.exp(-d / scale))
    return max(0.0, min(1.0, sim))


# ============================================================
# STRIPE WIDTHS
# ============================================================
def extract_stripe_widths(img: Image.Image, min_stripe_px: int = 3) -> Optional[np.ndarray]:
    arr = np.array(img.convert("RGB"))
    if arr.shape[0] < 10 or arr.shape[1] < 10:
        return None

    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB).astype(np.float32)
    L = lab[:, :, 0]

    prof = L.mean(axis=0)
    prof_s = cv2.GaussianBlur(prof.reshape(1, -1), (1, 9), 0).flatten()

    g = np.abs(np.diff(prof_s, prepend=prof_s[0]))
    thr = np.percentile(g, 90)
    edge_idx = np.where(g >= thr)[0]
    if len(edge_idx) < 2:
        return None

    edges = [int(edge_idx[0])]
    for x in edge_idx[1:]:
        if x - edges[-1] >= min_stripe_px:
            edges.append(int(x))

    if edges[0] > 0:
        edges = [0] + edges
    if edges[-1] < arr.shape[1] - 1:
        edges = edges + [arr.shape[1] - 1]

    widths = []
    for a, b in zip(edges[:-1], edges[1:]):
        w = b - a
        if w >= min_stripe_px:
            widths.append(w)

    if len(widths) < 2:
        return None

    widths = np.array(widths, dtype=np.float32)
    widths = widths / widths.sum()
    return widths


def dtw_distance(a: np.ndarray, b: np.ndarray) -> float:
    n, m = len(a), len(b)
    D = np.full((n + 1, m + 1), np.inf, dtype=np.float32)
    D[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = abs(a[i - 1] - b[j - 1])
            D[i, j] = cost + min(D[i - 1, j], D[i, j - 1], D[i - 1, j - 1])
    return float(D[n, m])


def stripe_ratio_similarity(w1: Optional[np.ndarray], w2: Optional[np.ndarray]) -> float:
    if w1 is None or w2 is None:
        return 0.0
    d = dtw_distance(w1, w2)
    scale = 0.5
    sim = float(np.exp(-d / scale))
    return max(0.0, min(1.0, sim))


# ============================================================
# FEATURE EXTRACTION
# ============================================================
def extract_feature_pack(img_path: str, model, device, size_category: str):
    try:
        img = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"Error (open): {img_path} - {e}")
        return None

    cnn = extract_cnn_features(img_path, model, device, size_category=size_category)
    if cnn is None:
        return None

    try:
        pal_c, pal_w = extract_lab_palette(img, n_colors=6)
    except:
        pal_c, pal_w = None, None

    try:
        sw = extract_stripe_widths(img, min_stripe_px=3)
    except:
        sw = None

    return {"cnn": cnn, "pal_c": pal_c, "pal_w": pal_w, "stripe_w": sw}


def combined_similarity(e_pack, i_pack, w_cnn=0.55, w_color=0.25, w_ratio=0.20) -> dict:
    cnn_sim = float(cosine_similarity([e_pack["cnn"]], [i_pack["cnn"]])[0][0])
    cnn_sim_01 = (cnn_sim + 1.0) / 2.0

    color_sim = 0.0
    if e_pack["pal_c"] is not None and i_pack["pal_c"] is not None:
        color_sim = lab_palette_similarity(e_pack["pal_c"], e_pack["pal_w"], i_pack["pal_c"], i_pack["pal_w"])

    ratio_sim = stripe_ratio_similarity(e_pack["stripe_w"], i_pack["stripe_w"])

    total = w_cnn * cnn_sim_01 + w_color * color_sim + w_ratio * ratio_sim
    return {
        "total": float(total),
        "cnn": float(cnn_sim_01),
        "color": float(color_sim),
        "ratio": float(ratio_sim)
    }


# ============================================================
# INDEX MANAGEMENT
# ============================================================
def save_england_index(index_path: str, eng_packs, eng_names, eng_sizes, meta: dict):
    obj = {"packs": eng_packs, "names": eng_names, "sizes": eng_sizes, "meta": meta}
    joblib.dump(obj, index_path)


def load_england_index(index_path: str):
    return joblib.load(index_path)


def build_england_index(england_path_20, england_path_10, england_20_files, england_10_files, model, device):
    eng_packs, eng_names, eng_sizes = [], [], []

    print("\nProcessing 20cm samples...")
    for img_name in tqdm(england_20_files, desc="England 20cm"):
        pack = extract_feature_pack(os.path.join(england_path_20, img_name), model, device, size_category='20cm')
        if pack is not None:
            eng_packs.append(pack)
            eng_names.append(img_name)
            eng_sizes.append('20cm')

    print("Processing 10cm samples...")
    for img_name in tqdm(england_10_files, desc="England 10cm"):
        pack = extract_feature_pack(os.path.join(england_path_10, img_name), model, device, size_category='10cm')
        if pack is not None:
            eng_packs.append(pack)
            eng_names.append(img_name)
            eng_sizes.append('10cm')

    return eng_packs, eng_names, eng_sizes


# ============================================================
# MAIN ANALYSIS
# ============================================================
def main():
    os.makedirs(RESULTS_DIR, exist_ok=True)

    # Paths
    estonia_path_10 = os.path.join(BASE_PATH, 'Eesti_10')
    estonia_path_20 = os.path.join(BASE_PATH, 'Eesti_20')
    england_path_20 = os.path.join(BASE_PATH, 'Inglismaa_20')
    england_path_10 = os.path.join(BASE_PATH, 'Inglismaa9')

    print("\n" + "=" * 70)
    print("IMAGE COUNT IN FOLDERS:")
    print("=" * 70)

    estonia_10_files = count_images(estonia_path_10, "Estonian 10cm")
    estonia_20_files = count_images(estonia_path_20, "Estonian 20cm")
    england_20_files = count_images(england_path_20, "English 20cm")
    england_10_files = count_images(england_path_10, "English 10cm")

    print(f"\nTOTAL ESTONIAN: {len(estonia_10_files) + len(estonia_20_files)} images")
    print(f"TOTAL ENGLISH: {len(england_20_files) + len(england_10_files)} images")

    # Load model
    print("\n" + "=" * 70)
    print("LOADING ResNet50 MODEL...")
    print("=" * 70)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    model = models.resnet50(pretrained=True)
    model = torch.nn.Sequential(*list(model.children())[:-1])
    model = model.to(device)
    model.eval()

    # Load or build index
    index_path = os.path.join(RESULTS_DIR, "england_index_v1.joblib")

    if os.path.exists(index_path):
        print("\n✓ Loading existing England index...")
        index = load_england_index(index_path)
        eng_packs = index["packs"]
        eng_names = index["names"]
        eng_sizes = index["sizes"]
        print(f"  Index contains {len(eng_packs)} fabrics")
    else:
        print("\n⚙ Building England index (this will take time)...")
        eng_packs, eng_names, eng_sizes = build_england_index(
            england_path_20, england_path_10, england_20_files, england_10_files, model, device
        )
        meta = {
            "base_path": BASE_PATH,
            "count_20": sum(1 for s in eng_sizes if s == "20cm"),
            "count_10": sum(1 for s in eng_sizes if s == "10cm")
        }
        save_england_index(index_path, eng_packs, eng_names, eng_sizes, meta)
        print(f"✓ Index saved: {index_path}")

    # Analyze all Estonian fabrics
    print("\n" + "=" * 70)
    print("ANALYZING ALL ESTONIAN FABRICS (TOP 5 matches each)")
    print("=" * 70)

    results = []

    def process_estonia_set(files, path, size_cat):
        for estonia_img in tqdm(files, desc=f"Estonian {size_cat}"):
            e_pack = extract_feature_pack(os.path.join(path, estonia_img), model, device, size_category=size_cat)
            if e_pack is None:
                continue

            # Calculate similarities with all English fabrics
            totals = np.zeros(len(eng_packs), dtype=np.float32)
            cnn_s = np.zeros(len(eng_packs), dtype=np.float32)
            col_s = np.zeros(len(eng_packs), dtype=np.float32)
            rat_s = np.zeros(len(eng_packs), dtype=np.float32)

            for i, i_pack in enumerate(eng_packs):
                s = combined_similarity(e_pack, i_pack, w_cnn=W_CNN, w_color=W_COLOR, w_ratio=W_RATIO)
                totals[i] = s["total"]
                cnn_s[i] = s["cnn"]
                col_s[i] = s["color"]
                rat_s[i] = s["ratio"]

            # Get TOP 5
            top_5_idx = np.argsort(totals)[-5:][::-1]

            for rank, idx in enumerate(top_5_idx, 1):
                results.append({
                    'estonia_name': estonia_img,
                    'estonia_size': size_cat,
                    'rank': rank,
                    'england_name': eng_names[int(idx)],
                    'england_size': eng_sizes[int(idx)],
                    'similarity_total': float(totals[int(idx)]),
                    'similarity_cnn': float(cnn_s[int(idx)]),
                    'similarity_color': float(col_s[int(idx)]),
                    'similarity_ratio': float(rat_s[int(idx)]),
                    'weights_used': f"CNN{W_CNN}_COLOR{W_COLOR}_RATIO{W_RATIO}"
                })

    process_estonia_set(estonia_10_files, estonia_path_10, '10cm')
    process_estonia_set(estonia_20_files, estonia_path_20, '20cm')

    # Save results
    df = pd.DataFrame(results)
    output_csv = os.path.join(RESULTS_DIR, 'results_similarity_main.csv')
    df.to_csv(output_csv, index=False, encoding='utf-8-sig')

    print("\n" + "=" * 70)
    print("✓ STEP 1 COMPLETE!")
    print("=" * 70)
    print(f"\nResults saved to:")
    print(f"  - {output_csv}")
    print(f"  - {index_path}")
    print(f"\nTotal Estonian fabrics analyzed: {len(df['estonia_name'].unique())}")
    print(f"Average similarity (TOP-1): {df[df['rank']==1]['similarity_total'].mean():.4f}")
    print("\n💡 Next: Run STEP 2 script for deep analysis of TOP 20!")


if __name__ == "__main__":
    main()

STEP 1: MAIN FABRIC SIMILARITY ANALYSIS
Weights: CNN=0.55, COLOR=0.25, RATIO=0.2
Results will be saved to: /content/drive/MyDrive/Masinõpe_tulemused

IMAGE COUNT IN FOLDERS:
Estonian 10cm                 :  134 images
Estonian 20cm                 :  556 images
English 20cm                  : 2590 images
English 10cm                  :  656 images

TOTAL ESTONIAN: 690 images
TOTAL ENGLISH: 3246 images

LOADING ResNet50 MODEL...
Using device: cpu


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 149MB/s]



⚙ Building England index (this will take time)...

Processing 20cm samples...


...

Processing 10cm samples...


England 10cm: 100%|██████████| 656/656 [08:07<00:00,  1.35it/s]


✓ Index saved: /content/drive/MyDrive/Masinõpe_tulemused/england_index_v1.joblib

ANALYZING ALL ESTONIAN FABRICS (TOP 5 matches each)


Estonian 20cm: 100%|██████████| 556/556 [2:30:00<00:00, 16.19s/it]



✓ STEP 1 COMPLETE!

Results saved to:
  - /content/drive/MyDrive/Masinõpe_tulemused/results_similarity_main.csv
  - /content/drive/MyDrive/Masinõpe_tulemused/england_index_v1.joblib

Total Estonian fabrics analyzed: 690
Average similarity (TOP-1): 0.7552

💡 Next: Run STEP 2 script for deep analysis of TOP 20!


## Etapp 2. Kaalude mõju detailanalüüs

### Eesmärk

Selle etapi eesmärk on uurida, kuidas sarnasuse arvutamisel kasutatavad kaalud mõjutavad saadud vasteid.

Eelmises etapis leiti iga Eesti kanga jaoks parimad Inglismaa vasted ühe kindla kaalude kombinatsiooniga. Selles etapis võetakse neist tulemused aluseks ning võrreldakse, kas parim vaste muutub siis, kui analüüsis eelistada rohkem värvi või rohkem mustrit.

### Sisendandmed

- **Põhitulemuste fail:** `results_similarity_main.csv`
- **Inglismaa kangaste indeks:** `england_index_v1.joblib`
- **Algpildid:** Eesti kangaste pildifailid kaustades `Eesti_10` ja `Eesti_20`

Analüüsi ei tehta uuesti kõigi Eesti kangastega, vaid valitakse välja **20 Eesti kangast**, millel oli esimeses etapis kõige kõrgem TOP 1 sarnasusskoor.

### Analüüsi loogika

Selles etapis võrreldakse samu Eesti kangaid Inglismaa kangastega mitme erineva kaalude kombinatsiooni abil.

Kasutatakse kolme strateegiat:

1. **Algne kaalumine**
   - tasakaalustatud mudel;
   - enim arvestab CNN-põhist üldist visuaalset sarnasust.

2. **Värvile keskenduv kaalumine**
   - annab suurima tähtsuse värvipaletile;
   - aitab näha, millised vasted leitakse siis, kui toonide sarnasus on kõige olulisem.

3. **Mustrile keskenduv kaalumine**
   - annab suurima tähtsuse triibumustri proportsioonidele;
   - aitab näha, millised vasted leitakse siis, kui mustri rütm ja triipude laiused on kõige olulisemad.

### Kasutatavad kaalud

| Kaalude konfiguratsioon | CNN | Värv | Muster |
|---|---:|---:|---:|
| `Original_55-25-20` | 0.55 | 0.25 | 0.20 |
| `Color_20-60-20` | 0.20 | 0.60 | 0.20 |
| `Pattern_20-20-60` | 0.20 | 0.20 | 0.60 |

### Peamised tegevused

Selles etapis tehakse järgmised tegevused:

1. laaditakse sisse esimese etapi tulemuste tabel;
2. laaditakse Inglismaa kangaste tunnuste indeks;
3. valitakse välja 20 kõige tugevama TOP 1 vastega Eesti kangast;
4. arvutatakse nende Eesti kangaste tunnused uuesti;
5. võrreldakse iga valitud Eesti kangast kõigi Inglismaa kangastega;
6. arvutatakse sarnasused kolme erineva kaalude konfiguratsiooniga;
7. leitakse iga Eesti kanga jaoks iga kaalustrateegia parim Inglismaa vaste;
8. salvestatakse tulemused uude CSV-faili.

### Väljundid

Selle etapi väljundiks on fail:

- `results_deep_analysis_top20.csv`

See tabel sisaldab iga valitud Eesti kanga kohta kolme rida: üks rida iga kaalude konfiguratsiooni kohta.

Tabelis on muu hulgas järgmised veerud:

- Eesti kanga faili nimi;
- Eesti kanga suuruskategooria;
- kasutatud kaalude konfiguratsioon;
- CNN-kaal;
- värvikaal;
- mustrikaal;
- parima Inglismaa vaste faili nimi;
- Inglismaa kanga suuruskategooria;
- koondsarnasus;
- CNN-sarnasus;
- värvisarnasus;
- mustrisarnasus.

### Ajahinnang

See samm on kiirem kui esimene etapp, sest analüüsi tehakse ainult 20 Eesti kanga põhjal.

Ajakulu sõltub siiski:

- sellest, kas Step 1 väljundfailid on olemas;
- sellest, kui kiiresti Eesti kangaste tunnused uuesti arvutatakse;
- GPU või CPU kasutusest;
- Inglismaa kangaste arvust, sest iga valitud Eesti kangast võrreldakse endiselt kõigi Inglismaa kangastega.

### Nähtavad vaheväljundid

Notebookis kuvatakse selle etapi jooksul:

- kasutatavad kaalude konfiguratsioonid;
- info selle kohta, kas vajalikud Step 1 failid leiti üles;
- laaditud tulemuste ridade arv;
- Inglismaa indeksis olevate kangaste arv;
- valitud TOP 20 Eesti kangaste nimekiri;
- kasutatav arvutusseade;
- edukalt töödeldud Eesti kangaste arv;
- iga kaalustrateegia keskmine sarnasusskoor.

Need väljundid aitavad kontrollida, kas analüüs kasutab õigeid sisendfaile ja kas kaalude mõju võrdlus on edukalt tehtud.

### Tõlgendamise märkus

Selle etapi tulemused näitavad, kas sama Eesti kangas saab erinevate kaalude korral sama või erineva parima vaste.

Kui vaste jääb kõigi kolme strateegia puhul samaks, võib seda pidada stabiilsemaks sarnasuseks.  
Kui vaste muutub, näitab see, et sarnasus sõltub tugevalt sellest, kas mudel peab olulisemaks üldist visuaalset muljet, värvi või triibumustrit.

See samm ei anna veel lõplikku vastust sellele, milline kaalude kombinatsioon on parim. Pigem aitab see hinnata, millised kaalud annavad uurija jaoks visuaalselt ja sisuliselt kõige paremini tõlgendatavad tulemused.

In [ ]:
"""
STEP 2: DEEP WEIGHT ANALYSIS
==============================
Takes TOP 20 Estonian fabrics from Step 1 and re-analyzes them
with 3 different weight configurations to see which English fabric
matches best for each weighting strategy.

INPUT:
- results_similarity_main.csv (from Step 1)
- england_index_v1.joblib (from Step 1)

OUTPUT:
- results_deep_analysis_top20.csv

WEIGHT CONFIGURATIONS:
1. Original: 55% CNN, 25% COLOR, 20% RATIO
2. Color-focused: 20% CNN, 60% COLOR, 20% RATIO
3. Pattern-focused: 20% CNN, 20% COLOR, 60% RATIO
"""

import os
import pandas as pd
import numpy as np
import joblib
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
import cv2
from PIL import Image
from typing import Optional, Tuple, List

# ============================================================
# SETTINGS
# ============================================================
BASE_PATH = "/content/drive/MyDrive/Kangad"
RESULTS_DIR = "/content/drive/MyDrive/Masinõpe_tulemused"

# Three weight configurations
WEIGHT_CONFIGS = {
    'Original_55-25-20': {'cnn': 0.55, 'color': 0.25, 'ratio': 0.20},
    'Color_20-60-20': {'cnn': 0.20, 'color': 0.60, 'ratio': 0.20},
    'Pattern_20-20-60': {'cnn': 0.20, 'color': 0.20, 'ratio': 0.60}
}

print("=" * 70)
print("STEP 2: DEEP WEIGHT ANALYSIS FOR TOP 20")
print("=" * 70)
print("\nWeight configurations:")
for name, weights in WEIGHT_CONFIGS.items():
    print(f"  {name}: CNN={weights['cnn']}, COLOR={weights['color']}, RATIO={weights['ratio']}")


# ============================================================
# SIMILARITY FUNCTIONS (copied from Step 1)
# ============================================================
def lab_palette_similarity(centers1: np.ndarray, w1: np.ndarray,
                          centers2: np.ndarray, w2: np.ndarray) -> float:
    if centers1 is None or centers2 is None:
        return 0.0

    cost = np.linalg.norm(centers1[:, None, :] - centers2[None, :, :], axis=2)

    try:
        from scipy.optimize import linear_sum_assignment
        r, c = linear_sum_assignment(cost)
        pairs = list(zip(r.tolist(), c.tolist()))
    except:
        # Greedy fallback
        cost_copy = cost.copy()
        pairs = []
        used_r, used_c = set(), set()
        while len(used_r) < cost.shape[0] and len(used_c) < cost.shape[1]:
            best = None
            best_val = None
            for i in range(cost.shape[0]):
                if i in used_r:
                    continue
                for j in range(cost.shape[1]):
                    if j in used_c:
                        continue
                    v = cost_copy[i, j]
                    if best_val is None or v < best_val:
                        best_val = v
                        best = (i, j)
            if best is None:
                break
            i, j = best
            used_r.add(i)
            used_c.add(j)
            pairs.append((i, j))

    d_sum, w_sum = 0.0, 0.0
    for i, j in pairs:
        wij = float(min(w1[i], w2[j]))
        d_sum += wij * float(cost[i, j])
        w_sum += wij

    if w_sum <= 1e-9:
        return 0.0

    d = d_sum / w_sum
    scale = 60.0
    sim = float(np.exp(-d / scale))
    return max(0.0, min(1.0, sim))


def dtw_distance(a: np.ndarray, b: np.ndarray) -> float:
    n, m = len(a), len(b)
    D = np.full((n + 1, m + 1), np.inf, dtype=np.float32)
    D[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = abs(a[i - 1] - b[j - 1])
            D[i, j] = cost + min(D[i - 1, j], D[i, j - 1], D[i - 1, j - 1])
    return float(D[n, m])


def stripe_ratio_similarity(w1: Optional[np.ndarray], w2: Optional[np.ndarray]) -> float:
    if w1 is None or w2 is None:
        return 0.0
    d = dtw_distance(w1, w2)
    scale = 0.5
    sim = float(np.exp(-d / scale))
    return max(0.0, min(1.0, sim))


def combined_similarity_with_weights(e_pack, i_pack, w_cnn, w_color, w_ratio) -> dict:
    """Calculate similarity with specified weights"""
    cnn_sim = float(cosine_similarity([e_pack["cnn"]], [i_pack["cnn"]])[0][0])
    cnn_sim_01 = (cnn_sim + 1.0) / 2.0

    color_sim = 0.0
    if e_pack["pal_c"] is not None and i_pack["pal_c"] is not None:
        color_sim = lab_palette_similarity(e_pack["pal_c"], e_pack["pal_w"],
                                          i_pack["pal_c"], i_pack["pal_w"])

    ratio_sim = stripe_ratio_similarity(e_pack["stripe_w"], i_pack["stripe_w"])

    total = w_cnn * cnn_sim_01 + w_color * color_sim + w_ratio * ratio_sim

    return {
        "total": float(total),
        "cnn": float(cnn_sim_01),
        "color": float(color_sim),
        "ratio": float(ratio_sim)
    }


# ============================================================
# MAIN DEEP ANALYSIS
# ============================================================
def main():
    # Check required files
    main_csv = os.path.join(RESULTS_DIR, 'results_similarity_main.csv')
    index_path = os.path.join(RESULTS_DIR, 'england_index_v1.joblib')

    if not os.path.exists(main_csv):
        print(f"❌ ERROR: {main_csv} not found!")
        print("Please run STEP 1 first!")
        return

    if not os.path.exists(index_path):
        print(f"❌ ERROR: {index_path} not found!")
        print("Please run STEP 1 first!")
        return

    # Load main results
    print("\n" + "=" * 70)
    print("LOADING DATA")
    print("=" * 70)

    df_main = pd.read_csv(main_csv)
    print(f"✓ Loaded main results: {len(df_main)} rows")

    # Load England index
    index = joblib.load(index_path)
    eng_packs = index["packs"]
    eng_names = index["names"]
    eng_sizes = index["sizes"]
    print(f"✓ Loaded England index: {len(eng_packs)} fabrics")

    # Get TOP 20 Estonian fabrics (by best match similarity)
    top20_estonia = (
        df_main[df_main['rank'] == 1]
        .nlargest(20, 'similarity_total')['estonia_name']
        .tolist()
    )

    print(f"\n✓ Selected TOP 20 Estonian fabrics:")
    for i, name in enumerate(top20_estonia, 1):
        best_sim = df_main[
            (df_main['estonia_name'] == name) &
            (df_main['rank'] == 1)
        ]['similarity_total'].iloc[0]
        print(f"  {i:2d}. {name[:60]} (similarity: {best_sim:.4f})")

    # Load Estonian fabric features (we need to reload them)
    # For simplicity, we'll use the pre-computed features if available
    # Otherwise, we'd need to re-extract them
    print("\n" + "=" * 70)
    print("RE-ANALYZING WITH DIFFERENT WEIGHTS")
    print("=" * 70)
    print("⚠ Note: This requires Estonian fabric features to be stored.")
    print("   Building Estonian fabric index...")

    # Build Estonian index for TOP 20
    import torch
    import torchvision.models as models
    from torchvision import transforms

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    model = models.resnet50(pretrained=True)
    model = torch.nn.Sequential(*list(model.children())[:-1])
    model = model.to(device)
    model.eval()

    # Helper functions for feature extraction
    def get_transform(size_category):
        if size_category == '10cm':
            return transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
        else:
            return transforms.Compose([
                transforms.Resize((256, 256)),
                transforms.CenterCrop(224),
                transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])

    def extract_cnn_features(img_path, model, device, size_category):
        try:
            img = Image.open(img_path).convert('RGB')
            transform = get_transform(size_category)
            img_t = transform(img).unsqueeze(0).to(device)
            with torch.no_grad():
                features = model(img_t).squeeze().cpu().numpy()
            return features
        except Exception as e:
            return None

    def extract_lab_palette(img, n_colors=6):
        arr = np.array(img)
        if arr.ndim != 3 or arr.shape[0] < 10 or arr.shape[1] < 10:
            return None, None

        lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB).astype(np.float32)
        pixels = lab.reshape(-1, 3)

        if len(pixels) > 50000:
            idx = np.random.choice(len(pixels), 50000, replace=False)
            pixels = pixels[idx]

        K = min(n_colors, max(3, len(pixels) // 5000 + 3))
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1.0)
        _, labels, centers = cv2.kmeans(pixels, K, None, criteria, 3, cv2.KMEANS_PP_CENTERS)

        labels = labels.flatten()
        centers = centers.astype(np.float32)
        counts = np.bincount(labels, minlength=K).astype(np.float32)
        weights = counts / counts.sum()
        order = np.argsort(weights)[::-1]

        return centers[order], weights[order]

    def extract_stripe_widths(img, min_stripe_px=3):
        arr = np.array(img.convert("RGB"))
        if arr.shape[0] < 10 or arr.shape[1] < 10:
            return None

        lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB).astype(np.float32)
        L = lab[:, :, 0]
        prof = L.mean(axis=0)
        prof_s = cv2.GaussianBlur(prof.reshape(1, -1), (1, 9), 0).flatten()
        g = np.abs(np.diff(prof_s, prepend=prof_s[0]))
        thr = np.percentile(g, 90)
        edge_idx = np.where(g >= thr)[0]

        if len(edge_idx) < 2:
            return None

        edges = [int(edge_idx[0])]
        for x in edge_idx[1:]:
            if x - edges[-1] >= min_stripe_px:
                edges.append(int(x))

        if edges[0] > 0:
            edges = [0] + edges
        if edges[-1] < arr.shape[1] - 1:
            edges = edges + [arr.shape[1] - 1]

        widths = []
        for a, b in zip(edges[:-1], edges[1:]):
            w = b - a
            if w >= min_stripe_px:
                widths.append(w)

        if len(widths) < 2:
            return None

        widths = np.array(widths, dtype=np.float32)
        return widths / widths.sum()

    def extract_feature_pack(img_path, model, device, size_category):
        try:
            img = Image.open(img_path).convert("RGB")
        except:
            return None

        cnn = extract_cnn_features(img_path, model, device, size_category)
        if cnn is None:
            return None

        try:
            pal_c, pal_w = extract_lab_palette(img, n_colors=6)
        except:
            pal_c, pal_w = None, None

        try:
            sw = extract_stripe_widths(img, min_stripe_px=3)
        except:
            sw = None

        return {"cnn": cnn, "pal_c": pal_c, "pal_w": pal_w, "stripe_w": sw}

    # Extract features for TOP 20 Estonian fabrics
    estonia_packs = {}
    estonia_path_10 = os.path.join(BASE_PATH, 'Eesti_10')
    estonia_path_20 = os.path.join(BASE_PATH, 'Eesti_20')

    print("\nExtracting features for TOP 20 Estonian fabrics...")
    for estonia_name in tqdm(top20_estonia):
        # Determine size and path
        estonia_row = df_main[df_main['estonia_name'] == estonia_name].iloc[0]
        size_cat = estonia_row['estonia_size']
        path = estonia_path_10 if size_cat == '10cm' else estonia_path_20
        full_path = os.path.join(path, estonia_name)

        pack = extract_feature_pack(full_path, model, device, size_cat)
        if pack is not None:
            estonia_packs[estonia_name] = {
                'pack': pack,
                'size': size_cat
            }

    print(f"✓ Extracted features for {len(estonia_packs)} fabrics")

    # Now analyze with different weights
    print("\n" + "=" * 70)
    print("FINDING BEST MATCHES WITH EACH WEIGHT CONFIGURATION")
    print("=" * 70)

    deep_results = []

    for estonia_name, estonia_data in tqdm(estonia_packs.items(), desc="Analyzing"):
        e_pack = estonia_data['pack']
        e_size = estonia_data['size']

        # For each weight configuration
        for config_name, weights in WEIGHT_CONFIGS.items():
            # Calculate similarity with ALL English fabrics
            similarities = []

            for i, eng_pack in enumerate(eng_packs):
                sim = combined_similarity_with_weights(
                    e_pack, eng_pack,
                    w_cnn=weights['cnn'],
                    w_color=weights['color'],
                    w_ratio=weights['ratio']
                )
                similarities.append({
                    'eng_idx': i,
                    'similarity': sim
                })

            # Find best match
            best_match = max(similarities, key=lambda x: x['similarity']['total'])
            best_idx = best_match['eng_idx']
            best_sim = best_match['similarity']

            # Add to results
            deep_results.append({
                'estonia_name': estonia_name,
                'estonia_size': e_size,
                'weight_config': config_name,
                'weight_cnn': weights['cnn'],
                'weight_color': weights['color'],
                'weight_ratio': weights['ratio'],
                'england_name': eng_names[best_idx],
                'england_size': eng_sizes[best_idx],
                'similarity_total': best_sim['total'],
                'similarity_cnn': best_sim['cnn'],
                'similarity_color': best_sim['color'],
                'similarity_ratio': best_sim['ratio']
            })

    # Save results
    df_deep = pd.DataFrame(deep_results)
    output_csv = os.path.join(RESULTS_DIR, 'results_deep_analysis_top20.csv')
    df_deep.to_csv(output_csv, index=False, encoding='utf-8-sig')

    print("\n" + "=" * 70)
    print("✓ STEP 2 COMPLETE!")
    print("=" * 70)
    print(f"\nResults saved to: {output_csv}")
    print(f"Total rows: {len(df_deep)} (20 fabrics × 3 weight configs)")

    # Quick summary
    print("\n📊 QUICK SUMMARY:")
    for config_name in WEIGHT_CONFIGS.keys():
        config_data = df_deep[df_deep['weight_config'] == config_name]
        avg_sim = config_data['similarity_total'].mean()
        print(f"  {config_name}: Average similarity = {avg_sim:.4f}")

    print("\n💡 Next: Run STEP 3 script to visualize these results!")


if __name__ == "__main__":
    main()

STEP 2: DEEP WEIGHT ANALYSIS FOR TOP 20

Weight configurations:
  Original_55-25-20: CNN=0.55, COLOR=0.25, RATIO=0.2
  Color_20-60-20: CNN=0.2, COLOR=0.6, RATIO=0.2
  Pattern_20-20-60: CNN=0.2, COLOR=0.2, RATIO=0.6

LOADING DATA
✓ Loaded main results: 3450 rows
✓ Loaded England index: 3246 fabrics

✓ Selected TOP 20 Estonian fabrics:
   1. 055813_ERM_5862_055813_pisipilt.jpg (similarity: 0.8141)
   2. 057968_ERM_A560_12_057968_pisipilt.jpg (similarity: 0.8120)
   3. 059615_ERM_A927_45_059615_pisipilt.jpg (similarity: 0.8115)
   4. 058105_ERM_A751_175_058105_pisipilt.jpg (similarity: 0.8114)
   5. 056775_ERM_A446_284_056775_pisipilt.jpg (similarity: 0.8094)
   6. 203072_ERM_A227_100_203072_pisipilt.jpg (similarity: 0.8082)
   7. 058099_ERM_A794_54_058099_pisipilt.jpg (similarity: 0.8074)
   8. 057971_ERM_A555_81_057971_pisipilt.jpg (similarity: 0.8074)
   9. 058038_ERM_A36_31_058038_pisipilt.jpg (similarity: 0.8072)
  10. 052509_ERM_A509_1977_052509_pisipilt.jpg (similarity: 0.8049)
  1

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Extracting features for TOP 20 Estonian fabrics...


100%|██████████| 20/20 [00:07<00:00,  2.71it/s]


✓ Extracted features for 20 fabrics

FINDING BEST MATCHES WITH EACH WEIGHT CONFIGURATION


Analyzing: 100%|██████████| 20/20 [16:11<00:00, 48.56s/it]


✓ STEP 2 COMPLETE!

Results saved to: /content/drive/MyDrive/Masinõpe_tulemused/results_deep_analysis_top20.csv
Total rows: 60 (20 fabrics × 3 weight configs)

📊 QUICK SUMMARY:
  Original_55-25-20: Average similarity = 0.8049
  Color_20-60-20: Average similarity = 0.7553
  Pattern_20-20-60: Average similarity = 0.6233

💡 Next: Run STEP 3 script to visualize these results!


## Etapp 3. Kaalude mõju visualiseerimine

### Eesmärk

Selle etapi eesmärk on visualiseerida teises etapis saadud tulemusi.

Kui teises etapis arvutati, kuidas erinevad kaalud mõjutavad parimate vastete leidmist, siis siin tehakse tulemused visuaalselt hinnatavaks. See aitab näha, kas kaalude muutmine mõjutab ainult sarnasuse skoori või muudab ka seda, milline Inglismaa kangas valitakse Eesti kanga parimaks vasteks.

### Sisendandmed

- **Sisendfail:** `results_deep_analysis_top20.csv`
- **Päritolu:** Step 2 / Etapp 2
- **Andmetüüp:** CSV-tabel
- **Sisu:** 20 Eesti kanga tulemused kolme kaalude konfiguratsiooni korral

Lisaks kasutatakse algseid kangapilte kaustadest:

- `Eesti_10`
- `Eesti_20`
- `Inglismaa_20`
- `Inglismaa9`

Neid kasutatakse selleks, et kuvada Eesti kangas ja tema parimad Inglismaa vasted kõrvuti.

### Analüüsi loogika

Selles etapis ei arvutata enam kangaste tunnuseid uuesti. Kood loeb sisse juba valmis tulemuste tabeli ja koostab selle põhjal graafikud ning võrdluspildid.

Visualiseerimine keskendub neljale küsimusele:

1. Kas erinevad kaalud annavad erineva keskmise sarnasusskoori?
2. Millised sarnasuse komponendid on üldiselt tugevamad või nõrgemad?
3. Kas sama Eesti kangas saab eri kaaludega sama või erineva Inglismaa vaste?
4. Kuidas muutused visuaalselt välja näevad, kui kangapilte kõrvuti võrrelda?

### Peamised tegevused

Selles etapis tehakse järgmised tegevused:

1. laaditakse sisse `results_deep_analysis_top20.csv`;
2. kontrollitakse, mitu Eesti kangast ja kaalude konfiguratsiooni failis on;
3. arvutatakse keskmised sarnasusskoorid kaalude konfiguratsioonide kaupa;
4. võrreldakse CNN-, värvi- ja mustrisarnasuse komponente;
5. hinnatakse, kas parim Inglismaa vaste jääb kaalude muutmisel samaks või muutub;
6. luuakse koondgraafik kaalude mõju hindamiseks;
7. luuakse üksikute kangaste kõrvutavad võrdluspildid;
8. luuakse komponentide heatmap;
9. luuakse vaste muutumise/stabiilsuse analüüs;
10. salvestatakse kõik visualiseeringud tulemuste kausta.

### Loodavad visualiseeringud

Selle etapi käigus luuakse mitu väljundpilti.

#### 1. `weight_impact_overview.png`

Koondvisualiseering, mis näitab:

- keskmist sarnasusskoori iga kaalude konfiguratsiooni korral;
- CNN-, värvi- ja mustrikomponentide keskmisi skoore;
- kui sageli jääb parim vaste samaks või muutub;
- sarnasusskooride jaotust eri kaalude korral.

Seda kasutatakse üldise ülevaate saamiseks.

#### 2. `weight_comparison_*.png`

Üksikute kangaste võrdluspildid.

Igal pildil kuvatakse:

- vasakul Eesti kangas;
- kõrval kolm Inglismaa vastet:
  - algsete kaaludega leitud vaste;
  - värvile keskenduva kaaluga leitud vaste;
  - mustrile keskenduva kaaluga leitud vaste.

Need pildid aitavad visuaalselt hinnata, kas arvutuslikult leitud erinevused on ka inimese silmale mõistetavad.

#### 3. `component_heatmap.png`

Heatmap, mis näitab CNN-, värvi- ja mustrikomponentide skoore valitud kangaste lõikes.

Seda kasutatakse selleks, et näha, millistes komponentides on sarnasused tugevamad ja millistes nõrgemad.

#### 4. `match_change_analysis.png`

Analüüs, mis näitab, millistel Eesti kangastel jäi parim Inglismaa vaste kõigi kaalude korral samaks ja millistel juhtudel see muutus.

See aitab eristada:

- **stabiilseid vasteid**, kus üks Inglismaa kangas on sarnane mitme mõõdiku järgi;
- **muutuvaid vasteid**, kus parim vaste sõltub sellest, kas eelistada värvi, mustrit või üldist visuaalset muljet.

### Väljundid

Selle etapi väljunditeks on pildifailid tulemuste kaustas:

- `weight_impact_overview.png`
- `weight_comparison_*.png`
- `component_heatmap.png`
- `match_change_analysis.png`

Need väljundid ei ole ainult tehnilised vahefailid, vaid neid saab kasutada ka uurimistulemuste tõlgendamisel ja esitlemisel.

### Ajahinnang

See samm on võrreldes esimese ja teise etapiga kiirem, sest uusi süvatunnuseid ei arvutata.

Ajakulu sõltub peamiselt:

- visualiseeritavate kangaste arvust;
- sellest, kas algsed pildifailid leitakse õigetest kaustadest;
- salvestatavate pildifailide arvust ja resolutsioonist.

Tavaliselt on see pigem kiire analüüsi- ja visualiseerimissamm.

### Nähtavad vaheväljundid

Notebookis kuvatakse:

- kas sisendfail leiti üles;
- mitu rida tulemuste tabelis on;
- mitu Eesti kangast analüüsis on;
- mitu kaalude konfiguratsiooni kasutatakse;
- millised visualiseeringud on loodud;
- kuhu failid salvestati.

Need vaheväljundid aitavad kontrollida, kas visualiseerimine põhineb õigel tulemuste failil.

### Tõlgendamise märkus

Selle etapi tulemused aitavad hinnata, kas arvutuslik sarnasus on sisuliselt mõistetav.

Kui sama Eesti kangas saab kõigi kaalude korral sama Inglismaa vaste, võib seda pidada tugevaks ja stabiilseks vasteks. Kui vaste muutub, tähendab see, et kanga sarnasus sõltub sellest, millist tunnust mudelile olulisemaks pidada.

Oluline on, et visualiseeringuid ei kasutata ainult tulemuste kaunistamiseks. Need on vajalikud selleks, et uurija saaks hinnata, kas mudeli valikud on visuaalselt põhjendatud ja kas valitud kaalude kombinatsioon sobib uurimisküsimusega.

In [ ]:
"""
STEP 3: WEIGHT COMPARISON VISUALIZATION
========================================
Visualizes how different weight configurations affect matching results
for the TOP 20 Estonian fabrics.

INPUT:
- results_deep_analysis_top20.csv (from Step 2)

OUTPUT:
- Various visualizations showing weight impact
- Side-by-side fabric comparisons
- Statistical analyses

FAST: Just reads CSV and creates plots!
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

# ============================================================
# SETTINGS
# ============================================================
BASE_PATH = "/content/drive/MyDrive/Kangad"
RESULTS_DIR = "/content/drive/MyDrive/Masinõpe_tulemused"

print("=" * 70)
print("STEP 3: WEIGHT COMPARISON VISUALIZATION")
print("=" * 70)


# ============================================================
# 1. WEIGHT IMPACT OVERVIEW
# ============================================================
def create_weight_impact_overview(df, out_dir):
    """
    Shows how changing weights affects similarity scores and matches.
    """
    print("\n" + "=" * 70)
    print("CREATING WEIGHT IMPACT OVERVIEW")
    print("=" * 70)

    fig, axes = plt.subplots(2, 2, figsize=(18, 14))

    # 1. Average similarity by weight configuration
    config_stats = df.groupby('weight_config')['similarity_total'].agg(['mean', 'std']).reset_index()

    ax = axes[0, 0]
    x_pos = np.arange(len(config_stats))
    ax.bar(x_pos, config_stats['mean'], yerr=config_stats['std'],
           capsize=5, alpha=0.7, color=['#1f77b4', '#ff7f0e', '#2ca02c'], edgecolor='black')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(config_stats['weight_config'], rotation=15, ha='right')
    ax.set_ylabel('Average Similarity', fontsize=12, fontweight='bold')
    ax.set_title('Average Similarity by Weight Configuration', fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0.7, 1.0])
    for i, (mean, std) in enumerate(zip(config_stats['mean'], config_stats['std'])):
        ax.text(i, mean + std + 0.01, f'{mean:.3f}', ha='center', fontweight='bold')

    # 2. Component contribution distribution
    ax = axes[0, 1]
    components = ['similarity_cnn', 'similarity_color', 'similarity_ratio']
    comp_means = [df[comp].mean() for comp in components]
    comp_labels = ['CNN\n(Texture)', 'COLOR\n(LAB)', 'RATIO\n(Stripes)']

    ax.bar(comp_labels, comp_means, alpha=0.7,
           color=['#1f77b4', '#ff7f0e', '#2ca02c'], edgecolor='black')
    ax.set_ylabel('Average Component Score', fontsize=12, fontweight='bold')
    ax.set_title('Average Component Scores (All Configs)', fontsize=14, fontweight='bold')
    ax.set_ylim([0, 1])
    ax.grid(axis='y', alpha=0.3)
    for i, v in enumerate(comp_means):
        ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

    # 3. Match consistency: How many times does the same English fabric win?
    ax = axes[1, 0]
    match_changes = []
    for estonia_name in df['estonia_name'].unique():
        fabric_data = df[df['estonia_name'] == estonia_name]
        unique_matches = fabric_data['england_name'].nunique()
        match_changes.append(unique_matches)

    consistency_counts = pd.Series(match_changes).value_counts().sort_index()

    ax.bar(consistency_counts.index, consistency_counts.values,
           alpha=0.7, color='steelblue', edgecolor='black')
    ax.set_xlabel('Number of Different English Matches', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Estonian Fabrics', fontsize=12, fontweight='bold')
    ax.set_title('Match Consistency Across Weight Configurations', fontsize=14, fontweight='bold')
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(['Same match\n(all 3 configs)', '2 different\nmatches', '3 different\nmatches'])
    ax.grid(axis='y', alpha=0.3)
    for i, (x, y) in enumerate(zip(consistency_counts.index, consistency_counts.values)):
        ax.text(x, y + 0.3, str(y), ha='center', fontweight='bold', fontsize=12)

    # 4. Similarity distribution by configuration
    ax = axes[1, 1]
    configs = df['weight_config'].unique()
    data_by_config = [df[df['weight_config'] == config]['similarity_total'].values
                      for config in configs]

    bp = ax.boxplot(data_by_config, labels=configs, patch_artist=True,
                    showmeans=True, meanline=True)
    for patch, color in zip(bp['boxes'], ['#1f77b4', '#ff7f0e', '#2ca02c']):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_xticklabels(configs, rotation=15, ha='right')
    ax.set_ylabel('Similarity Score', fontsize=12, fontweight='bold')
    ax.set_title('Similarity Distribution by Configuration', fontsize=14, fontweight='bold')
    ax.set_ylim([0.7, 1.0])
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    output_path = os.path.join(out_dir, 'weight_impact_overview.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"✓ Saved: {output_path}")


# ============================================================
# 2. INDIVIDUAL FABRIC COMPARISON
# ============================================================
def create_fabric_weight_comparison(df, out_dir, top_n=10):
    """
    Shows for each fabric how the best match changes with different weights.
    """
    print("\n" + "=" * 70)
    print(f"CREATING INDIVIDUAL FABRIC COMPARISONS (Top {top_n})")
    print("=" * 70)

    # Get top N fabrics by original weight similarity
    original_config = df[df['weight_config'].str.contains('Original')].copy()
    top_fabrics = original_config.nlargest(top_n, 'similarity_total')['estonia_name'].tolist()

    estonia_path_10 = os.path.join(BASE_PATH, 'Eesti_10')
    estonia_path_20 = os.path.join(BASE_PATH, 'Eesti_20')
    england_path_20 = os.path.join(BASE_PATH, 'Inglismaa_20')
    england_path_10 = os.path.join(BASE_PATH, 'Inglismaa9')

    for estonia_name in tqdm(top_fabrics, desc="Creating comparisons"):
        fabric_data = df[df['estonia_name'] == estonia_name].sort_values('weight_config')

        if len(fabric_data) == 0:
            continue

        # Create figure: 1 Estonian + 3 English matches
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))

        # Estonian fabric (left)
        estonia_row = fabric_data.iloc[0]
        estonia_path = estonia_path_10 if estonia_row['estonia_size'] == '10cm' else estonia_path_20
        estonia_full_path = os.path.join(estonia_path, estonia_name)

        try:
            estonia_img = Image.open(estonia_full_path)
            axes[0].imshow(estonia_img)
            axes[0].axis('off')
            axes[0].set_title(
                f"🇪🇪 ESTONIAN FABRIC\n{estonia_name[:40]}\n({estonia_row['estonia_size']})",
                fontsize=11, fontweight='bold', color='#1f77b4'
            )

            # Add thick border
            for spine in axes[0].spines.values():
                spine.set_edgecolor('#1f77b4')
                spine.set_linewidth(4)
                spine.set_visible(True)
        except:
            axes[0].text(0.5, 0.5, "Image\nnot found", ha='center', va='center')
            axes[0].axis('off')

        # Three weight configurations
        for idx, (_, row) in enumerate(fabric_data.iterrows(), 1):
            england_path = england_path_10 if row['england_size'] == '10cm' else england_path_20
            england_full_path = os.path.join(england_path, row['england_name'])

            try:
                england_img = Image.open(england_full_path)
                axes[idx].imshow(england_img)
                axes[idx].axis('off')

                # Color by configuration
                colors = {'Original': '#1f77b4', 'Color': '#ff7f0e', 'Pattern': '#2ca02c'}
                config_short = row['weight_config'].split('_')[0]
                title_color = colors.get(config_short, 'black')

                # Weights display
                weights_str = f"{int(row['weight_cnn']*100)}-{int(row['weight_color']*100)}-{int(row['weight_ratio']*100)}"

                axes[idx].set_title(
                    f"🇬🇧 {config_short.upper()} WEIGHTS\n"
                    f"({weights_str})\n"
                    f"{row['england_name'][:35]}\n"
                    f"Score: {row['similarity_total']:.3f}",
                    fontsize=9, fontweight='bold', color=title_color
                )

                # Component scores
                comp_text = f"CNN:{row['similarity_cnn']:.2f} | COLOR:{row['similarity_color']:.2f} | RATIO:{row['similarity_ratio']:.2f}"
                axes[idx].text(0.5, -0.08, comp_text, transform=axes[idx].transAxes,
                              fontsize=7, ha='center', color='#666')

                # Border
                for spine in axes[idx].spines.values():
                    spine.set_edgecolor(title_color)
                    spine.set_linewidth(3)
                    spine.set_visible(True)

            except Exception as e:
                axes[idx].text(0.5, 0.5, "Image\nnot found", ha='center', va='center')
                axes[idx].axis('off')

        plt.suptitle(
            f"Weight Configuration Impact on Match Selection",
            fontsize=16, fontweight='bold', y=0.98
        )
        plt.tight_layout(rect=[0, 0, 1, 0.96])

        safe_filename = estonia_name.replace('/', '_').replace(' ', '_').replace('\\', '_')[:50]
        output_path = os.path.join(out_dir, f'weight_comparison_{safe_filename}.png')
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()

    print(f"✓ Created {len(top_fabrics)} fabric comparison images")


# ============================================================
# 3. COMPONENT HEATMAP
# ============================================================
def create_component_heatmap(df, out_dir):
    """
    Heatmap showing how CNN, COLOR, and RATIO scores vary across fabrics.
    """
    print("\n" + "=" * 70)
    print("CREATING COMPONENT HEATMAP")
    print("=" * 70)

    # Pivot data for heatmap
    fabrics = df['estonia_name'].unique()[:15]  # Top 15 for readability
    configs = df['weight_config'].unique()

    fig, axes = plt.subplots(1, 3, figsize=(22, 10))
    components = ['similarity_cnn', 'similarity_color', 'similarity_ratio']
    titles = ['CNN (Texture) Scores', 'COLOR (LAB Palette) Scores', 'RATIO (Stripe Pattern) Scores']

    for idx, (component, title) in enumerate(zip(components, titles)):
        matrix = []
        for fabric in fabrics:
            row = []
            for config in configs:
                value = df[
                    (df['estonia_name'] == fabric) &
                    (df['weight_config'] == config)
                ][component].values
                row.append(value[0] if len(value) > 0 else 0)
            matrix.append(row)

        matrix = np.array(matrix)

        sns.heatmap(
            matrix,
            ax=axes[idx],
            xticklabels=configs,
            yticklabels=[name[:30] for name in fabrics],
            cmap='YlOrRd',
            vmin=0,
            vmax=1,
            cbar_kws={'label': 'Score'},
            annot=True,
            fmt='.2f',
            linewidths=0.5
        )

        axes[idx].set_title(title, fontsize=14, fontweight='bold', pad=15)
        axes[idx].set_xlabel('Weight Configuration', fontsize=11, fontweight='bold')
        if idx == 0:
            axes[idx].set_ylabel('Estonian Fabric', fontsize=11, fontweight='bold')
        else:
            axes[idx].set_ylabel('')

    plt.tight_layout()
    output_path = os.path.join(out_dir, 'component_heatmap.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"✓ Saved: {output_path}")


# ============================================================
# 4. MATCH CHANGE ANALYSIS
# ============================================================
def create_match_change_analysis(df, out_dir):
    """
    Analyzes which fabrics have stable vs changing matches.
    """
    print("\n" + "=" * 70)
    print("CREATING MATCH CHANGE ANALYSIS")
    print("=" * 70)

    match_stability = []

    for estonia_name in df['estonia_name'].unique():
        fabric_data = df[df['estonia_name'] == estonia_name]

        matches = fabric_data['england_name'].tolist()
        unique_matches = len(set(matches))

        original_match = fabric_data[fabric_data['weight_config'].str.contains('Original')]['england_name'].iloc[0]
        color_match = fabric_data[fabric_data['weight_config'].str.contains('Color')]['england_name'].iloc[0]
        pattern_match = fabric_data[fabric_data['weight_config'].str.contains('Pattern')]['england_name'].iloc[0]

        match_stability.append({
            'estonia_name': estonia_name[:40],
            'unique_matches': unique_matches,
            'original_match': original_match[:30],
            'color_match': color_match[:30],
            'pattern_match': pattern_match[:30],
            'all_same': unique_matches == 1
        })

    stability_df = pd.DataFrame(match_stability)

    # Create visualization
    fig, axes = plt.subplots(2, 1, figsize=(18, 12))

    # 1. Table showing changes
    stable = stability_df[stability_df['all_same'] == True]
    unstable = stability_df[stability_df['all_same'] == False].head(10)

    ax = axes[0]
    ax.axis('tight')
    ax.axis('off')

    table_data = []
    table_data.append(['Estonian Fabric', 'Original Match', 'Color Match', 'Pattern Match', 'Status'])

    for _, row in stable.head(5).iterrows():
        table_data.append([
            row['estonia_name'],
            row['original_match'],
            row['color_match'],
            row['pattern_match'],
            '✓ Stable'
        ])

    for _, row in unstable.iterrows():
        table_data.append([
            row['estonia_name'],
            row['original_match'],
            row['color_match'],
            row['pattern_match'],
            '⚠ Changes'
        ])

    table = ax.table(cellText=table_data, cellLoc='left', loc='center',
                    colWidths=[0.25, 0.20, 0.20, 0.20, 0.15])
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 2)

    # Header styling
    for i in range(5):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(weight='bold', color='white')

    # Row styling
    for i in range(1, len(table_data)):
        for j in range(5):
            if i <= 5:
                table[(i, j)].set_facecolor('#E7E6E6')
            else:
                table[(i, j)].set_facecolor('#FFF2CC')

    ax.set_title('Match Stability: Stable vs Changing Matches',
                fontsize=16, fontweight='bold', pad=20)

    # 2. Statistics
    ax = axes[1]
    stats_text = f"""
MATCH STABILITY STATISTICS:
{'='*60}

Total fabrics analyzed: {len(stability_df)}

Stable matches (same English fabric for all weight configs): {len(stable)} ({len(stable)/len(stability_df)*100:.1f}%)
Changing matches (different English fabrics): {len(unstable)} ({len(unstable)/len(stability_df)*100:.1f}%)

INTERPRETATION:
- Stable matches indicate the Estonian fabric has ONE clear English counterpart
  that dominates across all similarity measures (texture, color, pattern)

- Changing matches indicate the Estonian fabric has DIFFERENT best matches
  depending on which aspect is prioritized:
  * Color-focused weights → fabric with most similar color palette
  * Pattern-focused weights → fabric with most similar stripe pattern
    """

    ax.text(0.1, 0.5, stats_text, fontsize=11, family='monospace',
           verticalalignment='center', bbox=dict(boxstyle='round',
           facecolor='wheat', alpha=0.5))
    ax.axis('off')

    plt.tight_layout()
    output_path = os.path.join(out_dir, 'match_change_analysis.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"✓ Saved: {output_path}")


# ============================================================
# MAIN
# ============================================================
def main():
    # Load data
    csv_path = os.path.join(RESULTS_DIR, 'results_deep_analysis_top20.csv')

    if not os.path.exists(csv_path):
        print(f"❌ ERROR: {csv_path} not found!")
        print("Please run STEP 2 first!")
        return

    print("\n" + "=" * 70)
    print("LOADING DATA")
    print("=" * 70)

    df = pd.read_csv(csv_path)
    print(f"✓ Loaded: {len(df)} rows")
    print(f"  {len(df['estonia_name'].unique())} Estonian fabrics")
    print(f"  {len(df['weight_config'].unique())} weight configurations")

    # Create all visualizations
    create_weight_impact_overview(df, RESULTS_DIR)
    create_fabric_weight_comparison(df, RESULTS_DIR, top_n=10)
    create_component_heatmap(df, RESULTS_DIR)
    create_match_change_analysis(df, RESULTS_DIR)

    # Summary
    print("\n" + "=" * 70)
    print("✓ STEP 3 COMPLETE!")
    print("=" * 70)
    print(f"\nAll visualizations saved to: {RESULTS_DIR}")
    print("\nGenerated files:")
    print("  1. weight_impact_overview.png - Overall weight effects")
    print("  2. weight_comparison_*.png - Individual fabric comparisons (10 files)")
    print("  3. component_heatmap.png - Component scores across fabrics")
    print("  4. match_change_analysis.png - Match stability analysis")
    print("\n✨ Analysis complete! Check your results folder.")


if __name__ == "__main__":
    main()

STEP 3: WEIGHT COMPARISON VISUALIZATION

LOADING DATA
✓ Loaded: 60 rows
  20 Estonian fabrics
  3 weight configurations

CREATING WEIGHT IMPACT OVERVIEW


/tmp/ipython-input-2937764585.py:108: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_by_config, labels=configs, patch_artist=True,


✓ Saved: /content/drive/MyDrive/Masinõpe_tulemused/weight_impact_overview.png

CREATING INDIVIDUAL FABRIC COMPARISONS (Top 10)


Creating comparisons:   0%|          | 0/10 [00:00<?, ?it/s]/tmp/ipython-input-2937764585.py:225: UserWarning: Glyph 127466 (\N{REGIONAL INDICATOR SYMBOL LETTER E}) missing from font(s) DejaVu Sans.
  plt.tight_layout(rect=[0, 0, 1, 0.96])
/tmp/ipython-input-2937764585.py:225: UserWarning: Glyph 127468 (\N{REGIONAL INDICATOR SYMBOL LETTER G}) missing from font(s) DejaVu Sans.
  plt.tight_layout(rect=[0, 0, 1, 0.96])
/tmp/ipython-input-2937764585.py:225: UserWarning: Glyph 127463 (\N{REGIONAL INDICATOR SYMBOL LETTER B}) missing from font(s) DejaVu Sans.
  plt.tight_layout(rect=[0, 0, 1, 0.96])
/tmp/ipython-input-2937764585.py:229: UserWarning: Glyph 127466 (\N{REGIONAL INDICATOR SYMBOL LETTER E}) missing from font(s) DejaVu Sans.
  plt.savefig(output_path, dpi=300, bbox_inches='tight')
/tmp/ipython-input-2937764585.py:229: UserWarning: Glyph 127468 (\N{REGIONAL INDICATOR SYMBOL LETTER G}) missing from font(s) DejaVu Sans.
  plt.savefig(output_path, dpi=300, bbox_inches='tight')
/tmp/ipy

✓ Created 10 fabric comparison images

CREATING COMPONENT HEATMAP
✓ Saved: /content/drive/MyDrive/Masinõpe_tulemused/component_heatmap.png

CREATING MATCH CHANGE ANALYSIS
✓ Saved: /content/drive/MyDrive/Masinõpe_tulemused/match_change_analysis.png

✓ STEP 3 COMPLETE!

All visualizations saved to: /content/drive/MyDrive/Masinõpe_tulemused

Generated files:
  1. weight_impact_overview.png - Overall weight effects
  2. weight_comparison_*.png - Individual fabric comparisons (10 files)
  3. component_heatmap.png - Component scores across fabrics
  4. match_change_analysis.png - Match stability analysis

✨ Analysis complete! Check your results folder.


## Etapp 4. Eesti ja Inglismaa kangaste sarnasuste statistiline analüüs

### Eesmärk

Selle etapi eesmärk on analüüsida esimese sammu tulemuste põhjal Eesti ja Inglismaa kangaste sarnasusi üldisemal tasandil.

Kui esimeses etapis leiti iga Eesti kanga jaoks parimad Inglismaa vasted, siis siin uuritakse, mida need tulemused tervikuna näitavad. Eesmärk on hinnata, kas kõrged sarnasused on pigem üksikud juhuslikud kokkulangevused või viitavad need süstemaatilisematele mustritele.

### Uurimisküsimused

Selles etapis keskendutakse järgmistele küsimustele:

1. Kas kõrged sarnasusskoorid on haruldased või esinevad need suure osa kangaste puhul?
2. Kas teatud Eesti kangad sarnanevad Inglismaa kangastega järjepidevalt tugevamalt?
3. Kas kanga suuruskategooria mõjutab sarnasust?
4. Kas tulemused viitavad sarnaste kangaste rühmadele või klastritele?
5. Milline komponent — CNN, värv või triibumuster — on kogusarnasuse seisukohalt kõige olulisem?

### Sisendandmed

- **Sisendfail:** `results_similarity_main.csv`
- **Päritolu:** Step 1 / Etapp 1
- **Andmetüüp:** CSV-tabel
- **Sisu:** iga Eesti kanga viis parimat Inglismaa vastet

Tabelis kasutatakse peamiselt järgmisi veerge:

- `estonia_name`
- `estonia_size`
- `rank`
- `england_name`
- `england_size`
- `similarity_total`
- `similarity_cnn`
- `similarity_color`
- `similarity_ratio`

Statistilises analüüsis kasutatakse eelkõige **TOP 1 vasteid**, et iga Eesti kangas oleks analüüsis esindatud ühe parima Inglismaa vastega.

### Analüüsi loogika

Selles etapis ei arvutata pilditunnuseid uuesti. Analüüs põhineb esimese etapi tulemuste tabelil.

Kõigepealt valitakse igale Eesti kangale tema parim ehk `rank == 1` vaste. Seejärel hinnatakse sarnasuse jaotust, komponentide panust ning üldist mustrit.

Analüüsis kasutatakse sarnasuse tõlgendamiseks järgmisi lävendeid:

| Tase | Lävend | Tõlgendus |
|---|---:|---|
| Väga kõrge | ≥ 0.90 | Peaaegu identsed või väga tugev visuaalne sarnasus |
| Kõrge | ≥ 0.85 | Tugev sarnasus |
| Keskmine | ≥ 0.75 | Mõõdukas sarnasus |
| Madal | < 0.75 | Nõrk või juhuslik sarnasus |

Need lävendid on uurija määratud tõlgenduslikud piirid. Need ei tõesta automaatselt ühist algupära, vaid aitavad tulemusi süstemaatiliselt rühmitada.

### Peamised tegevused

Selles etapis tehakse järgmised tegevused:

1. laaditakse sisse `results_similarity_main.csv`;
2. kontrollitakse, mitu Eesti ja Inglismaa kangast tulemuste failis esineb;
3. valitakse välja iga Eesti kanga parim vaste;
4. visualiseeritakse parimate vastete sarnasusskooride jaotus;
5. arvutatakse keskmine, mediaan, standardhälve ja protsentiilid;
6. hinnatakse, kui suur osa kangastest ületab kõrge sarnasuse lävendeid;
7. võrreldakse CNN-, värvi- ja triibumustri komponentide jaotusi;
8. arvutatakse komponentide korrelatsioonid kogusarnasusega;
9. hinnatakse, milline komponent ennustab kogusarnasust kõige tugevamalt;
10. koostatakse lõplik koondhinnang sarnasuste mustrite kohta;
11. salvestatakse kõik graafikud PNG-failidena.

### Loodavad visualiseeringud

#### 1. `1_sarnasuse_jaotus_histogramm.png`

Histogramm näitab, kuidas parimate vastete sarnasusskoorid jaotuvad.

Seda kasutatakse selleks, et hinnata, kas tulemused koonduvad pigem kõrgete või madalate väärtuste juurde.

#### 2. `2_kumulatiivne_jaotus.png`

Kumulatiivne jaotus näitab, mitu protsenti Eesti kangastest jõuab teatud sarnasusskoorini.

See aitab hinnata, kui tavalised või haruldased on kõrged sarnasused.

#### 3. `3_normaaljaotuse_test.png`

Q-Q graafik ja Shapiro-Wilki test annavad ülevaate sellest, kas sarnasusskooride jaotus meenutab normaaljaotust.

See ei tõesta üksinda juhuslikkust ega mittejuhuslikkust, kuid aitab hinnata jaotuse kuju.

#### 4. `4_kategooriate_jaotus.png`

Sektordiagramm näitab, kui suur osa kangastest kuulub väga kõrge, kõrge, keskmise või madala sarnasuse kategooriasse.

#### 5. `5_komponentide_jaotused.png`

Boxplot võrdleb CNN-, värvi- ja triibumustri skooride jaotust.

Selle abil saab näha, milline komponent annab üldiselt kõrgemaid või madalamaid skoore.

#### 6. `6_komponentide_korrelatsioonid.png`

Korrelatsioonimaatriks näitab, kuidas üksikud komponendid on seotud kogusarnasusega.

#### 7. `7_komponentide_ennustusvõime.png`

Hajuvusdiagramm näitab iga komponendi seost kogusarnasusega ning lisab ligikaudse R² väärtuse.

Seda kasutatakse selleks, et hinnata, milline komponent on kogusarnasuse suhtes kõige ennustavam.

#### 8. `8_domineeriv_komponent.png`

Tulpdiagramm näitab, milline komponent on iga kanga puhul kõrgeima skooriga.

See aitab mõista, kas tulemused põhinevad rohkem üldisel visuaalsel sarnasusel, värvidel või triibumustril.

#### 9. `9_LOPLIK_HINNANG.png`

Koondvisualiseering, mis võtab kokku peamised statistilised tulemused ja annab tõlgendusliku hinnangu.

Seda kasutatakse töövoo põhijärelduse esitamiseks.

### Väljundid

Selle etapi väljunditeks on PNG-failid tulemuste kaustas:

- `1_sarnasuse_jaotus_histogramm.png`
- `2_kumulatiivne_jaotus.png`
- `3_normaaljaotuse_test.png`
- `4_kategooriate_jaotus.png`
- `5_komponentide_jaotused.png`
- `6_komponentide_korrelatsioonid.png`
- `7_komponentide_ennustusvõime.png`
- `8_domineeriv_komponent.png`
- `9_LOPLIK_HINNANG.png`

Need väljundid on mõeldud nii analüüsi kontrollimiseks kui ka tulemuste tõlgendamiseks ja esitlemiseks.

### Ajahinnang

See samm on kiirem kui pilditunnuste arvutamise etapid, sest analüüs põhineb juba valmis CSV-failil.

Ajakulu sõltub peamiselt:

- tulemuste tabeli suurusest;
- loodavate graafikute arvust;
- sellest, kas kõik vajalikud Python-teegid on keskkonnas olemas;
- pildifailide salvestamise kiirusest Google Drive’i.

### Nähtavad vaheväljundid

Notebookis kuvatakse:

- kas sisendfail leiti üles;
- mitu rida tulemuste tabelis on;
- mitu Eesti kangast ja Inglismaa kangast analüüsis esineb;
- millised graafikud salvestati;
- mitu kangast ületas määratud sarnasuse lävendeid;
- komponentide korrelatsioonid kogusarnasusega;
- lõplik tekstiline hinnang.

Need väljundid aitavad kontrollida, kas statistiline analüüs põhineb õigel sisendfailil ja kas visualiseeringud loodi edukalt.

### Tõlgendamise märkus

Selle etapi tulemusi tuleb tõlgendada ettevaatlikult.

Kõrge sarnasusskoor ei tähenda automaatselt, et kahel kangal on tõendatult ühine algupära. See tähendab, et mudel leidis nende vahel tugeva visuaalse sarnasuse valitud tunnuste ja kaalude põhjal.

Seetõttu on selles etapis kasutatud väljendid nagu “süstemaatiline muster” või “võimalik ühine traditsioon” tõlgenduslikud, mitte lõplikud ajaloolised tõendid. Tulemused sobivad eelkõige edasiseks uurimiseks, visuaalseks võrdlemiseks ja hüpoteeside loomiseks.

In [ ]:
"""
KANGASTE STATISTILINE ANALÜÜS JA ALGUPÄRA HINDAMINE
====================================================
Analüüsib eesti ja inglise kangaste sarnasusi ning hindab,
kas tulemused viitavad ühisele päritolule või juhuslikele kokkusattumusele.

UURIMISKÜSIMUSED:
1. Kas kõrged sarnasused on haruldased (juhuslik) või tavalised (süstemaatiline)?
2. Kas teatud eesti kangad sarnanevad järjepidevalt hästi?
3. Kas suuruste kombinatsioonid mõjutavad sarnasust?
4. Kas eksisteerivad sarnaste kangaste "klastrid"?
5. Milline komponent (CNN/VÄRV/TRIIP) on kõige ennustavam?

SISEND:
- results_similarity_main.csv (sarnasuste tabel)

VÄLJUND:
- Eraldi PNG failid igale analüüsile
- Jaotusanalüüsid
- Lõplik hinnang juhuslike vs süstemaatiliste mustrite kohta
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from tqdm import tqdm

# Eesti keele teksti jaoks
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# ============================================================
# SEADED
# ============================================================
RESULTS_DIR = "/content/drive/MyDrive/Masinõpe_tulemused"

# Lävendid tõlgendamiseks
KYNNIS_VAGA_KORGE = 0.90  # Peaaegu identsed
KYNNIS_KORGE = 0.85       # Tugev sarnasus
KYNNIS_KESKMINE = 0.75    # Mõõdukas sarnasus
KYNNIS_MADAL = 0.65       # Nõrk sarnasus

print("=" * 70)
print("KANGASTE STATISTILINE ANALÜÜS JA ALGUPÄRA HINDAMINE")
print("=" * 70)
print("\nUURIMISEESMÄRK: Kas sarnasused on juhuslikud või süstemaatilised?")
print(f"\nSarnasuse lävended:")
print(f"  Väga kõrge (≥{KYNNIS_VAGA_KORGE}): Peaaegu identsed")
print(f"  Kõrge (≥{KYNNIS_KORGE}): Tugev sarnasus, tõenäoliselt ühine algupära")
print(f"  Keskmine (≥{KYNNIS_KESKMINE}): Mõõdukas sarnasus")
print(f"  Madal (<{KYNNIS_KESKMINE}): Nõrk või juhuslik sarnasus")


# ============================================================
# 1. ÜLDINE SARNASUSE JAOTUS
# ============================================================
def loo_yldine_jaotus(df, valjund_kaust):
    """
    Näitab, kas sarnasused koonduvad teatud väärtuste ümber või on ühtlaselt jaotunud.
    Juhuslik = ühtlane jaotus. Süstemaatiline = koondumine kõrgete väärtuste juurde.
    """
    print("\n" + "=" * 70)
    print("1. ÜLDINE SARNASUSE JAOTUS")
    print("=" * 70)

    # Võta TOP-1 vasted iga eesti kanga kohta
    top1 = df[df['rank'] == 1].copy()

    # === 1.1 Histogramm lävenditega ===
    fig, ax = plt.subplots(figsize=(14, 10))

    n, bins, patches = ax.hist(top1['similarity_total'], bins=50,
                                edgecolor='black', alpha=0.7, color='steelblue')

    # Värvi tulbad lävendite järgi
    for i, patch in enumerate(patches):
        if bins[i] >= KYNNIS_VAGA_KORGE:
            patch.set_facecolor('#2ca02c')  # Roheline
        elif bins[i] >= KYNNIS_KORGE:
            patch.set_facecolor('#ff7f0e')  # Oranž
        elif bins[i] >= KYNNIS_KESKMINE:
            patch.set_facecolor('#1f77b4')  # Sinine
        else:
            patch.set_facecolor('#d62728')  # Punane

    # Lävendi jooned
    ax.axvline(KYNNIS_VAGA_KORGE, color='darkgreen', linestyle='--',
              linewidth=2.5, label=f'Väga kõrge ({KYNNIS_VAGA_KORGE})')
    ax.axvline(KYNNIS_KORGE, color='darkorange', linestyle='--',
              linewidth=2.5, label=f'Kõrge ({KYNNIS_KORGE})')
    ax.axvline(KYNNIS_KESKMINE, color='darkblue', linestyle='--',
              linewidth=2.5, label=f'Keskmine ({KYNNIS_KESKMINE})')

    keskmine = top1['similarity_total'].mean()
    mediaan = top1['similarity_total'].median()

    # Keskmise ja mediaani jooned
    line1 = ax.axvline(keskmine, color='red', linestyle='-', linewidth=3,
                       label=f'Keskmine ({keskmine:.3f})')
    line2 = ax.axvline(mediaan, color='purple', linestyle='-', linewidth=3,
                       label=f'Mediaan ({mediaan:.3f})')

    ax.set_xlabel('Sarnasuse skoor (parim vaste eesti kanga kohta)',
                  fontsize=14, fontweight='bold')
    ax.set_ylabel('Eesti kangaste arv', fontsize=14, fontweight='bold')
    ax.set_title('PARIMATE VASTETE JAOTUS\nKas sarnasus on juhuslik või koondunud?',
                fontsize=16, fontweight='bold')

    # X-telg 0.5-1.0 (50-100%)
    ax.set_xlim([0.5, 1.0])
    ax.set_xticks([0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0])
    ax.set_xticklabels(['50', '55', '60', '65', '70', '75', '80', '85', '90', '95', '100'])

    # Kaks eraldi legendi - üks üleval vasakul, teine all paremal
    legend1 = ax.legend(loc='upper left', fontsize=11, title='Lävended',
                       framealpha=0.95, edgecolor='black')
    ax.add_artist(legend1)
    ax.legend([line1, line2], [f'Keskmine ({keskmine:.3f})', f'Mediaan ({mediaan:.3f})'],
             loc='lower right', fontsize=11, title='Keskväärtused',
             framealpha=0.95, edgecolor='black')

    ax.grid(axis='y', alpha=0.3)

    # Statistika tekstikast
    stats_tekst = f"""
n = {len(top1)}
Keskmine: {keskmine:.4f}
Mediaan: {mediaan:.4f}
Standardhälve: {top1['similarity_total'].std():.4f}

Väga kõrge (≥{KYNNIS_VAGA_KORGE}): {(top1['similarity_total'] >= KYNNIS_VAGA_KORGE).sum()} tk ({(top1['similarity_total'] >= KYNNIS_VAGA_KORGE).sum()/len(top1)*100:.1f}%)
Kõrge (≥{KYNNIS_KORGE}): {(top1['similarity_total'] >= KYNNIS_KORGE).sum()} tk ({(top1['similarity_total'] >= KYNNIS_KORGE).sum()/len(top1)*100:.1f}%)
    """
    ax.text(0.02, 0.98, stats_tekst, transform=ax.transAxes,
           fontsize=10, verticalalignment='top', family='monospace',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plt.tight_layout()
    valjund_tee = os.path.join(valjund_kaust, '1_sarnasuse_jaotus_histogramm.png')
    plt.savefig(valjund_tee, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Salvestatud: {valjund_tee}")

    # === 1.2 Kumulatiivne jaotus (eraldi fail) ===
    fig, ax = plt.subplots(figsize=(14, 10))

    sorteeritud = np.sort(top1['similarity_total'])
    kumulatiivne = np.arange(1, len(sorteeritud) + 1) / len(sorteeritud) * 100

    ax.plot(sorteeritud, kumulatiivne, linewidth=3, color='steelblue')
    ax.axvline(KYNNIS_KORGE, color='darkorange', linestyle='--', linewidth=2,
              label=f'Kõrge lävi ({KYNNIS_KORGE})')
    ax.axhline(50, color='gray', linestyle=':', linewidth=1.5, label='50%')

    # Märgi protsentiilid
    protsentiilid = [25, 50, 75, 90, 95]
    for p in protsentiilid:
        val = np.percentile(top1['similarity_total'], p)
        ax.plot(val, p, 'ro', markersize=10)
        ax.text(val + 0.01, p, f'{p}%: {val:.3f}', fontsize=11)

    ax.set_xlabel('Sarnasuse skoor', fontsize=14, fontweight='bold')
    ax.set_ylabel('Kumulatiivne protsent (%)', fontsize=14, fontweight='bold')
    ax.set_title('KUMULATIIVNE JAOTUS\nMitu kangast jõuab kõrge sarnasuseni?',
                fontsize=16, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0.5, 1.0])

    plt.tight_layout()
    valjund_tee = os.path.join(valjund_kaust, '2_kumulatiivne_jaotus.png')
    plt.savefig(valjund_tee, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Salvestatud: {valjund_tee}")

    # === 1.3 Q-Q graafik (normaaljaotuse test, eraldi fail) ===
    fig, ax = plt.subplots(figsize=(12, 10))

    stats.probplot(top1['similarity_total'], dist="norm", plot=ax)
    ax.set_title('Q-Q GRAAFIK: Normaaljaotuse test\n(Sirge = normaaljaotus, kõver = ebajaotus)',
                fontsize=16, fontweight='bold')
    ax.grid(True, alpha=0.3)

    # Shapiro-Wilk test
    _, p_vaartus = stats.shapiro(top1['similarity_total'])
    norm_tekst = f"Shapiro-Wilki test\np-väärtus = {p_vaartus:.4f}\n"
    if p_vaartus < 0.05:
        norm_tekst += "→ EI OLE normaaljaotunud\n→ Viitab süstemaatilistele mustritele!"
    else:
        norm_tekst += "→ Võib olla juhuslik"

    ax.text(0.05, 0.95, norm_tekst, transform=ax.transAxes,
           fontsize=12, verticalalignment='top', family='monospace',
           bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    valjund_tee = os.path.join(valjund_kaust, '3_normaaljaotuse_test.png')
    plt.savefig(valjund_tee, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Salvestatud: {valjund_tee}")

    # === 1.4 Kategooriate jaotus (eraldi fail) ===
    fig, ax = plt.subplots(figsize=(12, 10))

    kategooriad = ['Väga kõrge\n(≥0.90)', 'Kõrge\n(0.85-0.90)',
                   'Keskmine\n(0.75-0.85)', 'Madal\n(<0.75)']
    arvud = [
        (top1['similarity_total'] >= KYNNIS_VAGA_KORGE).sum(),
        ((top1['similarity_total'] >= KYNNIS_KORGE) &
         (top1['similarity_total'] < KYNNIS_VAGA_KORGE)).sum(),
        ((top1['similarity_total'] >= KYNNIS_KESKMINE) &
         (top1['similarity_total'] < KYNNIS_KORGE)).sum(),
        (top1['similarity_total'] < KYNNIS_KESKMINE).sum()
    ]
    varvid = ['#2ca02c', '#ff7f0e', '#1f77b4', '#d62728']

    wedges, texts, autotexts = ax.pie(arvud, labels=kategooriad, autopct='%1.1f%%',
                                       colors=varvid, startangle=90,
                                       textprops={'fontsize': 12})
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
        autotext.set_fontsize(13)

    ax.set_title('SARNASUSE KATEGOORIAD\nMis osa näitab tugevat sarnasust?',
                fontsize=16, fontweight='bold')

    plt.tight_layout()
    valjund_tee = os.path.join(valjund_kaust, '4_kategooriate_jaotus.png')
    plt.savefig(valjund_tee, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Salvestatud: {valjund_tee}")

    # Tõlgendus
    print("\n📊 TÕLGENDUS:")
    korge_protsent = (top1['similarity_total'] >= KYNNIS_KORGE).sum() / len(top1) * 100

    if korge_protsent > 30:
        print(f"  → {korge_protsent:.1f}% kangastest näitab KÕRGET sarnasust (≥{KYNNIS_KORGE})")
        print("  → See on PALJU KÕRGEM kui juhuslikult oodatav!")
        print("  → TUGEV TÕEND süstemaatiliste seoste / ühise algupära kohta")
    elif korge_protsent > 10:
        print(f"  → {korge_protsent:.1f}% näitab kõrget sarnasust")
        print("  → MÕÕDUKAS TÕEND mittejuhuslike mustrite kohta")
    else:
        print(f"  → Ainult {korge_protsent:.1f}% näitab kõrget sarnasust")
        print("  → Rohkem kooskõlas juhusliku vastavusega")




# ============================================================
# 2. KOMPONENTIDE ANALÜÜS
# ============================================================
def loo_komponentide_analyys(df, valjund_kaust):
    """
    Analüüsib, milline komponent (CNN/VÄRV/TRIIP) sarnasust kõige rohkem mõjutab.
    """
    print("\n" + "=" * 70)
    print("2. KOMPONENTIDE PANUSE ANALÜÜS")
    print("=" * 70)

    top1 = df[df['rank'] == 1].copy()

    # === 2.1 Komponentide jaotused (boxplot, eraldi fail) ===
    fig, ax = plt.subplots(figsize=(14, 10))

    komponendid = ['similarity_cnn', 'similarity_color', 'similarity_ratio']
    sildid = ['CNN\n(Tekstuur)', 'VÄRV\n(Palett)', 'TRIIP\n(Muster)']

    bp = ax.boxplot([top1[komp] for komp in komponendid],
                    labels=sildid, patch_artist=True, showmeans=True)

    for patch, varv in zip(bp['boxes'], ['#1f77b4', '#ff7f0e', '#2ca02c']):
        patch.set_facecolor(varv)
        patch.set_alpha(0.7)

    ax.set_ylabel('Komponendi skoor', fontsize=14, fontweight='bold')
    ax.set_title('KOMPONENTIDE SKOORIDE JAOTUSED\nMilline aspekt näitab tugevimat sarnasust?',
                fontsize=16, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0, 1])

    # Lisa keskmised
    for i, komp in enumerate(komponendid):
        keskmine = top1[komp].mean()
        ax.text(i+1, keskmine + 0.02, f'{keskmine:.3f}',
               ha='center', fontweight='bold', fontsize=12)

    plt.tight_layout()
    valjund_tee = os.path.join(valjund_kaust, '5_komponentide_jaotused.png')
    plt.savefig(valjund_tee, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Salvestatud: {valjund_tee}")

    # === 2.2 Korrelatsioonimaatriks (eraldi fail) ===
    fig, ax = plt.subplots(figsize=(10, 8))

    korr_andmed = top1[komponendid + ['similarity_total']].corr()

    sns.heatmap(korr_andmed, annot=True, fmt='.3f', cmap='coolwarm',
               center=0, square=True, ax=ax, cbar_kws={'label': 'Korrelatsioon'},
               annot_kws={'fontsize': 11})
    ax.set_title('KOMPONENTIDE KORRELATSIOONID\nMis komponendid ennustavad üldist sarnasust?',
                fontsize=16, fontweight='bold', pad=20)
    ax.set_xticklabels(['CNN', 'VÄRV', 'TRIIP', 'KOKKU'], rotation=45, fontsize=12)
    ax.set_yticklabels(['CNN', 'VÄRV', 'TRIIP', 'KOKKU'], rotation=0, fontsize=12)

    plt.tight_layout()
    valjund_tee = os.path.join(valjund_kaust, '6_komponentide_korrelatsioonid.png')
    plt.savefig(valjund_tee, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Salvestatud: {valjund_tee}")

    # === 2.3 Hajuvusdiagramm: Kokku vs Komponendid (eraldi fail) ===
    fig, ax = plt.subplots(figsize=(14, 10))

    varvid_hajuvus = ['#1f77b4', '#ff7f0e', '#2ca02c']
    r2_vaartused = {}

    for komp, silt, varv in zip(komponendid, sildid, varvid_hajuvus):
        ax.scatter(top1[komp], top1['similarity_total'],
                  alpha=0.5, s=40, label=silt.replace('\n', ' '), color=varv)

        # Sobita sirge
        z = np.polyfit(top1[komp], top1['similarity_total'], 1)
        p = np.poly1d(z)
        x_sirge = np.linspace(top1[komp].min(), top1[komp].max(), 100)
        ax.plot(x_sirge, p(x_sirge), "--", color=varv, linewidth=2, alpha=0.7)

        # Arvuta R²
        korr = np.corrcoef(top1[komp], top1['similarity_total'])[0, 1]
        r2_vaartused[komp] = korr ** 2

    ax.set_xlabel('Komponendi skoor', fontsize=14, fontweight='bold')
    ax.set_ylabel('Kogu sarnasus', fontsize=14, fontweight='bold')
    ax.set_title('KOMPONENT vs KOGU SARNASUS\nMis komponent on kõige ennustavarm?',
                fontsize=16, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)

    # R² väärtused
    r2_tekst = "R² väärtused:\n"
    for komp, silt in zip(komponendid, sildid):
        r2_tekst += f"{silt.replace(chr(10), ' ')}: {r2_vaartused[komp]:.3f}\n"

    ax.text(0.02, 0.98, r2_tekst, transform=ax.transAxes,
           fontsize=11, verticalalignment='top', family='monospace',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plt.tight_layout()
    valjund_tee = os.path.join(valjund_kaust, '7_komponentide_ennustusvõime.png')
    plt.savefig(valjund_tee, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Salvestatud: {valjund_tee}")

    # === 2.4 Domineeriv komponent (eraldi fail) ===
    fig, ax = plt.subplots(figsize=(12, 10))

    # Iga kanga jaoks leia, milline komponent on kõrgeim
    top1['domineeriv_komponent'] = top1[komponendid].idxmax(axis=1)
    top1['domineeriv_komponent'] = top1['domineeriv_komponent'].map({
        'similarity_cnn': 'CNN',
        'similarity_color': 'VÄRV',
        'similarity_ratio': 'TRIIP'
    })

    dom_arvud = top1['domineeriv_komponent'].value_counts()

    ax.bar(range(len(dom_arvud)), dom_arvud.values,
          color=['#1f77b4', '#ff7f0e', '#2ca02c'][:len(dom_arvud)],
          edgecolor='black', alpha=0.7, width=0.6)
    ax.set_xticks(range(len(dom_arvud)))
    ax.set_xticklabels(dom_arvud.index, fontsize=13)
    ax.set_ylabel('Kangaste arv', fontsize=14, fontweight='bold')
    ax.set_title('DOMINEERIV KOMPONENT KANGA KOHTA\nMis aspekt määrab vastava?',
                fontsize=16, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

    for i, (komp, arv) in enumerate(zip(dom_arvud.index, dom_arvud.values)):
        protsent = arv / len(top1) * 100
        ax.text(i, arv + 3, f'{arv} tk\n({protsent:.1f}%)',
               ha='center', fontweight='bold', fontsize=12)

    plt.tight_layout()
    valjund_tee = os.path.join(valjund_kaust, '8_domineeriv_komponent.png')
    plt.savefig(valjund_tee, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Salvestatud: {valjund_tee}")

    # Tõlgendus
    print("\n📊 TÕLGENDUS:")
    for komp, silt in zip(komponendid, ['CNN (tekstuur)', 'VÄRV', 'TRIIP (muster)']):
        korr = np.corrcoef(top1[komp], top1['similarity_total'])[0, 1]
        print(f"  {silt}: korrelatsioon = {korr:.3f}, R² = {r2_vaartused[komp]:.3f}")

    parim = max(r2_vaartused, key=r2_vaartused.get).replace('similarity_', '').upper()
    print(f"\n  Kõige ennustavarm: {parim}")


# ============================================================
# 3. LÕPLIK HINNANG
# ============================================================
def loo_loplik_hinnang(df, valjund_kaust):
    """
    Sünteesib kõik analüüsid lõplikuks hinnanguks juhuslike vs süstemaatiliste seoste kohta.
    """
    print("\n" + "=" * 70)
    print("3. LÕPLIK HINNANG")
    print("=" * 70)

    top1 = df[df['rank'] == 1].copy()

    fig = plt.figure(figsize=(18, 24))
    gs = fig.add_gridspec(5, 2, hspace=0.5, wspace=0.3)

    # Põhitekst
    ax_peamine = fig.add_subplot(gs[0:3, :])
    ax_peamine.axis('off')

    # Arvuta põhinäitajad
    vaga_korge_arv = (top1['similarity_total'] >= KYNNIS_VAGA_KORGE).sum()
    korge_arv = ((top1['similarity_total'] >= KYNNIS_KORGE) &
                  (top1['similarity_total'] < KYNNIS_VAGA_KORGE)).sum()
    tugevaid_kokku = vaga_korge_arv + korge_arv
    tugevaid_protsent = tugevaid_kokku / len(top1) * 100

    keskmine = top1['similarity_total'].mean()
    mediaan = top1['similarity_total'].median()
    vinoorus = top1['similarity_total'].skew()

    # Määra hinnang
    if tugevaid_protsent > 30:
        hinnang = "TUGEVAD SÜSTEMAATILISED SEOSED"
        hinnang_varv = '#2ca02c'
        kindlus = "KÕRGE"
        selgitus = """
Andmed näitavad SELGEID MITTEJUHUSLIKKE MUSTREID:
• Üle 30% kangastest näitab kõrget sarnasust (≥0.85)
• See on KAUGELT üle selle, mida juhuslik kokkulangevus annaks
• Tugev tõend ÜHISE ALGUPÄRA või otsese kopeerimise kohta
        """
    elif tugevaid_protsent > 15:
        hinnang = "MÕÕDUKAD SÜSTEMAATILISED MUSTRID"
        hinnang_varv = '#ff7f0e'
        kindlus = "KESKMINE"
        selgitus = """
Andmed näitavad MÄRKIMISVÄÄRSEID MUSTREID üle juhusliku taseme:
• 15-30% kangastest näitab kõrget sarnasust
• See viitab mõningatele süstemaatilistele seostele
• Tõenäoliselt viitab ÜHISELE DISAINITRADITSIOONI
        """
    else:
        hinnang = "NÕRGAD VÕI JUHUSLIKUD MUSTRID"
        hinnang_varv = '#d62728'
        kindlus = "MADAL"
        selgitus = """
Andmed näitavad peamiselt JUHUSLIKKU VASTAVUST:
• Vähem kui 15% näitab kõrget sarnasust
• See on kooskõlas juhuslike kokkulangevustega
• Vähe tõendeid süstemaatiliste seoste kohta
        """

    hinnang_tekst = f"""
╔════════════════════════════════════════════════════════════╗
║              UURIMISTÖÖ LÕPLIK HINNANG                     ║
╚════════════════════════════════════════════════════════════╝

KÜSIMUS: Kas eesti-inglise kangaste sarnasused on juhuslikud
          või viitavad need ühisele algupärale/traditsioonile?

HINNANG: {hinnang}

KINDLUS: {kindlus}

════════════════════════════════════════════════════════════

PÕHITÕENDID:
{'='*60}

1. SARNASUSE JAOTUS:
   • Väga kõrge (≥0.90): {vaga_korge_arv} kangast ({vaga_korge_arv/len(top1)*100:.1f}%)
   • Kõrge (≥0.85): {korge_arv} kangast ({korge_arv/len(top1)*100:.1f}%)
   • KOKKU tugevaid vasteid: {tugevaid_kokku} ({tugevaid_protsent:.1f}%)

2. KESKTENDENTSID:
   • Keskmine sarnasus: {keskmine:.4f}
   • Mediaansarnasus: {mediaan:.4f}
   • {'Üleval' if keskmine > 0.80 else 'All'} süstemaatiliste seoste lävendist

3. JAOTUSE KUJU:
   • {'PAREMALE VINDU: Palju kõrgeid väärtusi (mittejuhuslik!)' if vinoorus > 0.5 else 'NORMAALJAOTUSE SARNANE: Juhuslikku meenutav jaotus'}
   • Shapiro-Wilki p-väärtus: {stats.shapiro(top1['similarity_total'])[1]:.4f}

4. KOMPONENTIDE ANALÜÜS:
   • CNN (tekstuur) korrelatsioon: {np.corrcoef(top1['similarity_cnn'], top1['similarity_total'])[0,1]:.3f}
   • VÄRV korrelatsioon: {np.corrcoef(top1['similarity_color'], top1['similarity_total'])[0,1]:.3f}
   • TRIIP korrelatsioon: {np.corrcoef(top1['similarity_ratio'], top1['similarity_total'])[0,1]:.3f}

════════════════════════════════════════════════════════════

TÕLGENDUS:
{selgitus}

JÄRELDUS UURIMISTÖÖKS:
{'✓ JAH - Tugev tõend ühise algupära/traditsiooni kohta' if tugevaid_protsent > 20 else '? VÕIB-OLLA - Mõningad tõendid seoste kohta, vajab täiendavat uurimist' if tugevaid_protsent > 10 else '✗ EI - Tulemused kooskõlas juhusliku kokkulangevusega'}
    """

    ax_peamine.text(0.5, 0.5, hinnang_tekst, fontsize=10, family='monospace',
                verticalalignment='center', horizontalalignment='center',
                bbox=dict(boxstyle='round', facecolor=hinnang_varv,
                         alpha=0.2, edgecolor=hinnang_varv, linewidth=4))

    # Toetavad visualiseeringud

    # Sektordiagramm
    ax1 = fig.add_subplot(gs[3, 0])
    kategooriad = ['Väga kõrge\n(≥0.90)', 'Kõrge\n(0.85-0.90)',
                   'Keskmine\n(0.75-0.85)', 'Madal\n(<0.75)']
    arvud = [
        vaga_korge_arv,
        korge_arv,
        ((top1['similarity_total'] >= KYNNIS_KESKMINE) &
         (top1['similarity_total'] < KYNNIS_KORGE)).sum(),
        (top1['similarity_total'] < KYNNIS_KESKMINE).sum()
    ]
    varvid_sektor = ['#2ca02c', '#ff7f0e', '#1f77b4', '#d62728']

    ax1.pie(arvud, labels=kategooriad, autopct='%1.1f%%', colors=varvid_sektor,
            startangle=90, textprops={'fontsize': 10})
    ax1.set_title('JAOTUS', fontsize=13, fontweight='bold')

    # Tulpdiagramm
    ax2 = fig.add_subplot(gs[3, 1])
    komponendid_nimed = ['CNN', 'VÄRV', 'TRIIP']
    komp_keskmised = [
        top1['similarity_cnn'].mean(),
        top1['similarity_color'].mean(),
        top1['similarity_ratio'].mean()
    ]
    ax2.bar(komponendid_nimed, komp_keskmised,
            color=['#1f77b4', '#ff7f0e', '#2ca02c'], alpha=0.7, edgecolor='black')
    ax2.set_ylabel('Keskmine skoor', fontsize=11)
    ax2.set_title('KOMPONENDID', fontsize=13, fontweight='bold')
    ax2.set_ylim([0, 1])
    ax2.grid(axis='y', alpha=0.3)

    for i, val in enumerate(komp_keskmised):
        ax2.text(i, val + 0.02, f'{val:.3f}', ha='center', fontweight='bold')

    # Histogramm
    ax3 = fig.add_subplot(gs[4, :])
    ax3.hist(top1['similarity_total'], bins=50, edgecolor='black',
             alpha=0.7, color='steelblue')
    ax3.axvline(KYNNIS_VAGA_KORGE, color='darkgreen', linestyle='--', linewidth=2,
               label=f'Väga kõrge ({KYNNIS_VAGA_KORGE})')
    ax3.axvline(KYNNIS_KORGE, color='darkorange', linestyle='--', linewidth=2,
               label=f'Kõrge ({KYNNIS_KORGE})')
    ax3.axvline(keskmine, color='red', linestyle='-', linewidth=2.5,
               label=f'Keskmine ({keskmine:.3f})')
    ax3.set_xlabel('Sarnasuse skoor', fontsize=13, fontweight='bold')
    ax3.set_ylabel('Kangaste arv', fontsize=13, fontweight='bold')
    ax3.set_title('ÜLDINE JAOTUS', fontsize=15, fontweight='bold')
    ax3.legend(fontsize=11)
    ax3.grid(axis='y', alpha=0.3)

    plt.savefig(os.path.join(valjund_kaust, '9_LOPLIK_HINNANG.png'),
                dpi=300, bbox_inches='tight')
    plt.close()

    print(f"✓ Salvestatud: 9_LOPLIK_HINNANG.png")

    print("\n" + "="*70)
    print("LÕPLIK VASTUS:")
    print("="*70)
    print(f"\n{hinnang}")
    print(f"Kindlus: {kindlus}")
    print(f"\nTugevaid vasteid: {tugevaid_kokku}/{len(top1)} ({tugevaid_protsent:.1f}%)")

    if tugevaid_protsent > 30:
        print("\n→ Teie kangad näitavad TUGEVAT tõendit ühise algupära kohta!")
        print("→ See EI OLE juhuslik - on selged süstemaatilised seosed")
    elif tugevaid_protsent > 15:
        print("\n→ Teie kangad näitavad MÕÕDUKALT tõendeid seoste kohta")
        print("→ Mõned süstemaatilised mustrid eksisteerivad üle juhusliku taseme")
    else:
        print("\n→ Tulemused on valdavalt kooskõlas juhusliku vastavusega")
        print("→ Piiratud tõendid süstemaatilise ühise algupära kohta")


# ============================================================
# PÕHIFUNKTSIOON
# ============================================================
def main():
    # Laadi andmed
    csv_tee = os.path.join(RESULTS_DIR, 'results_similarity_main.csv')

    if not os.path.exists(csv_tee):
        print(f"❌ VIGA: {csv_tee} ei leitud!")
        print("Palun käivitage esmalt SAMM 1!")
        return

    print("\n" + "=" * 70)
    print("ANDMETE LAADIMINE")
    print("=" * 70)

    df = pd.read_csv(csv_tee)
    print(f"✓ Laaditud: {len(df)} rida")
    print(f"  {len(df['estonia_name'].unique())} eesti kangast")
    print(f"  {len(df['england_name'].unique())} inglise kangast")

    # Käivita kõik analüüsid
    loo_yldine_jaotus(df, RESULTS_DIR)
    loo_komponentide_analyys(df, RESULTS_DIR)
    loo_loplik_hinnang(df, RESULTS_DIR)

    # Kokkuvõte
    print("\n" + "=" * 70)
    print("✓ ANALÜÜS VALMIS!")
    print("=" * 70)
    print(f"\nKõik analüüsid salvestatud kausta: {RESULTS_DIR}")
    print("\nLoodud failid:")
    print("  1. 1_sarnasuse_jaotus_histogramm.png")
    print("  2. 2_kumulatiivne_jaotus.png")
    print("  3. 3_normaaljaotuse_test.png")
    print("  4. 4_kategooriate_jaotus.png")
    print("  5. 5_komponentide_jaotused.png")
    print("  6. 6_komponentide_korrelatsioonid.png")
    print("  7. 7_komponentide_ennustusvõime.png")
    print("  8. 8_domineeriv_komponent.png")
    print("  9. 9_LOPLIK_HINNANG.png - PÕHIJÄRELDUS!")
    print("\n🎯 Vaata '9_LOPLIK_HINNANG.png' põhijärelduse jaoks!")


if __name__ == "__main__":
    main()

KANGASTE STATISTILINE ANALÜÜS JA ALGUPÄRA HINDAMINE

UURIMISEESMÄRK: Kas sarnasused on juhuslikud või süstemaatilised?

Sarnasuse lävended:
  Väga kõrge (≥0.9): Peaaegu identsed
  Kõrge (≥0.85): Tugev sarnasus, tõenäoliselt ühine algupära
  Keskmine (≥0.75): Mõõdukas sarnasus
  Madal (<0.75): Nõrk või juhuslik sarnasus

ANDMETE LAADIMINE
✓ Laaditud: 3450 rida
  690 eesti kangast
  651 inglise kangast

1. ÜLDINE SARNASUSE JAOTUS
✓ Salvestatud: /content/drive/MyDrive/Masinõpe_tulemused/1_sarnasuse_jaotus_histogramm.png
✓ Salvestatud: /content/drive/MyDrive/Masinõpe_tulemused/2_kumulatiivne_jaotus.png
✓ Salvestatud: /content/drive/MyDrive/Masinõpe_tulemused/3_normaaljaotuse_test.png
✓ Salvestatud: /content/drive/MyDrive/Masinõpe_tulemused/4_kategooriate_jaotus.png

📊 TÕLGENDUS:
  → Ainult 0.0% näitab kõrget sarnasust
  → Rohkem kooskõlas juhusliku vastavusega

2. KOMPONENTIDE PANUSE ANALÜÜS


/tmp/ipython-input-269857375.py:275: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot([top1[komp] for komp in komponendid],


✓ Salvestatud: /content/drive/MyDrive/Masinõpe_tulemused/5_komponentide_jaotused.png
✓ Salvestatud: /content/drive/MyDrive/Masinõpe_tulemused/6_komponentide_korrelatsioonid.png
✓ Salvestatud: /content/drive/MyDrive/Masinõpe_tulemused/7_komponentide_ennustusvõime.png
✓ Salvestatud: /content/drive/MyDrive/Masinõpe_tulemused/8_domineeriv_komponent.png

📊 TÕLGENDUS:
  CNN (tekstuur): korrelatsioon = 0.517, R² = 0.267
  VÄRV: korrelatsioon = 0.638, R² = 0.407
  TRIIP (muster): korrelatsioon = 0.341, R² = 0.116

  Kõige ennustavarm: COLOR

3. LÕPLIK HINNANG
✓ Salvestatud: 9_LOPLIK_HINNANG.png

LÕPLIK VASTUS:

NÕRGAD VÕI JUHUSLIKUD MUSTRID
Kindlus: MADAL

Tugevaid vasteid: 0/690 (0.0%)

→ Tulemused on valdavalt kooskõlas juhusliku vastavusega
→ Piiratud tõendid süstemaatilise ühise algupära kohta

✓ ANALÜÜS VALMIS!

Kõik analüüsid salvestatud kausta: /content/drive/MyDrive/Masinõpe_tulemused

Loodud failid:
  1. 1_sarnasuse_jaotus_histogramm.png
  2. 2_kumulatiivne_jaotus.png
  3. 3_normaalja

## Etapp 5. Baasvõrdlus: Eesti–Eesti, Inglise–Inglise ja Eesti–Inglise sarnasused

### Eesmärk

Selle etapi eesmärk on hinnata, kas Eesti ja Inglismaa kangaste vahel leitud sarnasused on eripärased või jäävad need samale tasemele nagu sarnasused ühe kogu sees.

Selleks võrreldakse kolme tüüpi sarnasusi:

1. Eesti kangas vs Eesti kangas
2. Inglismaa kangas vs Inglismaa kangas
3. Eesti kangas vs Inglismaa kangas

Kui Eesti–Inglise sarnasused on kõrgemad kui kogusisesed võrdlused, võib see toetada hüpoteesi, et kahe kogu vahel esineb süstemaatilisi visuaalseid seoseid.

### Uurimisküsimus

Kas Eesti ja Inglismaa kangaste vahelised sarnasused on kõrgemad kui juhuslikult oodatav sarnasus sarnaste kangakogude sees?

### Hüpoteesid

- **Nullhüpotees:** Eesti–Inglise sarnasused ei erine oluliselt baasvõrdlustest.
- **Alternatiivne hüpotees:** Eesti–Inglise sarnasused on baasvõrdlustest kõrgemad ja võivad viidata süstemaatilisele seosele.

### Sisendandmed

- **Inglismaa kangaste indeks:** `england_index_v1.joblib`
- **Algpildid:**
  - `Eesti_10`
  - `Eesti_20`
  - `Inglismaa_20`
  - `Inglismaa9`

Analüüsis kasutatakse valimit, mitte kõiki võimalikke paaride kombinatsioone, sest kõikide kangaste omavaheline võrdlemine oleks arvutuslikult mahukas.

### Valimi suurus

Kood kasutab järgmisi valimeid:

- `100` Eesti kangast
- `100` Inglismaa kangast

Valik tehakse juhuslikult, kuid fikseeritud seemnega (`np.random.seed(42)`), et tulemus oleks korratav.

### Analüüsi loogika

Kõigepealt valitakse Eesti ja Inglismaa kangastest juhuslik valim. Seejärel arvutatakse igale pildile samad tunnused nagu esimeses etapis:

- CNN-tunnused;
- LAB-värvipalett;
- triibumustri proportsioonid.

Seejärel arvutatakse kolm sarnasuste jaotust:

1. **Eesti–Eesti**
   - võrdleb Eesti kangaid omavahel;
   - toimib ühe baasjaotusena.

2. **Inglise–Inglise**
   - võrdleb Inglismaa kangaid omavahel;
   - toimib teise baasjaotusena.

3. **Eesti–Inglise**
   - võrdleb Eesti ja Inglismaa kangaid omavahel;
   - see on põhivõrdlus.

### Peamised tegevused

Selles etapis tehakse järgmised tegevused:

1. kontrollitakse, kas `england_index_v1.joblib` on olemas;
2. laaditakse ResNet50 mudel;
3. valitakse juhuslikult Eesti ja Inglismaa kangaste valim;
4. arvutatakse valitud piltide tunnused;
5. arvutatakse Eesti–Eesti paaride sarnasused;
6. arvutatakse Inglise–Inglise paaride sarnasused;
7. arvutatakse Eesti–Inglise paaride sarnasused;
8. salvestatakse kõik sarnasusskoorid CSV-faili;
9. võrreldakse kolme jaotust statistiliselt;
10. luuakse koondvisualiseering ja lõplik tõlgendus.

### Statistiline võrdlus

Analüüsis kasutatakse mitut võrdlust:

- keskmiste ja mediaanide võrdlus;
- jaotuste visuaalne võrdlus histogrammide abil;
- boxplot keskväärtuste ja hajuvuse hindamiseks;
- t-testid Eesti–Inglise ja baasjaotuste vahel;
- ANOVA kolme jaotuse üldiseks võrdlemiseks;
- Coheni d efekti suuruse hindamiseks.

Need testid aitavad hinnata, kas Eesti–Inglise sarnasused erinevad baasvõrdlustest statistiliselt märgatavalt.

### Väljundid

Selle etapi väljunditeks on kaks faili:

- `baseline_similarities.csv`  
  Toorandmete tabel kõigi arvutatud sarnasusskooridega.

- `baseline_comparison.png`  
  Koondvisualiseering, mis näitab jaotuste võrdlust, statistilisi teste ja lõplikku tõlgendust.

### Ajahinnang

See samm võib olla ajamahukas, sest pilditunnused arvutatakse valitud kangastele uuesti ning seejärel võrreldakse palju pildipaare.

Ajakulu sõltub peamiselt:

- valimi suurusest;
- piltide arvust;
- GPU või CPU kasutusest;
- pildifailide lugemise kiirusest Google Drive’ist.

Valimi kasutamine muudab selle sammu oluliselt kiiremaks kui kõigi võimalike paaride võrdlemine.

### Nähtavad vaheväljundid

Notebookis kuvatakse:

- kasutatav arvutusseade;
- valimisse võetud Eesti ja Inglismaa kangaste arv;
- mitu tunnusepaketti õnnestus arvutada;
- mitu paarikaupa sarnasust arvutati;
- keskmised sarnasused kolme võrdlustüübi kaupa;
- p-väärtused;
- efekti suurused;
- lõplik tekstiline hinnang.

Need väljundid aitavad kontrollida, kas baasvõrdlus tehti korrektselt ja kas Eesti–Inglise tulemused erinevad kogusisestest võrdlustest.

### Tõlgendamise märkus

Selle etapi tulemused aitavad hinnata, kas Eesti–Inglise sarnasused on tugevamad kui võiks oodata ainult juhusliku või kogusisese varieeruvuse põhjal.

Kui Eesti–Inglise keskmine sarnasus on baasvõrdlustest kõrgem ja statistilised testid näitavad olulist erinevust, toetab see ideed, et kahe kogu vahel võib olla süstemaatiline visuaalne seos.

Kui erinevust ei ilmne, võib see tähendada, et Eesti–Inglise vasted ei eristu selgelt sellest, kui sarnased on kangad ühe kogu sees.

Oluline on siiski rõhutada, et ka statistiliselt oluline erinevus ei tõesta iseseisvalt ajaloolist ühist algupära. See annab pigem kvantitatiivse aluse edasiseks visuaalseks ja sisuliseks tõlgenduseks.

In [ ]:
"""
STEP 5: BASELINE COMPARISON ANALYSIS
=====================================
Compares Estonia-England similarities with baseline comparisons:
1. Estonia-Estonia (within same collection)
2. England-England (within same collection)

RESEARCH QUESTION:
Are Estonia-England similarities HIGHER than expected by random
chance within similar collections?

NULL HYPOTHESIS: Estonia-England similarities are SAME as Estonia-Estonia
ALTERNATIVE: Estonia-England similarities are HIGHER (indicating common origin)

INPUT:
- england_index_v1.joblib (from Step 1)
- Estonian fabric images

OUTPUT:
- Baseline similarity distributions
- Statistical comparison tests
- Final verdict on whether cross-collection similarity is significant
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from tqdm import tqdm
import joblib
import torch
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import cv2
from typing import Optional, Tuple
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
# SETTINGS
# ============================================================
BASE_PATH = "/content/drive/MyDrive/Kangad"
RESULTS_DIR = "/content/drive/MyDrive/Masinõpe_tulemused"

# Sample sizes (to make computation feasible)
N_ESTONIA_SAMPLES = 100  # Sample 100 Estonian fabrics
N_ENGLAND_SAMPLES = 100  # Sample 100 English fabrics

print("=" * 70)
print("STEP 5: BASELINE COMPARISON ANALYSIS")
print("=" * 70)
print("\nRESEARCH QUESTION:")
print("  Are Estonia-England similarities HIGHER than random chance?")
print("\nCOMPARISONS:")
print("  1. Estonia vs Estonia (baseline)")
print("  2. England vs England (baseline)")
print("  3. Estonia vs England (our main result)")


# ============================================================
# HELPER FUNCTIONS (copied from Step 1)
# ============================================================
def get_transform(size_category):
    if size_category == '10cm':
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.CenterCrop(224),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])


def extract_cnn_features(img_path, model, device, size_category):
    try:
        img = Image.open(img_path).convert('RGB')
        transform = get_transform(size_category)
        img_t = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            features = model(img_t).squeeze().cpu().numpy()
        return features
    except Exception as e:
        return None


def extract_lab_palette(img, n_colors=6):
    arr = np.array(img)
    if arr.ndim != 3 or arr.shape[0] < 10 or arr.shape[1] < 10:
        return None, None

    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB).astype(np.float32)
    pixels = lab.reshape(-1, 3)

    if len(pixels) > 50000:
        idx = np.random.choice(len(pixels), 50000, replace=False)
        pixels = pixels[idx]

    K = min(n_colors, max(3, len(pixels) // 5000 + 3))
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1.0)
    _, labels, centers = cv2.kmeans(pixels, K, None, criteria, 3, cv2.KMEANS_PP_CENTERS)

    labels = labels.flatten()
    centers = centers.astype(np.float32)
    counts = np.bincount(labels, minlength=K).astype(np.float32)
    weights = counts / counts.sum()
    order = np.argsort(weights)[::-1]

    return centers[order], weights[order]


def extract_stripe_widths(img, min_stripe_px=3):
    arr = np.array(img.convert("RGB"))
    if arr.shape[0] < 10 or arr.shape[1] < 10:
        return None

    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB).astype(np.float32)
    L = lab[:, :, 0]
    prof = L.mean(axis=0)
    prof_s = cv2.GaussianBlur(prof.reshape(1, -1), (1, 9), 0).flatten()
    g = np.abs(np.diff(prof_s, prepend=prof_s[0]))
    thr = np.percentile(g, 90)
    edge_idx = np.where(g >= thr)[0]

    if len(edge_idx) < 2:
        return None

    edges = [int(edge_idx[0])]
    for x in edge_idx[1:]:
        if x - edges[-1] >= min_stripe_px:
            edges.append(int(x))

    if edges[0] > 0:
        edges = [0] + edges
    if edges[-1] < arr.shape[1] - 1:
        edges = edges + [arr.shape[1] - 1]

    widths = []
    for a, b in zip(edges[:-1], edges[1:]):
        w = b - a
        if w >= min_stripe_px:
            widths.append(w)

    if len(widths) < 2:
        return None

    widths = np.array(widths, dtype=np.float32)
    return widths / widths.sum()


def extract_feature_pack(img_path, model, device, size_category):
    try:
        img = Image.open(img_path).convert("RGB")
    except:
        return None

    cnn = extract_cnn_features(img_path, model, device, size_category)
    if cnn is None:
        return None

    try:
        pal_c, pal_w = extract_lab_palette(img, n_colors=6)
    except:
        pal_c, pal_w = None, None

    try:
        sw = extract_stripe_widths(img, min_stripe_px=3)
    except:
        sw = None

    return {"cnn": cnn, "pal_c": pal_c, "pal_w": pal_w, "stripe_w": sw}


def lab_palette_similarity(centers1: np.ndarray, w1: np.ndarray,
                          centers2: np.ndarray, w2: np.ndarray) -> float:
    if centers1 is None or centers2 is None:
        return 0.0

    cost = np.linalg.norm(centers1[:, None, :] - centers2[None, :, :], axis=2)

    try:
        from scipy.optimize import linear_sum_assignment
        r, c = linear_sum_assignment(cost)
        pairs = list(zip(r.tolist(), c.tolist()))
    except:
        # Greedy fallback
        cost_copy = cost.copy()
        pairs = []
        used_r, used_c = set(), set()
        while len(used_r) < cost.shape[0] and len(used_c) < cost.shape[1]:
            best = None
            best_val = None
            for i in range(cost.shape[0]):
                if i in used_r:
                    continue
                for j in range(cost.shape[1]):
                    if j in used_c:
                        continue
                    v = cost_copy[i, j]
                    if best_val is None or v < best_val:
                        best_val = v
                        best = (i, j)
            if best is None:
                break
            i, j = best
            used_r.add(i)
            used_c.add(j)
            pairs.append((i, j))

    d_sum, w_sum = 0.0, 0.0
    for i, j in pairs:
        wij = float(min(w1[i], w2[j]))
        d_sum += wij * float(cost[i, j])
        w_sum += wij

    if w_sum <= 1e-9:
        return 0.0

    d = d_sum / w_sum
    scale = 60.0
    sim = float(np.exp(-d / scale))
    return max(0.0, min(1.0, sim))


def dtw_distance(a: np.ndarray, b: np.ndarray) -> float:
    n, m = len(a), len(b)
    D = np.full((n + 1, m + 1), np.inf, dtype=np.float32)
    D[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = abs(a[i - 1] - b[j - 1])
            D[i, j] = cost + min(D[i - 1, j], D[i, j - 1], D[i - 1, j - 1])
    return float(D[n, m])


def stripe_ratio_similarity(w1: Optional[np.ndarray], w2: Optional[np.ndarray]) -> float:
    if w1 is None or w2 is None:
        return 0.0
    d = dtw_distance(w1, w2)
    scale = 0.5
    sim = float(np.exp(-d / scale))
    return max(0.0, min(1.0, sim))


def combined_similarity(e_pack, i_pack, w_cnn=0.55, w_color=0.25, w_ratio=0.20):
    cnn_sim = float(cosine_similarity([e_pack["cnn"]], [i_pack["cnn"]])[0][0])
    cnn_sim_01 = (cnn_sim + 1.0) / 2.0

    color_sim = 0.0
    if e_pack["pal_c"] is not None and i_pack["pal_c"] is not None:
        color_sim = lab_palette_similarity(e_pack["pal_c"], e_pack["pal_w"],
                                          i_pack["pal_c"], i_pack["pal_w"])

    ratio_sim = stripe_ratio_similarity(e_pack["stripe_w"], i_pack["stripe_w"])

    total = w_cnn * cnn_sim_01 + w_color * color_sim + w_ratio * ratio_sim
    return float(total)


# ============================================================
# 1. CALCULATE BASELINE SIMILARITIES
# ============================================================
def calculate_baseline_similarities(model, device):
    """
    Calculate within-collection similarities for baseline comparison.
    """
    print("\n" + "=" * 70)
    print("CALCULATING BASELINE SIMILARITIES")
    print("=" * 70)

    # Paths
    estonia_path_10 = os.path.join(BASE_PATH, 'Eesti_10')
    estonia_path_20 = os.path.join(BASE_PATH, 'Eesti_20')
    england_path_20 = os.path.join(BASE_PATH, 'Inglismaa_20')
    england_path_10 = os.path.join(BASE_PATH, 'Inglismaa9')

    # Get file lists
    estonia_10_files = [f for f in os.listdir(estonia_path_10) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    estonia_20_files = [f for f in os.listdir(estonia_path_20) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    england_20_files = [f for f in os.listdir(england_path_20) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    england_10_files = [f for f in os.listdir(england_path_10) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]

    all_estonia_files = [(f, '10cm', estonia_path_10) for f in estonia_10_files] + \
                        [(f, '20cm', estonia_path_20) for f in estonia_20_files]

    all_england_files = [(f, '20cm', england_path_20) for f in england_20_files] + \
                        [(f, '10cm', england_path_10) for f in england_10_files]

    # Sample for computational efficiency
    np.random.seed(42)
    estonia_sample = np.random.choice(len(all_estonia_files),
                                     min(N_ESTONIA_SAMPLES, len(all_estonia_files)),
                                     replace=False)
    england_sample = np.random.choice(len(all_england_files),
                                     min(N_ENGLAND_SAMPLES, len(all_england_files)),
                                     replace=False)

    print(f"\nSampling:")
    print(f"  Estonian fabrics: {len(estonia_sample)} out of {len(all_estonia_files)}")
    print(f"  English fabrics: {len(england_sample)} out of {len(all_england_files)}")

    # Extract features for Estonian sample
    print("\nExtracting Estonian features...")
    estonia_packs = []
    estonia_names = []

    for idx in tqdm(estonia_sample):
        fname, size, path = all_estonia_files[idx]
        full_path = os.path.join(path, fname)
        pack = extract_feature_pack(full_path, model, device, size)
        if pack is not None:
            estonia_packs.append(pack)
            estonia_names.append(fname)

    print(f"  Extracted {len(estonia_packs)} Estonian feature packs")

    # Extract features for English sample
    print("\nExtracting English features...")
    england_packs = []
    england_names = []

    for idx in tqdm(england_sample):
        fname, size, path = all_england_files[idx]
        full_path = os.path.join(path, fname)
        pack = extract_feature_pack(full_path, model, device, size)
        if pack is not None:
            england_packs.append(pack)
            england_names.append(fname)

    print(f"  Extracted {len(england_packs)} English feature packs")

    # Calculate Estonia-Estonia similarities
    print("\n1. Calculating Estonia-Estonia similarities...")
    estonia_estonia_sims = []

    for i in tqdm(range(len(estonia_packs)), desc="Estonia-Estonia"):
        for j in range(i + 1, len(estonia_packs)):  # Only upper triangle
            sim = combined_similarity(estonia_packs[i], estonia_packs[j])
            estonia_estonia_sims.append(sim)

    print(f"  Calculated {len(estonia_estonia_sims)} pairwise similarities")

    # Calculate England-England similarities
    print("\n2. Calculating England-England similarities...")
    england_england_sims = []

    for i in tqdm(range(len(england_packs)), desc="England-England"):
        for j in range(i + 1, len(england_packs)):  # Only upper triangle
            sim = combined_similarity(england_packs[i], england_packs[j])
            england_england_sims.append(sim)

    print(f"  Calculated {len(england_england_sims)} pairwise similarities")

    # Calculate Estonia-England similarities (cross-collection)
    print("\n3. Calculating Estonia-England similarities...")
    estonia_england_sims = []

    for e_pack in tqdm(estonia_packs, desc="Estonia-England"):
        for i_pack in england_packs:
            sim = combined_similarity(e_pack, i_pack)
            estonia_england_sims.append(sim)

    print(f"  Calculated {len(estonia_england_sims)} pairwise similarities")

    return {
        'estonia_estonia': np.array(estonia_estonia_sims),
        'england_england': np.array(england_england_sims),
        'estonia_england': np.array(estonia_england_sims)
    }


# ============================================================
# 2. VISUALIZE BASELINE COMPARISON
# ============================================================
def visualize_baseline_comparison(similarities, out_dir):
    """
    Compare the three distributions to test if Estonia-England is special.
    """
    print("\n" + "=" * 70)
    print("CREATING BASELINE COMPARISON VISUALIZATIONS")
    print("=" * 70)

    ee_sims = similarities['estonia_estonia']
    ii_sims = similarities['england_england']
    ei_sims = similarities['estonia_england']

    fig, axes = plt.subplots(2, 2, figsize=(18, 14))

    # 1. Overlapping histograms
    ax = axes[0, 0]
    ax.hist(ee_sims, bins=50, alpha=0.5, label='Estonia-Estonia', color='#1f77b4', edgecolor='black')
    ax.hist(ii_sims, bins=50, alpha=0.5, label='England-England', color='#ff7f0e', edgecolor='black')
    ax.hist(ei_sims, bins=50, alpha=0.7, label='Estonia-England', color='#2ca02c', edgecolor='black')

    # Add means
    ax.axvline(ee_sims.mean(), color='#1f77b4', linestyle='--', linewidth=2,
              label=f'EE mean: {ee_sims.mean():.3f}')
    ax.axvline(ii_sims.mean(), color='#ff7f0e', linestyle='--', linewidth=2,
              label=f'II mean: {ii_sims.mean():.3f}')
    ax.axvline(ei_sims.mean(), color='#2ca02c', linestyle='-', linewidth=3,
              label=f'EI mean: {ei_sims.mean():.3f}')

    ax.set_xlabel('Similarity Score', fontsize=13, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=13, fontweight='bold')
    ax.set_title('DISTRIBUTION COMPARISON\nIs Estonia-England different from baselines?',
                fontsize=15, fontweight='bold')
    ax.legend(fontsize=10, loc='upper left')
    ax.grid(axis='y', alpha=0.3)

    # 2. Box plots
    ax = axes[0, 1]
    bp = ax.boxplot([ee_sims, ii_sims, ei_sims],
                    labels=['Estonia-\nEstonia', 'England-\nEngland', 'Estonia-\nEngland'],
                    patch_artist=True, showmeans=True)

    for patch, color in zip(bp['boxes'], ['#1f77b4', '#ff7f0e', '#2ca02c']):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_ylabel('Similarity Score', fontsize=13, fontweight='bold')
    ax.set_title('BOX PLOT COMPARISON\nCentral tendency and spread',
                fontsize=15, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0.4, 1.0])

    # Add mean values
    for i, (data, label) in enumerate([(ee_sims, 'EE'), (ii_sims, 'II'), (ei_sims, 'EI')], 1):
        mean_val = data.mean()
        ax.text(i, mean_val + 0.02, f'{mean_val:.3f}', ha='center', fontweight='bold', fontsize=11)

    # 3. Statistical tests summary
    ax = axes[1, 0]
    ax.axis('off')

    # Perform statistical tests
    # Test 1: Estonia-England vs Estonia-Estonia
    t_stat_ee, p_val_ee = stats.ttest_ind(ei_sims, ee_sims)

    # Test 2: Estonia-England vs England-England
    t_stat_ii, p_val_ii = stats.ttest_ind(ei_sims, ii_sims)

    # Effect sizes (Cohen's d)
    def cohens_d(x, y):
        nx, ny = len(x), len(y)
        dof = nx + ny - 2
        return (x.mean() - y.mean()) / np.sqrt(((nx-1)*x.std()**2 + (ny-1)*y.std()**2) / dof)

    d_ee = cohens_d(ei_sims, ee_sims)
    d_ii = cohens_d(ei_sims, ii_sims)

    # One-way ANOVA for all three groups
    f_stat, p_anova = stats.f_oneway(ee_sims, ii_sims, ei_sims)

    stats_text = f"""
╔════════════════════════════════════════════════════════════╗
║           STATISTICAL TESTS SUMMARY                        ║
╚════════════════════════════════════════════════════════════╝

DESCRIPTIVE STATISTICS:
{'='*60}

Estonia-Estonia (within collection):
  • Mean: {ee_sims.mean():.4f}
  • Median: {np.median(ee_sims):.4f}
  • Std: {ee_sims.std():.4f}
  • n = {len(ee_sims)}

England-England (within collection):
  • Mean: {ii_sims.mean():.4f}
  • Median: {np.median(ii_sims):.4f}
  • Std: {ii_sims.std():.4f}
  • n = {len(ii_sims)}

Estonia-England (cross-collection):
  • Mean: {ei_sims.mean():.4f}
  • Median: {np.median(ei_sims):.4f}
  • Std: {ei_sims.std():.4f}
  • n = {len(ei_sims)}

HYPOTHESIS TESTS:
{'='*60}

H0: Estonia-England similarities = baseline (random)
H1: Estonia-England similarities > baseline (systematic)

Test 1: Estonia-England vs Estonia-Estonia
  • t-statistic: {t_stat_ee:.4f}
  • p-value: {p_val_ee:.6f}
  • Cohen's d: {d_ee:.4f}
  • Result: {'REJECT H0 - Significantly different!' if p_val_ee < 0.05 else 'Cannot reject H0'}

Test 2: Estonia-England vs England-England
  • t-statistic: {t_stat_ii:.4f}
  • p-value: {p_val_ii:.6f}
  • Cohen's d: {d_ii:.4f}
  • Result: {'REJECT H0 - Significantly different!' if p_val_ii < 0.05 else 'Cannot reject H0'}

ANOVA (all three groups):
  • F-statistic: {f_stat:.4f}
  • p-value: {p_anova:.6f}
  • Result: {'Groups are significantly different!' if p_anova < 0.05 else 'No significant difference'}

EFFECT SIZE INTERPRETATION:
  • |d| < 0.2: Small effect
  • |d| < 0.5: Medium effect
  • |d| ≥ 0.5: Large effect
    """

    ax.text(0.1, 0.5, stats_text, fontsize=9.5, family='monospace',
           verticalalignment='center', bbox=dict(boxstyle='round',
           facecolor='lightyellow', alpha=0.8, edgecolor='black', linewidth=2))

    # 4. Final verdict
    ax = axes[1, 1]
    ax.axis('off')

    # Determine verdict
    ei_mean = ei_sims.mean()
    ee_mean = ee_sims.mean()
    ii_mean = ii_sims.mean()

    baseline_mean = (ee_mean + ii_mean) / 2
    difference = ei_mean - baseline_mean

    if p_val_ee < 0.05 and p_val_ii < 0.05 and difference > 0:
        verdict = "✓ ESTONIA-ENGLAND SHOWS SIGNIFICANTLY HIGHER SIMILARITY"
        verdict_color = '#2ca02c'
        interpretation = """
CONCLUSION: Estonia-England similarities are
SIGNIFICANTLY HIGHER than within-collection baselines.

This provides STRONG EVIDENCE for:
• Common origin or tradition
• Direct cultural exchange
• Shared design patterns

The difference is STATISTICALLY SIGNIFICANT
and NOT due to random chance!
        """
    elif difference > 0.02:
        verdict = "→ ESTONIA-ENGLAND SHOWS MODERATELY HIGHER SIMILARITY"
        verdict_color = '#ff7f0e'
        interpretation = """
CONCLUSION: Estonia-England similarities are
SOMEWHAT HIGHER than baselines.

This suggests:
• Some systematic relationships
• Possible common influences
• Worth further investigation

Statistical evidence is MODERATE.
        """
    else:
        verdict = "✗ NO SIGNIFICANT DIFFERENCE FROM BASELINE"
        verdict_color = '#d62728'
        interpretation = """
CONCLUSION: Estonia-England similarities are
NOT significantly higher than random baseline.

This suggests:
• Matches may be coincidental
• No strong evidence of common origin
• Similar to within-collection variation

Results consistent with RANDOM CHANCE.
        """

    verdict_text = f"""
╔════════════════════════════════════════════════════════════╗
║                    FINAL VERDICT                           ║
╚════════════════════════════════════════════════════════════╝

{verdict}

KEY FINDING:
Estonia-England mean: {ei_mean:.4f}
Baseline mean: {baseline_mean:.4f}
Difference: {difference:+.4f} ({difference/baseline_mean*100:+.1f}%)

{interpretation}

RECOMMENDATION:
{'→ Your research hypothesis is SUPPORTED!' if p_val_ee < 0.05 and difference > 0 else '→ Need more evidence or different approach' if difference > 0 else '→ Consider alternative explanations'}
    """

    ax.text(0.5, 0.5, verdict_text, fontsize=10, family='monospace',
           verticalalignment='center', horizontalalignment='center',
           bbox=dict(boxstyle='round', facecolor=verdict_color,
           alpha=0.3, edgecolor=verdict_color, linewidth=3))

    plt.tight_layout()
    output_path = os.path.join(out_dir, 'baseline_comparison.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"✓ Saved: {output_path}")

    # Print summary to console
    print("\n" + "=" * 70)
    print("STATISTICAL SUMMARY")
    print("=" * 70)
    print(f"\nMeans:")
    print(f"  Estonia-Estonia: {ee_mean:.4f}")
    print(f"  England-England: {ii_mean:.4f}")
    print(f"  Estonia-England: {ei_mean:.4f}")
    print(f"\nDifference from baseline: {difference:+.4f} ({difference/baseline_mean*100:+.1f}%)")
    print(f"\nStatistical tests:")
    print(f"  p-value (vs Estonia-Estonia): {p_val_ee:.6f}")
    print(f"  p-value (vs England-England): {p_val_ii:.6f}")
    print(f"\nVERDICT: {verdict}")


# ============================================================
# MAIN
# ============================================================
def main():
    # Check if index exists
    index_path = os.path.join(RESULTS_DIR, "england_index_v1.joblib")

    if not os.path.exists(index_path):
        print(f"❌ ERROR: {index_path} not found!")
        print("Please run STEP 1 first!")
        return

    print("\n" + "=" * 70)
    print("LOADING MODEL")
    print("=" * 70)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    model = models.resnet50(pretrained=True)
    model = torch.nn.Sequential(*list(model.children())[:-1])
    model = model.to(device)
    model.eval()

    # Calculate similarities
    similarities = calculate_baseline_similarities(model, device)

    # Save raw data
    baseline_data = pd.DataFrame({
        'comparison_type': ['estonia_estonia'] * len(similarities['estonia_estonia']) +
                          ['england_england'] * len(similarities['england_england']) +
                          ['estonia_england'] * len(similarities['estonia_england']),
        'similarity': np.concatenate([
            similarities['estonia_estonia'],
            similarities['england_england'],
            similarities['estonia_england']
        ])
    })

    output_csv = os.path.join(RESULTS_DIR, 'baseline_similarities.csv')
    baseline_data.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"\n✓ Raw data saved: {output_csv}")

    # Visualize
    visualize_baseline_comparison(similarities, RESULTS_DIR)

    # Summary
    print("\n" + "=" * 70)
    print("✓ STEP 5 COMPLETE!")
    print("=" * 70)
    print(f"\nResults saved to: {RESULTS_DIR}")
    print("\nGenerated files:")
    print("  1. baseline_similarities.csv - Raw similarity data")
    print("  2. baseline_comparison.png - Statistical comparison")
    print("\n🎯 Check 'baseline_comparison.png' for final verdict!")
    print("\n💡 This analysis answers: Is your cross-collection similarity")
    print("   HIGHER than expected by random chance?")


if __name__ == "__main__":
    main()

STEP 5: BASELINE COMPARISON ANALYSIS

RESEARCH QUESTION:
  Are Estonia-England similarities HIGHER than random chance?

COMPARISONS:
  1. Estonia vs Estonia (baseline)
  2. England vs England (baseline)
  3. Estonia vs England (our main result)

LOADING MODEL
Using device: cpu


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



CALCULATING BASELINE SIMILARITIES

Sampling:
  Estonian fabrics: 100 out of 690
  English fabrics: 100 out of 3246

Extracting Estonian features...


100%|██████████| 100/100 [00:37<00:00,  2.66it/s]


  Extracted 100 Estonian feature packs

Extracting English features...


...

  Calculated 4950 pairwise similarities

3. Calculating Estonia-England similarities...


Estonia-England: 100%|██████████| 100/100 [00:49<00:00,  2.02it/s]


  Calculated 10000 pairwise similarities

✓ Raw data saved: /content/drive/MyDrive/Masinõpe_tulemused/baseline_similarities.csv

CREATING BASELINE COMPARISON VISUALIZATIONS


/tmp/ipython-input-2174819251.py:419: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot([ee_sims, ii_sims, ei_sims],


✓ Saved: /content/drive/MyDrive/Masinõpe_tulemused/baseline_comparison.png

STATISTICAL SUMMARY

Means:
  Estonia-Estonia: 0.6835
  England-England: 0.7146
  Estonia-England: 0.6486

Difference from baseline: -0.0505 (-7.2%)

Statistical tests:
  p-value (vs Estonia-Estonia): 0.000000
  p-value (vs England-England): 0.000000

VERDICT: ✗ NO SIGNIFICANT DIFFERENCE FROM BASELINE

✓ STEP 5 COMPLETE!

Results saved to: /content/drive/MyDrive/Masinõpe_tulemused

Generated files:
  1. baseline_similarities.csv - Raw similarity data
  2. baseline_comparison.png - Statistical comparison

🎯 Check 'baseline_comparison.png' for final verdict!

💡 This analysis answers: Is your cross-collection similarity
   HIGHER than expected by random chance?


## Etapp 6. Eesti kangaste indeksi loomine

### Eesmärk

Selle etapi eesmärk on luua Eesti kangaste tunnuste andmebaas ehk indeks.

Varasemates etappides kasutati peamiselt Inglismaa kangaste indeksit, et võrrelda Eesti kangaid Inglismaa kangastega. Selles etapis luuakse samasugune indeks ka Eesti kangastele, et hiljem oleks võimalik võrrelda üksikut kasutaja valitud kangast ka Eesti kogus olevate kangastega.

### Sisendandmed

- **Algpildid:**
  - `Eesti_10`
  - `Eesti_20`

- **Andmetüüp:** pildifailid  
- **Formaadid:** `.jpg`, `.jpeg`, `.png`
- **Asukoht:** Google Drive / Colab

Kaustad vastavad kahele erinevale kangaproovi suurusele:

- `Eesti_10` — 10 cm proovid;
- `Eesti_20` — 20 cm proovid.

### Analüüsi loogika

Selles etapis ei otsita veel sarnaseid vasteid. Selle asemel arvutatakse igale Eesti kangapildile samad tunnused, mida kasutati varasemates sarnasuse arvutustes.

Iga pildi kohta arvutatakse:

1. **CNN-tunnused**
   - ResNet50 mudeli abil;
   - kirjeldavad pildi üldist visuaalset struktuuri.

2. **LAB-värvipalett**
   - kirjeldab domineerivaid värve ja nende osakaale.

3. **Triibumustri proportsioonid**
   - kirjeldavad mustri rütmi ja triipude suhtelisi laiusi.

Need tunnused salvestatakse indeksisse, mida saab hiljem kiiresti uuesti kasutada.

### Peamised tegevused

Selles etapis tehakse järgmised tegevused:

1. kontrollitakse, kas Eesti kangaste kaustad on olemas;
2. otsitakse kaustadest kõik pildifailid;
3. laaditakse ResNet50 mudel;
4. arvutatakse iga 10 cm Eesti kanga tunnused;
5. salvestatakse 10 cm kangaste indeks;
6. arvutatakse iga 20 cm Eesti kanga tunnused;
7. salvestatakse 20 cm kangaste indeks;
8. kuvatakse kokkuvõte edukalt töödeldud ja ebaõnnestunud failidest.

### Väljundid

Selle etapi tulemusena luuakse kaks indeksfaili:

- `estonian_10cm_index_v1.joblib`
- `estonian_20cm_index_v1.joblib`

Mõlemad failid sisaldavad:

- piltide tunnusepakette;
- failinimesid;
- suuruskategooriat;
- töödeldud piltide arvu;
- algkausta infot.

Need failid on järgmiste sammude sisendiks, kus kasutaja saab valida ühe konkreetse kanga ja otsida sellele sarnaseid vasteid.

### Ajahinnang

See samm võib võtta aega, sest iga Eesti kangapildi kohta tuleb arvutada mitu tunnuste rühma.

Ajakulu sõltub peamiselt:

- Eesti piltide arvust;
- pildifailide suurusest;
- GPU või CPU kasutusest;
- Google Drive’i lugemiskiirusest;
- sellest, kas kõik pildifailid on korrektselt avatavad.

10 cm ja 20 cm kaustad töödeldakse eraldi, kuid sama mudelit kasutatakse mõlema puhul.

### Nähtavad vaheväljundid

Notebookis kuvatakse:

- kas kaustad leiti üles;
- mitu pilti igas kaustas leiti;
- millisel seadmel mudel töötab;
- mitu pilti õnnestus töödelda;
- mitu pilti ebaõnnestus;
- kuhu indeksfailid salvestati;
- indeksfailide suurus.

Need väljundid aitavad kontrollida, kas Eesti kangaste andmebaas loodi edukalt.

### Tõlgendamise märkus

Selle etapi väljund ei ole veel analüütiline järeldus, vaid tehniline alus järgmisteks sammudeks.

Indeks võimaldab vältida seda, et Eesti kangaste tunnuseid peaks iga päringu või võrdluse ajal uuesti arvutama. See muudab hilisema ühe kanga otsingu ja Eesti-sisese võrdluse oluliselt kiiremaks.

Kui mõni pilt ebaõnnestub, tuleb kontrollida, kas fail on rikutud, vales formaadis või liiga väike tunnuste arvutamiseks.

In [ ]:
"""
ESTONIAN FABRIC INDEX BUILDER - BOTH SIZES
===========================================
Creates indices for BOTH Estonian fabric datasets (10cm and 20cm).

This builds separate indices for:
- Eesti_10 (10cm samples)
- Eesti_20 (20cm samples)

USAGE IN COLAB:
1. Make sure Estonian fabric images are in:
   - /content/drive/MyDrive/Kangad/Eesti_10/
   - /content/drive/MyDrive/Kangad/Eesti_20/
2. Run this script
3. It will process BOTH folders automatically
4. Indices will be saved as:
   - estonian_10cm_index_v1.joblib
   - estonian_20cm_index_v1.joblib

Then you can use the Visual Query Tool!
"""

import os
import numpy as np
import joblib
import torch
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import cv2
from tqdm import tqdm
import glob

# ============================================================
# SETTINGS
# ============================================================
BASE_PATH = "/content/drive/MyDrive/Kangad"
EESTI_10_DIR = os.path.join(BASE_PATH, "Eesti_10")
EESTI_20_DIR = os.path.join(BASE_PATH, "Eesti_20")
RESULTS_DIR = "/content/drive/MyDrive/Masinõpe_tulemused"

print("=" * 70)
print("ESTONIAN FABRIC INDEX BUILDER - BOTH SIZES")
print("=" * 70)


# ============================================================
# FEATURE EXTRACTION FUNCTIONS (same as query tool)
# ============================================================
def get_transform(size_category: str):
    if size_category == "10cm":
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.CenterCrop(224),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])


def extract_cnn_features(img_path: str, model, device, size_category: str):
    img = Image.open(img_path).convert("RGB")
    img_t = get_transform(size_category)(img).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = model(img_t).squeeze().cpu().numpy()
    return feat


def extract_lab_palette(img: Image.Image, n_colors: int = 6):
    arr = np.array(img)
    if arr.ndim != 3 or arr.shape[0] < 10 or arr.shape[1] < 10:
        return None, None

    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB).astype(np.float32)
    pixels = lab.reshape(-1, 3)

    if len(pixels) > 50000:
        idx = np.random.choice(len(pixels), 50000, replace=False)
        pixels = pixels[idx]

    K = min(n_colors, max(3, len(pixels) // 5000 + 3))
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1.0)

    try:
        _, labels, centers = cv2.kmeans(pixels, K, None, criteria, 3, cv2.KMEANS_PP_CENTERS)
    except:
        return None, None

    labels = labels.flatten()
    centers = centers.astype(np.float32)
    counts = np.bincount(labels, minlength=K).astype(np.float32)
    weights = counts / counts.sum()
    order = np.argsort(weights)[::-1]

    return centers[order], weights[order]


def extract_stripe_widths(img: Image.Image, min_stripe_px: int = 3):
    arr = np.array(img.convert("RGB"))
    if arr.shape[0] < 10 or arr.shape[1] < 10:
        return None

    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB).astype(np.float32)
    L = lab[:, :, 0]
    prof = L.mean(axis=0)
    prof_s = cv2.GaussianBlur(prof.reshape(1, -1), (1, 9), 0).flatten()
    g = np.abs(np.diff(prof_s, prepend=prof_s[0]))
    thr = np.percentile(g, 90)
    edge_idx = np.where(g >= thr)[0]

    if len(edge_idx) < 2:
        return None

    edges = [int(edge_idx[0])]
    for x in edge_idx[1:]:
        if x - edges[-1] >= min_stripe_px:
            edges.append(int(x))

    if edges[0] > 0:
        edges = [0] + edges
    if edges[-1] < arr.shape[1] - 1:
        edges = edges + [arr.shape[1] - 1]

    widths = []
    for a, b in zip(edges[:-1], edges[1:]):
        w = b - a
        if w >= min_stripe_px:
            widths.append(w)

    if len(widths) < 2:
        return None

    widths = np.array(widths, dtype=np.float32)
    return widths / widths.sum()


def extract_pack(img_path: str, size_cat: str, model, device):
    """Extract all features for one image"""
    try:
        img = Image.open(img_path).convert("RGB")
        cnn = extract_cnn_features(img_path, model, device, size_category=size_cat)
        pal_c, pal_w = extract_lab_palette(img, n_colors=6)
        sw = extract_stripe_widths(img, min_stripe_px=3)
        return {"cnn": cnn, "pal_c": pal_c, "pal_w": pal_w, "stripe_w": sw}
    except Exception as e:
        print(f"\n⚠️ Error processing {os.path.basename(img_path)}: {e}")
        return None


# ============================================================
# MAIN INDEX BUILDING FUNCTION
# ============================================================
def build_index_for_folder(img_dir: str, size_category: str, output_path: str, model, device):
    """Build index for one folder"""

    print("\n" + "=" * 70)
    print(f"PROCESSING: {os.path.basename(img_dir)} ({size_category})")
    print("=" * 70)

    # Check if source folder exists
    if not os.path.exists(img_dir):
        print(f"⚠️ WARNING: Folder not found: {img_dir}")
        print("   Skipping this dataset...")
        return False

    # Get all image files
    print("\nScanning for images...")
    image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
    image_paths = []
    for ext in image_extensions:
        image_paths.extend(glob.glob(os.path.join(img_dir, ext)))

    if len(image_paths) == 0:
        print(f"⚠️ WARNING: No images found in {img_dir}")
        print("   Skipping this dataset...")
        return False

    print(f"✓ Found {len(image_paths)} images")

    # Extract features for all images
    print(f"\nExtracting features (this may take 5-20 minutes)...")

    packs = []
    names = []
    sizes = []
    failed_count = 0

    for img_path in tqdm(image_paths, desc="Processing"):
        pack = extract_pack(img_path, size_category, model, device)

        if pack is not None:
            packs.append(pack)
            names.append(os.path.basename(img_path))
            sizes.append(size_category)
        else:
            failed_count += 1

    print(f"\n✓ Successfully processed: {len(packs)} images")
    if failed_count > 0:
        print(f"⚠️ Failed to process: {failed_count} images")

    if len(packs) == 0:
        print("❌ ERROR: No images were successfully processed!")
        return False

    # Save index
    print(f"\nSaving index to: {output_path}")

    index_data = {
        "packs": packs,
        "names": names,
        "sizes": sizes,
        "size_category": size_category,
        "total_images": len(packs),
        "source_folder": img_dir
    }

    joblib.dump(index_data, output_path)

    file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"✓ Index saved successfully!")
    print(f"  Total fabrics: {len(packs)}")
    print(f"  File size: {file_size_mb:.2f} MB")

    return True


def build_all_estonian_indices():
    """Build indices for all Estonian datasets"""

    # Create results directory if needed
    os.makedirs(RESULTS_DIR, exist_ok=True)

    # Load ResNet50 model (load once, use for both)
    print("\n" + "=" * 70)
    print("LOADING MODEL")
    print("=" * 70)
    print("Loading ResNet50...")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = models.resnet50(pretrained=True)
    model = torch.nn.Sequential(*list(model.children())[:-1]).to(device)
    model.eval()

    print(f"✓ Model loaded on device: {device}")

    # Build indices
    results = {}

    # 1. Build Eesti_10 index
    index_10_path = os.path.join(RESULTS_DIR, "estonian_10cm_index_v1.joblib")
    success_10 = build_index_for_folder(EESTI_10_DIR, "10cm", index_10_path, model, device)
    results["10cm"] = success_10

    # 2. Build Eesti_20 index
    index_20_path = os.path.join(RESULTS_DIR, "estonian_20cm_index_v1.joblib")
    success_20 = build_index_for_folder(EESTI_20_DIR, "20cm", index_20_path, model, device)
    results["20cm"] = success_20

    # Summary
    print("\n" + "=" * 70)
    print("BUILD SUMMARY")
    print("=" * 70)

    if results["10cm"]:
        print("✓ Eesti_10 (10cm) index created successfully")
        print(f"  Location: {index_10_path}")
    else:
        print("✗ Eesti_10 (10cm) index failed or skipped")

    if results["20cm"]:
        print("✓ Eesti_20 (20cm) index created successfully")
        print(f"  Location: {index_20_path}")
    else:
        print("✗ Eesti_20 (20cm) index failed or skipped")

    if results["10cm"] or results["20cm"]:
        print("\n" + "=" * 70)
        print("✓ ESTONIAN INDICES CREATED!")
        print("=" * 70)
        print("\nYou can now use the Visual Fabric Query Tool!")
        print("It will automatically use the correct index based on your query image size.")
    else:
        print("\n❌ ERROR: No indices were created successfully!")
        print("   Please check that image folders exist and contain images.")


# ============================================================
# RUN
# ============================================================
if __name__ == "__main__":
    try:
        build_all_estonian_indices()
    except KeyboardInterrupt:
        print("\n\nIndex building cancelled by user.")
    except Exception as e:
        print(f"\n❌ ERROR: {e}")
        import traceback
        traceback.print_exc()

ESTONIAN FABRIC INDEX BUILDER - BOTH SIZES

LOADING MODEL
Loading ResNet50...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


✓ Model loaded on device: cpu

PROCESSING: Eesti_10 (10cm)

Scanning for images...
✓ Found 134 images

Extracting features (this may take 5-20 minutes)...


Processing: 100%|██████████| 134/134 [00:40<00:00,  3.32it/s]



✓ Successfully processed: 134 images

Saving index to: /content/drive/MyDrive/Masinõpe_tulemused/estonian_10cm_index_v1.joblib
✓ Index saved successfully!
  Total fabrics: 134
  File size: 1.12 MB

PROCESSING: Eesti_20 (20cm)

Scanning for images...
✓ Found 556 images

Extracting features (this may take 5-20 minutes)...


Processing: 100%|██████████| 556/556 [02:34<00:00,  3.61it/s]



✓ Successfully processed: 556 images

Saving index to: /content/drive/MyDrive/Masinõpe_tulemused/estonian_20cm_index_v1.joblib
✓ Index saved successfully!
  Total fabrics: 556
  File size: 4.61 MB

BUILD SUMMARY
✓ Eesti_10 (10cm) index created successfully
  Location: /content/drive/MyDrive/Masinõpe_tulemused/estonian_10cm_index_v1.joblib
✓ Eesti_20 (20cm) index created successfully
  Location: /content/drive/MyDrive/Masinõpe_tulemused/estonian_20cm_index_v1.joblib

✓ ESTONIAN INDICES CREATED!

You can now use the Visual Fabric Query Tool!
It will automatically use the correct index based on your query image size.


## Etapp 7. Ühe valitud kanga visuaalne päring ja TOP 10 vastete leidmine

### Eesmärk

Selle etapi eesmärk on võrrelda ühte kasutaja valitud kangapilti nii Inglismaa kui ka Eesti kangaste andmebaasiga.

Kui varasemates etappides analüüsiti kogu andmestikku korraga, siis siin saab uurija valida ühe konkreetse kanga ja otsida sellele kõige sarnasemaid vasteid. Tulemuseks kuvatakse eraldi **TOP 10 Inglismaa vastet** ja **TOP 10 Eesti vastet**.

### Sisendandmed

Selles etapis kasutatakse järgmisi sisendeid:

- kasutaja valitud kangapilt;
- Inglismaa kangaste indeks `england_index_v1.joblib`;
- Eesti 10 cm kangaste indeks `estonian_10cm_index_v1.joblib`;
- Eesti 20 cm kangaste indeks `estonian_20cm_index_v1.joblib`;
- algsed pildikaustad visualiseerimiseks:
  - `Inglismaa9`
  - `Inglismaa_20`
  - `Eesti_10`
  - `Eesti_20`

Kasutaja sisestab notebookis või Colabis valitud pildi faili asukoha ning määrab, kas tegemist on 10 cm või 20 cm kangaprooviga.

### Analüüsi loogika

Kõigepealt arvutatakse valitud kangapildile samad tunnused, mida kasutati kogu töövoos:

1. CNN-tunnused;
2. LAB-värvipalett;
3. triibumustri proportsioonid.

Seejärel võrreldakse seda ühte kangast kõigi indeksites olevate kangastega.

Võrdlus tehakse kahes osas:

1. valitud kangas vs kõik Inglismaa kangad;
2. valitud kangas vs kõik Eesti kangad.

Eesti kangaste puhul kasutatakse korraga nii 10 cm kui ka 20 cm indeksit, et vasteid saaks otsida kogu Eesti andmestikust.

### Peamised tegevused

Selles etapis tehakse järgmised tegevused:

1. kontrollitakse, kas vajalikud Inglismaa ja Eesti indeksid on olemas;
2. küsitakse kasutajalt päringupildi failitee;
3. kontrollitakse, kas sisestatud fail on olemas;
4. küsitakse, kas päringupilt on 10 cm või 20 cm proov;
5. laaditakse Eesti ja Inglismaa indeksid;
6. laaditakse ResNet50 mudel;
7. arvutatakse päringupildi tunnused;
8. võrreldakse päringupilti kõigi Inglismaa kangastega;
9. valitakse 10 parimat Inglismaa vastet;
10. võrreldakse päringupilti kõigi Eesti kangastega;
11. valitakse 10 parimat Eesti vastet;
12. kuvatakse tulemused tabelina;
13. luuakse visuaalne võrdluspilt;
14. salvestatakse tulemused CSV-failidena.

### Väljundid

Selle etapi väljunditeks on kolm faili:

- `visual_query_<pildi_nimi>.png`  
  Visuaalne võrdlustabel, kus kuvatakse päringupilt, TOP 10 Inglismaa vastet ja TOP 10 Eesti vastet.

- `query_english_<pildi_nimi>.csv`  
  Tabel 10 parima Inglismaa vastega.

- `query_estonian_<pildi_nimi>.csv`  
  Tabel 10 parima Eesti vastega.

CSV-failides on muu hulgas järgmised veerud:

- vaste järjekoht;
- faili nimi;
- suuruskategooria;
- koondsarnasus;
- CNN-sarnasus;
- värvisarnasus;
- mustrisarnasus.

### Visualiseering

Loodav visuaalne võrdlus koondab tulemused ühele pildile.

Visualiseeringul on:

- üleval päringupilt;
- vasakul 10 parimat Inglismaa vastet;
- paremal 10 parimat Eesti vastet;
- iga vaste juures sarnasusskoor;
- värviline raam, mis näitab sarnasuse tugevust.

Raamide tõlgendus:

- roheline = väga tugev või tugev sarnasus;
- oranž = mõõdukas sarnasus;
- punane = nõrgem sarnasus.

See võimaldab tulemusi hinnata mitte ainult arvuliselt, vaid ka visuaalselt.

### Ajahinnang

See samm on kiirem kui kogu andmestiku analüüs, sest tunnused arvutatakse ainult ühele päringupildile.

Ajakulu sõltub peamiselt:

- indeksite suurusest;
- sellest, kui kiiresti mudel päringupildi tunnused arvutab;
- GPU või CPU kasutusest;
- pildifailide lugemise kiirusest;
- visualiseeringu salvestamise ajast.

Kui indeksid on juba loodud, on see samm mõeldud korduvaks kasutamiseks erinevate üksikute kangaste uurimisel.

### Nähtavad vaheväljundid

Notebookis kuvatakse:

- kas vajalikud indeksid leiti üles;
- mitu Inglismaa kangast laaditi;
- mitu Eesti kangast laaditi;
- millist päringupilti analüüsitakse;
- päringupildi suuruskategooria;
- TOP 10 Inglismaa vastete tabel;
- TOP 10 Eesti vastete tabel;
- parim Inglismaa vaste;
- parim Eesti vaste;
- salvestatud failide asukohad.

Need väljundid aitavad kontrollida, kas päring õnnestus ja kas tulemused salvestati õigesse kausta.

### Tõlgendamise märkus

Selle etapi tulemused sobivad ühe konkreetse kanga detailseks uurimiseks.

Kõrge sarnasusskoor näitab, et valitud kangas on mudeli kasutatud tunnuste põhjal visuaalselt sarnane mõne andmebaasis oleva kangaga. See ei tähenda automaatselt ajaloolist seost või ühist päritolu, vaid annab uurijale kandidaadid, mida saab edasi käsitsi ja sisuliselt võrrelda.

Eriti kasulik on see samm siis, kui uurijat huvitab üks konkreetne kangas ning ta soovib kiiresti näha, kas sarnaseid mustreid leidub rohkem Eesti või Inglismaa kogus.

In [ ]:
"""
VISUAL FABRIC QUERY TOOL - ENGLISH & ESTONIAN MATCHES
======================================================
Upload your fabric image and see visual comparison with both datasets!

FEATURES:
- Finds top 10 matches from English fabrics
- Finds top 10 matches from Estonian fabrics (automatically selects 10cm or 20cm)
- Creates visual comparison with similarity scores
- Saves results as image and CSV

USAGE IN COLAB:
1. Upload your image to Drive
2. Run this script
3. Enter the image path when prompted
4. View beautiful visual results!
"""

import os
import numpy as np
import pandas as pd
import joblib
import torch
import torchvision.models as models
from torchvision import transforms
from PIL import Image, ImageDraw, ImageFont
import cv2
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# ============================================================
# SETTINGS
# ============================================================
BASE_PATH = "/content/drive/MyDrive/Kangad"
RESULTS_DIR = "/content/drive/MyDrive/Masinõpe_tulemused"
ENGLISH_INDEX_PATH = os.path.join(RESULTS_DIR, "england_index_v1.joblib")
ESTONIAN_10CM_INDEX_PATH = os.path.join(RESULTS_DIR, "estonian_10cm_index_v1.joblib")
ESTONIAN_20CM_INDEX_PATH = os.path.join(RESULTS_DIR, "estonian_20cm_index_v1.joblib")

# Image paths
ENGLISH_9_IMG_DIR = os.path.join(BASE_PATH, "Inglismaa9")
ENGLISH_20_IMG_DIR = os.path.join(BASE_PATH, "Inglismaa_20")
ESTONIAN_10_IMG_DIR = os.path.join(BASE_PATH, "Eesti_10")
ESTONIAN_20_IMG_DIR = os.path.join(BASE_PATH, "Eesti_20")

print("=" * 70)
print("VISUAL FABRIC QUERY TOOL - ENGLISH & ESTONIAN")
print("=" * 70)
print("\nFinds matches in BOTH datasets with visual comparison!")


# ============================================================
# FEATURE EXTRACTION FUNCTIONS (same as original)
# ============================================================
def get_transform(size_category: str):
    if size_category == "10cm":
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.CenterCrop(224),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])


def extract_cnn_features(img_path: str, model, device, size_category: str):
    img = Image.open(img_path).convert("RGB")
    img_t = get_transform(size_category)(img).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = model(img_t).squeeze().cpu().numpy()
    return feat


def extract_lab_palette(img: Image.Image, n_colors: int = 6):
    arr = np.array(img)
    if arr.ndim != 3 or arr.shape[0] < 10 or arr.shape[1] < 10:
        return None, None

    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB).astype(np.float32)
    pixels = lab.reshape(-1, 3)

    if len(pixels) > 50000:
        idx = np.random.choice(len(pixels), 50000, replace=False)
        pixels = pixels[idx]

    K = min(n_colors, max(3, len(pixels) // 5000 + 3))
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1.0)

    try:
        _, labels, centers = cv2.kmeans(pixels, K, None, criteria, 3, cv2.KMEANS_PP_CENTERS)
    except:
        return None, None

    labels = labels.flatten()
    centers = centers.astype(np.float32)
    counts = np.bincount(labels, minlength=K).astype(np.float32)
    weights = counts / counts.sum()
    order = np.argsort(weights)[::-1]

    return centers[order], weights[order]


def _greedy_min_cost_matching(cost):
    pairs = []
    used_r, used_c = set(), set()
    while len(used_r) < cost.shape[0] and len(used_c) < cost.shape[1]:
        best = None
        best_val = None
        for i in range(cost.shape[0]):
            if i in used_r:
                continue
            for j in range(cost.shape[1]):
                if j in used_c:
                    continue
                v = cost[i, j]
                if best_val is None or v < best_val:
                    best_val = v
                    best = (i, j)
        if best is None:
            break
        used_r.add(best[0])
        used_c.add(best[1])
        pairs.append(best)
    return pairs


def lab_palette_similarity(c1, w1, c2, w2):
    if c1 is None or c2 is None:
        return 0.0

    cost = np.linalg.norm(c1[:, None, :] - c2[None, :, :], axis=2)

    try:
        from scipy.optimize import linear_sum_assignment
        r, c = linear_sum_assignment(cost)
        pairs = list(zip(r.tolist(), c.tolist()))
    except:
        pairs = _greedy_min_cost_matching(cost)

    d_sum, w_sum = 0.0, 0.0
    for i, j in pairs:
        wij = float(min(w1[i], w2[j]))
        d_sum += wij * float(cost[i, j])
        w_sum += wij

    if w_sum <= 1e-9:
        return 0.0

    d = d_sum / w_sum
    return float(np.exp(-d / 60.0))


def extract_stripe_widths(img: Image.Image, min_stripe_px: int = 3):
    arr = np.array(img.convert("RGB"))
    if arr.shape[0] < 10 or arr.shape[1] < 10:
        return None

    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB).astype(np.float32)
    L = lab[:, :, 0]
    prof = L.mean(axis=0)
    prof_s = cv2.GaussianBlur(prof.reshape(1, -1), (1, 9), 0).flatten()
    g = np.abs(np.diff(prof_s, prepend=prof_s[0]))
    thr = np.percentile(g, 90)
    edge_idx = np.where(g >= thr)[0]

    if len(edge_idx) < 2:
        return None

    edges = [int(edge_idx[0])]
    for x in edge_idx[1:]:
        if x - edges[-1] >= min_stripe_px:
            edges.append(int(x))

    if edges[0] > 0:
        edges = [0] + edges
    if edges[-1] < arr.shape[1] - 1:
        edges = edges + [arr.shape[1] - 1]

    widths = []
    for a, b in zip(edges[:-1], edges[1:]):
        w = b - a
        if w >= min_stripe_px:
            widths.append(w)

    if len(widths) < 2:
        return None

    widths = np.array(widths, dtype=np.float32)
    return widths / widths.sum()


def dtw_distance(a, b):
    n, m = len(a), len(b)
    D = np.full((n + 1, m + 1), np.inf, dtype=np.float32)
    D[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = abs(a[i - 1] - b[j - 1])
            D[i, j] = cost + min(D[i - 1, j], D[i, j - 1], D[i - 1, j - 1])
    return float(D[n, m])


def stripe_ratio_similarity(w1, w2):
    if w1 is None or w2 is None:
        return 0.0
    d = dtw_distance(w1, w2)
    return float(np.exp(-d / 0.5))


def extract_pack(img_path: str, size_cat: str, model, device):
    img = Image.open(img_path).convert("RGB")
    cnn = extract_cnn_features(img_path, model, device, size_category=size_cat)
    pal_c, pal_w = extract_lab_palette(img, n_colors=6)
    sw = extract_stripe_widths(img, min_stripe_px=3)
    return {"cnn": cnn, "pal_c": pal_c, "pal_w": pal_w, "stripe_w": sw}


def combined_similarity(a, b, w_cnn=0.55, w_color=0.25, w_ratio=0.20):
    cnn_sim = float(cosine_similarity([a["cnn"]], [b["cnn"]])[0][0])
    cnn_sim_01 = (cnn_sim + 1.0) / 2.0
    color_sim = lab_palette_similarity(a["pal_c"], a["pal_w"], b["pal_c"], b["pal_w"])
    ratio_sim = stripe_ratio_similarity(a["stripe_w"], b["stripe_w"])
    total = w_cnn * cnn_sim_01 + w_color * color_sim + w_ratio * ratio_sim
    return total, cnn_sim_01, color_sim, ratio_sim


# ============================================================
# VISUALIZATION FUNCTIONS
# ============================================================
def create_visual_comparison(query_path, query_name, eng_results, est_results, eng_img_dir, output_path):
    """
    Creates a visual comparison grid:
    - Top: Query image with info
    - Bottom left: 10 English matches
    - Bottom right: 10 Estonian matches (from all sizes)
    """

    fig = plt.figure(figsize=(20, 26))
    gs = fig.add_gridspec(13, 2, hspace=0.3, wspace=0.2)

    # ===== QUERY IMAGE (top, spanning both columns) =====
    ax_query = fig.add_subplot(gs[0:2, :])

    try:
        query_img = Image.open(query_path).convert("RGB")
        ax_query.imshow(query_img)
        ax_query.axis('off')
        ax_query.set_title(f'PÄRING / QUERY: {query_name}',
                          fontsize=20, fontweight='bold', pad=20)
    except Exception as e:
        ax_query.text(0.5, 0.5, f'Error loading query image:\n{query_name}',
                     ha='center', va='center', fontsize=12)
        ax_query.axis('off')

    # ===== ENGLISH MATCHES (left column) =====
    ax_eng_title = fig.add_subplot(gs[2, 0])
    ax_eng_title.text(0.5, 0.5, 'TOP 10 INGLISE KANGAD\n(English Fabrics)',
                     ha='center', va='center', fontsize=16, fontweight='bold')
    ax_eng_title.axis('off')

    for i, row in enumerate(eng_results):
        ax = fig.add_subplot(gs[3+i, 0])

        # Try to find image in both English folders
        img_path = os.path.join(eng_img_dir, row['Fabric'])
        if not os.path.exists(img_path):
            # Try the other English folder
            alt_dir = ENGLISH_9_IMG_DIR if eng_img_dir == ENGLISH_20_IMG_DIR else ENGLISH_20_IMG_DIR
            img_path = os.path.join(alt_dir, row['Fabric'])

        try:
            if os.path.exists(img_path):
                img = Image.open(img_path).convert("RGB")
                ax.imshow(img)
            else:
                ax.text(0.5, 0.5, f'Image not found:\n{row["Fabric"]}',
                       ha='center', va='center', fontsize=8)
        except Exception as e:
            ax.text(0.5, 0.5, f'Error loading:\n{row["Fabric"]}',
                   ha='center', va='center', fontsize=8)

        # Add rank, name and similarity score
        title = f"#{i+1}: {row['Fabric']}\nSkoor: {row['Similarity']:.4f}"
        ax.set_title(title, fontsize=10, pad=5)
        ax.axis('off')

        # Color-code border by similarity
        sim = row['Similarity']
        if sim >= 0.85:
            color = 'green'
        elif sim >= 0.75:
            color = 'orange'
        else:
            color = 'red'

        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3)

    # ===== ESTONIAN MATCHES (right column) =====
    ax_est_title = fig.add_subplot(gs[2, 1])
    ax_est_title.text(0.5, 0.5, 'TOP 10 EESTI KANGAD\n(Estonian Fabrics)',
                     ha='center', va='center', fontsize=16, fontweight='bold')
    ax_est_title.axis('off')

    for i, row in enumerate(est_results):
        ax = fig.add_subplot(gs[3+i, 1])

        # Get the correct image directory for this specific result
        est_img_dir = row['img_dir']
        img_path = os.path.join(est_img_dir, row['Fabric'])

        try:
            if os.path.exists(img_path):
                img = Image.open(img_path).convert("RGB")
                ax.imshow(img)
            else:
                ax.text(0.5, 0.5, f'Image not found:\n{row["Fabric"]}',
                       ha='center', va='center', fontsize=8)
        except Exception as e:
            ax.text(0.5, 0.5, f'Error loading:\n{row["Fabric"]}',
                   ha='center', va='center', fontsize=8)

        # Add rank, name, SIZE, and similarity score
        title = f"#{i+1}: {row['Fabric']}\n({row['Size']}) Skoor: {row['Similarity']:.4f}"
        ax.set_title(title, fontsize=10, pad=5)
        ax.axis('off')

        # Color-code border by similarity
        sim = row['Similarity']
        if sim >= 0.85:
            color = 'green'
        elif sim >= 0.75:
            color = 'orange'
        else:
            color = 'red'

        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3)

    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()

    print(f"\n✓ Visual comparison saved: {output_path}")


# ============================================================
# MAIN QUERY FUNCTION
# ============================================================
def query_fabric_visual():
    # Check if English index exists
    if not os.path.exists(ENGLISH_INDEX_PATH):
        print(f"\n❌ ERROR: English index not found!")
        print(f"   Expected: {ENGLISH_INDEX_PATH}")
        return

    # Check if at least one Estonian index exists
    has_est_10 = os.path.exists(ESTONIAN_10CM_INDEX_PATH)
    has_est_20 = os.path.exists(ESTONIAN_20CM_INDEX_PATH)

    if not has_est_10 and not has_est_20:
        print(f"\n❌ ERROR: No Estonian indices found!")
        print(f"   Expected one of:")
        print(f"   - {ESTONIAN_10CM_INDEX_PATH}")
        print(f"   - {ESTONIAN_20CM_INDEX_PATH}")
        print("\n   Please run Estonian Index Builder first!")
        return

    # Get query image path
    print("\n" + "=" * 70)
    print("SISESTA KANGAPILDI TEE / ENTER FABRIC IMAGE PATH")
    print("=" * 70)
    print("\nNäited / Examples:")
    print("  /content/drive/MyDrive/Kangad/Eesti_20/my_fabric.jpg")
    print("  /content/drive/MyDrive/test_image.png")

    query_path = input("\nPildi tee / Image path: ").strip().strip('"').strip("'")

    if not os.path.exists(query_path):
        print(f"\n❌ VIGA / ERROR: Faili ei leitud / File not found: {query_path}")
        return

    # Ask for size
    print("\n" + "=" * 70)
    print("VALI PILDI SUURUS / SELECT IMAGE SIZE")
    print("=" * 70)
    print("Kas see on 10cm või 20cm proov? / Is this 10cm or 20cm sample?")
    print("  1 = 10cm (väiksem proov / smaller sample)")
    print("  2 = 20cm (suurem proov / larger sample)")

    size_choice = input("\nValik / Choice (1 or 2): ").strip()
    query_size = "10cm" if size_choice == "1" else "20cm"

    # Load BOTH Estonian indices if available (combine them for cross-size comparison)
    estonian_indices = []

    if has_est_10:
        print("Laadin Eesti_10 indeksit / Loading Eesti_10 index...")
        est_10_index = joblib.load(ESTONIAN_10CM_INDEX_PATH)
        estonian_indices.append({
            'packs': est_10_index["packs"],
            'names': est_10_index["names"],
            'sizes': est_10_index["sizes"],
            'img_dir': ESTONIAN_10_IMG_DIR
        })
        print(f"✓ Laaditud / Loaded {len(est_10_index['packs'])} kangast (10cm)")

    if has_est_20:
        print("Laadin Eesti_20 indeksit / Loading Eesti_20 index...")
        est_20_index = joblib.load(ESTONIAN_20CM_INDEX_PATH)
        estonian_indices.append({
            'packs': est_20_index["packs"],
            'names': est_20_index["names"],
            'sizes': est_20_index["sizes"],
            'img_dir': ESTONIAN_20_IMG_DIR
        })
        print(f"✓ Laaditud / Loaded {len(est_20_index['packs'])} kangast (20cm)")

    # Select correct English image directory
    english_img_dir = ENGLISH_20_IMG_DIR  # Default to 20cm folder
    # We'll handle both folders in visualization

    # Load indices
    print("\n" + "=" * 70)
    print("LAADIN ANDMEID / LOADING DATA")
    print("=" * 70)

    print("Laadin inglise kangaste indeksit / Loading English index...")
    eng_index = joblib.load(ENGLISH_INDEX_PATH)
    eng_packs = eng_index["packs"]
    eng_names = eng_index["names"]
    eng_sizes = eng_index["sizes"]
    print(f"✓ Laaditud / Loaded {len(eng_packs)} inglise kangast / English fabrics")

    # Load model
    print("Laadin ResNet50 mudelit / Loading ResNet50 model...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = models.resnet50(pretrained=True)
    model = torch.nn.Sequential(*list(model.children())[:-1]).to(device)
    model.eval()
    print(f"✓ Kasutan seadet / Using device: {device}")

    # Extract query features
    print("\n" + "=" * 70)
    print("ANALÜÜSIN TEIE KANGAST / ANALYZING YOUR FABRIC")
    print("=" * 70)
    print(f"Pilt / Image: {os.path.basename(query_path)}")
    print(f"Suurus / Size: {query_size}")
    print("\nEraldan tunnuseid / Extracting features...")

    q_pack = extract_pack(query_path, query_size, model, device)
    print("✓ Tunnused eraldatud / Features extracted")

    # Compare with English fabrics
    print(f"\nVõrdlen {len(eng_packs)} inglise kangaga / Comparing with English fabrics...")

    eng_totals = np.zeros(len(eng_packs), dtype=np.float32)
    eng_cnn = np.zeros(len(eng_packs), dtype=np.float32)
    eng_col = np.zeros(len(eng_packs), dtype=np.float32)
    eng_rat = np.zeros(len(eng_packs), dtype=np.float32)

    for i, pack in enumerate(eng_packs):
        t, c, col, r = combined_similarity(q_pack, pack)
        eng_totals[i] = t
        eng_cnn[i] = c
        eng_col[i] = col
        eng_rat[i] = r

    # Get top 10 English matches
    eng_idxs = np.argsort(eng_totals)[-10:][::-1]

    eng_results = []
    for rank, idx in enumerate(eng_idxs, 1):
        eng_results.append({
            "Rank": rank,
            "Fabric": eng_names[int(idx)],
            "Size": eng_sizes[int(idx)],
            "Similarity": float(eng_totals[int(idx)]),
            "CNN": float(eng_cnn[int(idx)]),
            "COLOR": float(eng_col[int(idx)]),
            "RATIO": float(eng_rat[int(idx)]),
        })

    # Compare with Estonian fabrics (ALL sizes combined)
    print(f"\nVõrdlen eesti kangastega (kõik suurused) / Comparing with Estonian fabrics (all sizes)...")

    all_est_totals = []
    all_est_cnn = []
    all_est_col = []
    all_est_rat = []
    all_est_names = []
    all_est_sizes = []
    all_est_img_dirs = []

    for est_idx_data in estonian_indices:
        est_packs = est_idx_data['packs']

        for i, pack in enumerate(est_packs):
            t, c, col, r = combined_similarity(q_pack, pack)
            all_est_totals.append(t)
            all_est_cnn.append(c)
            all_est_col.append(col)
            all_est_rat.append(r)
            all_est_names.append(est_idx_data['names'][i])
            all_est_sizes.append(est_idx_data['sizes'][i])
            all_est_img_dirs.append(est_idx_data['img_dir'])

    # Convert to numpy arrays
    all_est_totals = np.array(all_est_totals, dtype=np.float32)
    all_est_cnn = np.array(all_est_cnn, dtype=np.float32)
    all_est_col = np.array(all_est_col, dtype=np.float32)
    all_est_rat = np.array(all_est_rat, dtype=np.float32)

    print(f"✓ Võrreldi {len(all_est_totals)} eesti kangaga kokku / Compared with {len(all_est_totals)} Estonian fabrics total")

    # Get top 10 Estonian matches from ALL sizes
    est_idxs = np.argsort(all_est_totals)[-10:][::-1]

    est_results = []
    for rank, idx in enumerate(est_idxs, 1):
        est_results.append({
            "Rank": rank,
            "Fabric": all_est_names[int(idx)],
            "Size": all_est_sizes[int(idx)],
            "Similarity": float(all_est_totals[int(idx)]),
            "CNN": float(all_est_cnn[int(idx)]),
            "COLOR": float(all_est_col[int(idx)]),
            "RATIO": float(all_est_rat[int(idx)]),
            "img_dir": all_est_img_dirs[int(idx)]
        })

    # Display results
    print("\n" + "=" * 70)
    print("INGLISE KANGAD - TOP 10 / ENGLISH FABRICS - TOP 10")
    print("=" * 70)
    eng_df = pd.DataFrame(eng_results)
    print(eng_df.to_string(index=False))

    print("\n" + "=" * 70)
    print("EESTI KANGAD - TOP 10 / ESTONIAN FABRICS - TOP 10")
    print("=" * 70)
    est_df = pd.DataFrame(est_results)
    print(est_df.to_string(index=False))

    # Create visual comparison
    print("\n" + "=" * 70)
    print("LOON VISUAALSET VÕRDLUST / CREATING VISUAL COMPARISON")
    print("=" * 70)

    base_name = os.path.splitext(os.path.basename(query_path))[0]
    visual_path = os.path.join(RESULTS_DIR, f"visual_query_{base_name}.png")

    create_visual_comparison(
        query_path=query_path,
        query_name=os.path.basename(query_path),
        eng_results=eng_results,
        est_results=est_results,
        eng_img_dir=english_img_dir,
        output_path=visual_path
    )

    # Save CSV results
    eng_csv = os.path.join(RESULTS_DIR, f"query_english_{base_name}.csv")
    est_csv = os.path.join(RESULTS_DIR, f"query_estonian_{base_name}.csv")

    eng_df.to_csv(eng_csv, index=False, encoding="utf-8-sig")
    est_df.to_csv(est_csv, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 70)
    print("✓ TULEMUSED SALVESTATUD / RESULTS SAVED")
    print("=" * 70)
    print(f"Visuaalne võrdlus / Visual: {visual_path}")
    print(f"Inglise CSV / English CSV: {eng_csv}")
    print(f"Eesti CSV / Estonian CSV: {est_csv}")

    # Summary
    print("\n" + "=" * 70)
    print("KOKKUVÕTE / SUMMARY")
    print("=" * 70)
    print(f"Parim inglise vaste / Best English match: {eng_results[0]['Fabric']}")
    print(f"  Sarnasus / Similarity: {eng_results[0]['Similarity']:.4f}")
    print(f"Parim eesti vaste / Best Estonian match: {est_results[0]['Fabric']}")
    print(f"  Sarnasus / Similarity: {est_results[0]['Similarity']:.4f}")

    # Ask if user wants to query another
    print("\n" + "=" * 70)
    another = input("\nOtsi veel? / Query another? (y/n): ").strip().lower()
    if another == 'y':
        print("\n\n")
        query_fabric_visual()


# ============================================================
# RUN
# ============================================================
if __name__ == "__main__":
    try:
        query_fabric_visual()
    except KeyboardInterrupt:
        print("\n\nPäring katkestatud / Query cancelled.")
    except Exception as e:
        print(f"\n❌ VIGA / ERROR: {e}")
        import traceback
        traceback.print_exc()

VISUAL FABRIC QUERY TOOL - ENGLISH & ESTONIAN

Finds matches in BOTH datasets with visual comparison!

SISESTA KANGAPILDI TEE / ENTER FABRIC IMAGE PATH

Näited / Examples:
  /content/drive/MyDrive/Kangad/Eesti_20/my_fabric.jpg
  /content/drive/MyDrive/test_image.png

Pildi tee / Image path: /content/drive/MyDrive/Kangad/IMG_0389.jpg

VALI PILDI SUURUS / SELECT IMAGE SIZE
Kas see on 10cm või 20cm proov? / Is this 10cm or 20cm sample?
  1 = 10cm (väiksem proov / smaller sample)
  2 = 20cm (suurem proov / larger sample)

Valik / Choice (1 or 2): 2
Laadin Eesti_10 indeksit / Loading Eesti_10 index...
✓ Laaditud / Loaded 134 kangast (10cm)
Laadin Eesti_20 indeksit / Loading Eesti_20 index...
✓ Laaditud / Loaded 556 kangast (20cm)

LAADIN ANDMEID / LOADING DATA
Laadin inglise kangaste indeksit / Loading English index...
✓ Laaditud / Loaded 3246 inglise kangast / English fabrics
Laadin ResNet50 mudelit / Loading ResNet50 model...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 141MB/s]


✓ Kasutan seadet / Using device: cpu

ANALÜÜSIN TEIE KANGAST / ANALYZING YOUR FABRIC
Pilt / Image: IMG_0389.jpg
Suurus / Size: 20cm

Eraldan tunnuseid / Extracting features...
✓ Tunnused eraldatud / Features extracted

Võrdlen 3246 inglise kangaga / Comparing with English fabrics...

Võrdlen eesti kangastega (kõik suurused) / Comparing with Estonian fabrics (all sizes)...
✓ Võrreldi 690 eesti kangaga kokku / Compared with 690 Estonian fabrics total

INGLISE KANGAD - TOP 10 / ENGLISH FABRICS - TOP 10
 Rank                Fabric Size  Similarity      CNN    COLOR    RATIO
    1 NM.0017648B_16 12.jpg 20cm    0.737740 0.889646 0.811369 0.227961
    2  NM.0017648B_16 5.jpg 20cm    0.735572 0.916794 0.738925 0.233019
    3 NM.0017648B_12U 6.jpg 20cm    0.729140 0.946781 0.727990 0.132067
    4 NM.0017648B_12V 4.jpg 20cm    0.726351 0.940160 0.702169 0.168603
    5 NM.0017648B_16 14.jpg 20cm    0.721584 0.918668 0.710211 0.193817
    6 NM.0017648B_12T 1.jpg 20cm    0.710988 0.905033 0.727358 

## Etapp 8. Kanga analüüsi visualiseerimine

### Eesmärk

Selle etapi eesmärk on näidata, mida algoritm kangapildi analüüsimisel tegelikult kasutab.

Kui varasemates etappides anti tulemuseks sarnasusskoorid ja parimad vasted, siis selles etapis visualiseeritakse kanga tunnused eraldi. See aitab aru saada, kuidas masin kangapilti lihtsustab ja milliste tunnuste põhjal hiljem sarnasust arvutab.

### Sisendandmed

- **Sisend:** üks kasutaja valitud kangapilt
- **Formaat:** pildifail, näiteks `.jpg` või `.png`
- **Asukoht:** Google Drive / Colab

Kasutaja sisestab pildi failitee käsitsi.

### Analüüsi loogika

Pildist eraldatakse kolm visuaalset tunnust:

1. **Originaalpilt**
   - näitab, milline kangas inimese silmale paistab.

2. **Domineerivad värvid**
   - pildist leitakse kuni 6 peamist värvi LAB-värviruumis;
   - iga värvi juurde arvutatakse selle ligikaudne osakaal pildis.

3. **Triibud ja servad**
   - pilt teisendatakse LAB-värviruumi;
   - kasutatakse heleduskanalit;
   - arvutatakse horisontaalne heledusprofiil;
   - otsitakse järske muutusi ehk võimalikke triibuservi;
   - arvutatakse triipude laiused ja nende proportsioonid.

### Peamised tegevused

Selles etapis tehakse järgmised tegevused:

1. küsitakse kasutajalt pildi failitee;
2. kontrollitakse, kas fail on olemas;
3. avatakse pilt;
4. arvutatakse domineeriv värvipalett;
5. arvutatakse heledusprofiil;
6. tuvastatakse võimalikud triipude servad;
7. arvutatakse triipude laiused;
8. visualiseeritakse kõik vahetulemused;
9. luuakse lihtsustatud rekonstruktsioonid;
10. salvestatakse tulemused PNG-failidena.

### Loodavad visualiseeringud

#### 1. Originaalpilt

Näitab analüüsitavat kangast muutmata kujul.

#### 2. Värvipalett

Näitab kuni 6 domineerivat värvi ja nende osakaale.

See aitab hinnata, milliseid värve algoritm peab pildi kõige olulisemateks toonideks.

#### 3. Triipude tuvastus

Originaalpildile kantakse punaste joontega tuvastatud servad.

See näitab, kus algoritm tajub võimalikke triipude või mustri üleminekuid.

#### 4. Heleduse profiil

Graafik näitab pildi heledust horisontaalsuunas.

Silutud profiil aitab vähendada müra ja tuua esile suuremad mustrimuutused.

#### 5. Gradient

Gradient näitab, kus heledus muutub kõige järsemalt.

Need järsud muutused on aluseks triibuservade leidmisele.

#### 6. Triipude laiused

Tulpdiagramm näitab tuvastatud triipude laiusi pikslites.

#### 7. Normaliseeritud proportsioonid

Sektordiagramm näitab, kui suure osa kogu mustrist iga tuvastatud triip moodustab.

#### 8. Digitaalne rekonstruktsioon

Lisaks luuakse lihtsustatud vaade sellest, mida algoritm pildist “näeb”:

- originaalpilt;
- pilt ainult domineerivate värvidega;
- triipude mudel keskmiste värvide põhjal.

### Väljundid

Selle etapi väljunditeks on kaks PNG-faili:

- `analysis_<pildi_nimi>.png`  
  Põhivisualiseering värvide, triipude, profiili ja gradientidega.

- `analysis_<pildi_nimi>_reconstruction.png`  
  Rekonstruktsioon, mis näitab pildi lihtsustatud värvi- ja triibumudelit.

### Ajahinnang

See samm on suhteliselt kiire, sest analüüsitakse ainult ühte pilti.

Ajakulu sõltub peamiselt:

- pildi suurusest;
- sellest, kui palju piksleid tuleb värvipaleti jaoks töödelda;
- sellest, kas triibud on selgelt tuvastatavad;
- pildifailide salvestamise kiirusest Google Drive’i.

### Nähtavad vaheväljundid

Notebookis kuvatakse:

- analüüsitava pildi nimi;
- pildi mõõtmed;
- mitu domineerivat värvi leiti;
- iga värvi osakaal;
- kas triibud tuvastati;
- mitu triipu leiti;
- triipude keskmine, minimaalne ja maksimaalne laius;
- salvestatud failide asukohad.

Need vaheväljundid aitavad mõista, kas algoritm suutis pildist vajalikud tunnused kätte saada.

### Tõlgendamise märkus

See etapp on oluline töövoo läbipaistvuse jaoks.

Masin ei “näe” kangast samamoodi nagu inimene. Ta teisendab pildi arvulisteks tunnusteks: värvideks, heledusmuutusteks ja triibumustri proportsioonideks. Seetõttu aitab see visualiseering kontrollida, kas algoritmi leitud tunnused vastavad sellele, mida uurija pildil visuaalselt oluliseks peab.

Kui triibud või värvid on valesti tuvastatud, võib see mõjutada ka sarnasuse arvutamise tulemusi järgmistes või varasemates etappides.

In [ ]:
"""
KANGASTE ANALÜÜSI VISUALISEERIJA
==================================
Näitab, mida algoritm TEGELIKULT näeb:
- Originaalpilt
- 6 domineerivat värvi (LAB ruumis)
- Leitud triibud ja servad
- Triipude laiused
- Gradiendi tugevus

KASUTAMINE:
Sisesta pildi tee ja vaata, mida algoritm analüüsib!
"""

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import cv2
from PIL import Image

print("=" * 70)
print("KANGASTE ANALÜÜSI VISUALISEERIJA")
print("=" * 70)
print("\nSee näitab, mida masinõpe algoritm sinu kangast analüüsides NÄEB!")


# ============================================================
# EXTRACT FUNCTIONS (sama mis põhikoodis)
# ============================================================
def extract_lab_palette(img: Image.Image, n_colors: int = 6):
    """Extract dominant colors in LAB color space"""
    arr = np.array(img)
    if arr.ndim != 3 or arr.shape[0] < 10 or arr.shape[1] < 10:
        return None, None, None

    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB).astype(np.float32)
    pixels = lab.reshape(-1, 3)

    if len(pixels) > 50000:
        idx = np.random.choice(len(pixels), 50000, replace=False)
        pixels = pixels[idx]

    K = min(n_colors, max(3, len(pixels) // 5000 + 3))
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1.0)

    try:
        _, labels, centers = cv2.kmeans(pixels, K, None, criteria, 3, cv2.KMEANS_PP_CENTERS)
    except:
        return None, None, None

    labels = labels.flatten()
    centers = centers.astype(np.float32)
    counts = np.bincount(labels, minlength=K).astype(np.float32)
    weights = counts / counts.sum()
    order = np.argsort(weights)[::-1]

    # Convert LAB back to RGB for display
    centers_rgb = []
    for lab_color in centers[order]:
        lab_img = np.uint8([[lab_color]])
        rgb = cv2.cvtColor(lab_img, cv2.COLOR_LAB2RGB)[0][0]
        centers_rgb.append(rgb / 255.0)

    return centers[order], weights[order], centers_rgb


def extract_stripe_analysis(img: Image.Image, min_stripe_px: int = 3):
    """Extract stripe information with detailed analysis"""
    arr = np.array(img.convert("RGB"))
    if arr.shape[0] < 10 or arr.shape[1] < 10:
        return None

    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB).astype(np.float32)
    L = lab[:, :, 0]

    # Profile
    prof = L.mean(axis=0)
    prof_s = cv2.GaussianBlur(prof.reshape(1, -1), (1, 9), 0).flatten()

    # Gradient
    g = np.abs(np.diff(prof_s, prepend=prof_s[0]))
    thr = np.percentile(g, 90)
    edge_idx = np.where(g >= thr)[0]

    if len(edge_idx) < 2:
        return {
            'success': False,
            'message': 'Ei leidnud piisavalt triipe (gradient liiga tasane)'
        }

    edges = [int(edge_idx[0])]
    for x in edge_idx[1:]:
        if x - edges[-1] >= min_stripe_px:
            edges.append(int(x))

    if edges[0] > 0:
        edges = [0] + edges
    if edges[-1] < arr.shape[1] - 1:
        edges = edges + [arr.shape[1] - 1]

    widths = []
    for a, b in zip(edges[:-1], edges[1:]):
        w = b - a
        if w >= min_stripe_px:
            widths.append(w)

    if len(widths) < 2:
        return {
            'success': False,
            'message': 'Liiga vähe triipe leitud'
        }

    return {
        'success': True,
        'profile': prof,
        'profile_smoothed': prof_s,
        'gradient': g,
        'threshold': thr,
        'edges': edges,
        'widths': np.array(widths),
        'widths_normalized': np.array(widths) / sum(widths)
    }


# ============================================================
# VISUALIZATION
# ============================================================
def visualize_fabric_analysis(img_path: str, output_path: str = None):
    """
    Create comprehensive visualization of fabric analysis
    """
    # Load image
    if not os.path.exists(img_path):
        print(f"❌ Ei leia faili: {img_path}")
        return

    img = Image.open(img_path).convert("RGB")
    img_array = np.array(img)

    print(f"\n📸 Analüüsin: {os.path.basename(img_path)}")
    print(f"   Suurus: {img.size[0]}×{img.size[1]} px")

    # Extract features
    print("\n1️⃣ Ekstraktin värvipaleti...")
    lab_centers, weights, rgb_colors = extract_lab_palette(img, n_colors=6)

    print("2️⃣ Analüüsin triipe...")
    stripe_data = extract_stripe_analysis(img, min_stripe_px=3)

    # Create visualization
    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

    # ============================================================
    # 1. ORIGINAL IMAGE
    # ============================================================
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.imshow(img_array)
    ax1.set_title('ORIGINAALPILT', fontsize=14, fontweight='bold', pad=10)
    ax1.axis('off')

    # Add info text
    info_text = f"Suurus: {img.size[0]}×{img.size[1]} px\n"
    info_text += f"Formaat: {img.format if img.format else 'unknown'}"
    ax1.text(0.02, 0.98, info_text, transform=ax1.transAxes,
            fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    # ============================================================
    # 2. COLOR PALETTE (LAB)
    # ============================================================
    ax2 = fig.add_subplot(gs[0, 1])

    if lab_centers is not None and rgb_colors is not None:
        # Draw color bars
        bar_height = 1.0 / len(rgb_colors)
        for i, (rgb, weight) in enumerate(zip(rgb_colors, weights)):
            y = 1.0 - (i + 1) * bar_height
            rect = Rectangle((0, y), 1.0, bar_height,
                           facecolor=rgb, edgecolor='black', linewidth=2)
            ax2.add_patch(rect)

            # Add percentage
            ax2.text(0.5, y + bar_height/2, f'{weight*100:.1f}%',
                    ha='center', va='center', fontweight='bold',
                    fontsize=11, color='white' if sum(rgb) < 1.5 else 'black',
                    bbox=dict(boxstyle='round', facecolor='black' if sum(rgb) > 1.5 else 'white',
                             alpha=0.7, pad=0.3))

        ax2.set_xlim(0, 1)
        ax2.set_ylim(0, 1)
        ax2.set_title(f'VÄRVIPALETT (LAB)\n{len(rgb_colors)} domineerivat värvi',
                     fontsize=14, fontweight='bold', pad=10)
        ax2.axis('off')

        # Add legend
        legend_text = "Värvid on sorteeritud\nkaaluvuse järgi (ülevalt alla)"
        ax2.text(0.02, 0.02, legend_text, transform=ax2.transAxes,
                fontsize=8, verticalalignment='bottom',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    else:
        ax2.text(0.5, 0.5, '❌ Värvipaleti\nekstraheerimine ebaõnnestus',
                ha='center', va='center', fontsize=12)
        ax2.set_xlim(0, 1)
        ax2.set_ylim(0, 1)
        ax2.axis('off')

    # ============================================================
    # 3. STRIPE DETECTION OVERLAY
    # ============================================================
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.imshow(img_array)

    if stripe_data and stripe_data['success']:
        edges = stripe_data['edges']
        # Draw detected edges
        for edge in edges:
            ax3.axvline(x=edge, color='red', linewidth=2, alpha=0.7)

        ax3.set_title(f'TRIIPUDE TUVASTUS\nLeitud {len(edges)} serva',
                     fontsize=14, fontweight='bold', pad=10, color='darkred')

        # Add stripe count
        n_stripes = len(edges) - 1
        ax3.text(0.02, 0.98, f'Triipe: {n_stripes}', transform=ax3.transAxes,
                fontsize=11, verticalalignment='top', fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='red', alpha=0.7),
                color='white')
    else:
        ax3.set_title('TRIIPUDE TUVASTUS\n❌ Ei leidnud triipe',
                     fontsize=14, fontweight='bold', pad=10, color='gray')

    ax3.axis('off')

    # ============================================================
    # 4. LIGHTNESS PROFILE
    # ============================================================
    ax4 = fig.add_subplot(gs[1, :])

    if stripe_data and stripe_data['success']:
        prof = stripe_data['profile']
        prof_s = stripe_data['profile_smoothed']

        x = np.arange(len(prof))
        ax4.plot(x, prof, 'gray', alpha=0.3, linewidth=1, label='Originaal profiil')
        ax4.plot(x, prof_s, 'blue', linewidth=2, label='Silutud profiil (Gaussian blur)')

        # Mark edges
        edges = stripe_data['edges']
        for edge in edges:
            ax4.axvline(x=edge, color='red', linewidth=1.5, alpha=0.7, linestyle='--')

        ax4.set_xlabel('Horisontaalne positsioon (pikslid)', fontsize=12, fontweight='bold')
        ax4.set_ylabel('Heledus (L kanal LAB ruumis)', fontsize=12, fontweight='bold')
        ax4.set_title('HELEDUSE PROFIIL (vertikaalne keskmine)',
                     fontsize=14, fontweight='bold', pad=10)
        ax4.legend(fontsize=10, loc='upper right')
        ax4.grid(True, alpha=0.3)
        ax4.set_xlim(0, len(prof))
    else:
        ax4.text(0.5, 0.5, '❌ Profiili ei saanud arvutada',
                ha='center', va='center', fontsize=12, transform=ax4.transAxes)

    # ============================================================
    # 5. GRADIENT (Edge Detection)
    # ============================================================
    ax5 = fig.add_subplot(gs[2, 0])

    if stripe_data and stripe_data['success']:
        g = stripe_data['gradient']
        thr = stripe_data['threshold']

        x = np.arange(len(g))
        ax5.plot(x, g, 'darkgreen', linewidth=1.5, label='Gradient (muutuse kiirus)')
        ax5.axhline(y=thr, color='red', linewidth=2, linestyle='--',
                   label=f'Lävi (90. pertsentiil = {thr:.2f})')

        # Highlight detected edges
        edges = stripe_data['edges']
        for edge in edges:
            if edge < len(g):
                ax5.plot(edge, g[edge], 'ro', markersize=8)

        ax5.set_xlabel('Positsioon (pikslid)', fontsize=11, fontweight='bold')
        ax5.set_ylabel('Gradiendi tugevus', fontsize=11, fontweight='bold')
        ax5.set_title('GRADIENT (Servade tuvastamine)', fontsize=13, fontweight='bold', pad=10)
        ax5.legend(fontsize=9)
        ax5.grid(True, alpha=0.3)
        ax5.set_xlim(0, len(g))

        # Add info
        mean_grad = g.mean()
        max_grad = g.max()
        info = f"Keskmine: {mean_grad:.2f}\nMaksimum: {max_grad:.2f}"
        ax5.text(0.98, 0.98, info, transform=ax5.transAxes,
                fontsize=9, verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    else:
        ax5.text(0.5, 0.5, '❌ Gradienti ei saanud arvutada',
                ha='center', va='center', fontsize=11, transform=ax5.transAxes)

    # ============================================================
    # 6. STRIPE WIDTHS (Bar Chart)
    # ============================================================
    ax6 = fig.add_subplot(gs[2, 1])

    if stripe_data and stripe_data['success']:
        widths = stripe_data['widths']
        widths_norm = stripe_data['widths_normalized']

        x = np.arange(len(widths))
        bars = ax6.bar(x, widths, color='steelblue', edgecolor='black', alpha=0.7)

        # Color by relative width
        for i, (bar, w_norm) in enumerate(zip(bars, widths_norm)):
            if w_norm > 0.15:  # Wide stripe
                bar.set_color('darkblue')
            elif w_norm < 0.05:  # Narrow stripe
                bar.set_color('lightblue')

        ax6.set_xlabel('Triip #', fontsize=11, fontweight='bold')
        ax6.set_ylabel('Laius (pikslid)', fontsize=11, fontweight='bold')
        ax6.set_title(f'TRIIPUDE LAIUSED\nKokku {len(widths)} triipa',
                     fontsize=13, fontweight='bold', pad=10)
        ax6.set_xticks(x)
        ax6.set_xticklabels([f'{i+1}' for i in x])
        ax6.grid(axis='y', alpha=0.3)

        # Add percentage labels
        for i, (w, w_norm) in enumerate(zip(widths, widths_norm)):
            ax6.text(i, w + 2, f'{w_norm*100:.1f}%',
                    ha='center', fontsize=8, fontweight='bold')
    else:
        ax6.text(0.5, 0.5, stripe_data.get('message', '❌ Triipe ei leitud') if stripe_data else '❌ Analüüs ebaõnnestus',
                ha='center', va='center', fontsize=11, transform=ax6.transAxes,
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
        ax6.set_xlim(0, 1)
        ax6.set_ylim(0, 1)

    # ============================================================
    # 7. NORMALIZED WIDTHS (Proportions)
    # ============================================================
    ax7 = fig.add_subplot(gs[2, 2])

    if stripe_data and stripe_data['success']:
        widths_norm = stripe_data['widths_normalized']

        # Pie chart
        colors_pie = plt.cm.Blues(np.linspace(0.3, 0.9, len(widths_norm)))
        wedges, texts, autotexts = ax7.pie(widths_norm,
                                            autopct='%1.1f%%',
                                            colors=colors_pie,
                                            startangle=90)

        for autotext in autotexts:
            autotext.set_color('white')
            autotext.set_fontweight('bold')
            autotext.set_fontsize(9)

        ax7.set_title('PROPORTSIOONID\n(Normaliseeritud laiused)',
                     fontsize=13, fontweight='bold', pad=10)

        # Add legend
        legend_labels = [f'Triip {i+1}' for i in range(len(widths_norm))]
        ax7.legend(legend_labels, loc='upper left', bbox_to_anchor=(1.1, 1), fontsize=8)
    else:
        ax7.text(0.5, 0.5, '❌ Proportsioonid\npuuduvad',
                ha='center', va='center', fontsize=11, transform=ax7.transAxes)
        ax7.set_xlim(0, 1)
        ax7.set_ylim(0, 1)

    # ============================================================
    # 8. RECONSTRUCTED SIMPLIFIED IMAGE (NEW!)
    # ============================================================
    # Add new subplot for reconstruction
    fig2, axes2 = plt.subplots(1, 3, figsize=(18, 6))

    # Original
    axes2[0].imshow(img_array)
    axes2[0].set_title('ORIGINAAL', fontsize=14, fontweight='bold')
    axes2[0].axis('off')

    # Color-only reconstruction
    if lab_centers is not None and rgb_colors is not None:
        # Create simplified color image (6 colors only)
        arr = np.array(img)
        lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB).astype(np.float32)
        pixels = lab.reshape(-1, 3)

        # Assign each pixel to nearest color
        simplified = np.zeros_like(arr)
        for i in range(arr.shape[0]):
            for j in range(arr.shape[1]):
                pixel_lab = lab[i, j]
                distances = [np.linalg.norm(pixel_lab - center) for center in lab_centers]
                nearest = np.argmin(distances)
                # Use the RGB color directly
                simplified[i, j] = (np.array(rgb_colors[nearest]) * 255).astype(np.uint8)

        axes2[1].imshow(simplified)
        axes2[1].set_title(f'LIHTSUSTATUD VÄRVID\n(Ainult {len(rgb_colors)} värvi)',
                          fontsize=14, fontweight='bold')
        axes2[1].axis('off')
    else:
        axes2[1].text(0.5, 0.5, '❌ Ei saa\nrekonstrueerida',
                     ha='center', va='center', fontsize=12)
        axes2[1].axis('off')

    # Stripe-based reconstruction
    if stripe_data and stripe_data['success']:
        edges = stripe_data['edges']
        widths = stripe_data['widths']

        # Create striped image using average colors per stripe
        reconstructed = np.zeros((img_array.shape[0], img_array.shape[1], 3), dtype=np.uint8)

        for i in range(len(edges) - 1):
            start = edges[i]
            end = edges[i + 1]

            # Get average color in this stripe
            stripe_region = img_array[:, start:end, :]
            avg_color = stripe_region.mean(axis=(0, 1)).astype(np.uint8)

            # Fill stripe with average color
            reconstructed[:, start:end, :] = avg_color

        axes2[2].imshow(reconstructed)
        axes2[2].set_title(f'TRIIPUDE MUDEL\n({len(widths)} triipa, keskm. värvidega)',
                          fontsize=14, fontweight='bold')
        axes2[2].axis('off')
    else:
        axes2[2].text(0.5, 0.5, '❌ Ei saa\nrekonstrueerida',
                     ha='center', va='center', fontsize=12)
        axes2[2].axis('off')

    fig2.suptitle('DIGITAALNE REKONSTRUKTSIOON: Mida algoritm "näeb"',
                 fontsize=16, fontweight='bold')
    plt.tight_layout()

    if output_path:
        base, ext = os.path.splitext(output_path)
        reconstruction_path = f"{base}_reconstruction{ext}"
        fig2.savefig(reconstruction_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"✓ Salvestatud rekonstruktsioon: {reconstruction_path}")
    else:
        plt.show()

    plt.close(fig2)

    # Main title
    fig.suptitle(f'KANGASTE ANALÜÜSI VISUALISEERIMINE: {os.path.basename(img_path)}',
                fontsize=16, fontweight='bold', y=0.995)

    # Save or show
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"\n✓ Salvestatud: {output_path}")
    else:
        plt.tight_layout()
        plt.show()

    plt.close()

    # Print summary
    print("\n" + "=" * 70)
    print("ANALÜÜSI KOKKUVÕTE")
    print("=" * 70)

    if lab_centers is not None:
        print(f"\n✓ Värvipalett: {len(rgb_colors)} domineerivat värvi")
        for i, (weight) in enumerate(weights):
            print(f"  Värv {i+1}: {weight*100:.1f}% pildist")
    else:
        print("\n❌ Värvipalett: ekstraheerimine ebaõnnestus")

    if stripe_data and stripe_data['success']:
        print(f"\n✓ Triibud: {len(stripe_data['widths'])} triipa leitud")
        print(f"  Keskmine laius: {stripe_data['widths'].mean():.1f} px")
        print(f"  Min laius: {stripe_data['widths'].min():.0f} px")
        print(f"  Max laius: {stripe_data['widths'].max():.0f} px")
    else:
        print(f"\n❌ Triibud: {stripe_data.get('message', 'ei leitud') if stripe_data else 'analüüs ebaõnnestus'}")


# ============================================================
# MAIN
# ============================================================
def main():
    print("\n" + "=" * 70)
    print("SISESTA PILDI TEE")
    print("=" * 70)
    print("\nNäited:")
    print("  /content/drive/MyDrive/Kangad/Eesti_20/kangas.jpg")
    print("  /content/drive/MyDrive/test_image.png")

    img_path = input("\nPildi tee: ").strip().strip('"').strip("'")

    if not os.path.exists(img_path):
        print(f"\n❌ VIGA: Faili ei leitud: {img_path}")
        return

    # Output path
    output_dir = "/content/drive/MyDrive/Masinõpe_tulemused"
    os.makedirs(output_dir, exist_ok=True)

    base_name = os.path.splitext(os.path.basename(img_path))[0]
    output_path = os.path.join(output_dir, f"analysis_{base_name}.png")

    # Visualize
    visualize_fabric_analysis(img_path, output_path)

    print("\n" + "=" * 70)
    print("✓ VALMIS!")
    print("=" * 70)


if __name__ == "__main__":
    main()

KANGASTE ANALÜÜSI VISUALISEERIJA

See näitab, mida masinõpe algoritm sinu kangast analüüsides NÄEB!

SISESTA PILDI TEE

Näited:
  /content/drive/MyDrive/Kangad/Eesti_20/kangas.jpg
  /content/drive/MyDrive/test_image.png

Pildi tee: /content/drive/MyDrive/Kangad/Eesti_20/050350_ERM_10316_050350_pisipilt.jpg

📸 Analüüsin: 050350_ERM_10316_050350_pisipilt.jpg
   Suurus: 557×60 px

1️⃣ Ekstraktin värvipaleti...
2️⃣ Analüüsin triipe...
✓ Salvestatud rekonstruktsioon: /content/drive/MyDrive/Masinõpe_tulemused/analysis_050350_ERM_10316_050350_pisipilt_reconstruction.png

✓ Salvestatud: /content/drive/MyDrive/Masinõpe_tulemused/analysis_050350_ERM_10316_050350_pisipilt.png

ANALÜÜSI KOKKUVÕTE

✓ Värvipalett: 6 domineerivat värvi
  Värv 1: 21.5% pildist
  Värv 2: 20.0% pildist
  Värv 3: 16.9% pildist
  Värv 4: 15.8% pildist
  Värv 5: 14.0% pildist
  Värv 6: 11.8% pildist

✓ Triibud: 31 triipa leitud
  Keskmine laius: 17.9 px
  Min laius: 3 px
  Max laius: 46 px

✓ VALMIS!
